# MASTER baseline reproduction v6 — ONE-CLICK ALL 4 · TestSet300 + TestSetNew46
## bindEmbed21DL · bindNode24 · ESMBind · MetalBind

This notebook implements one common locked-model experiment:

\[
\boxed{\text{method's own native model creation} \rightarrow \text{LOCK} \rightarrow \text{TestSet300 + TestSetNew46}}
\]

### Direct all-binding models

- **bindEmbed21DL**: official released five-CNN ensemble; score metal, nucleic acid, small molecule, and any ligand.
- **bindNode24**: official released five-GNN ensemble; score metal, nucleic acid, small molecule, and any ligand.

### Metal-specific models

- **ESMBind**: official released five-model seven-ion ensemble. Run **all proteins** in TestSet300/TestSetNew46, pool seven ion probabilities by residue-wise maximum, and score only `metal_binding`.
- **MetalBind**: official released 13 ion-specific checkpoints. Run **all proteins** in TestSet300/TestSetNew46 with the authors' surface/chemistry pipeline, map surface predictions back to every canonical residue, pool 13 ion probabilities by residue-wise maximum, and score only `metal_binding`.

No external label is used to update weights. No method is allowed to silently drop a failed protein or residue.

### Primary comparison artifact

The final comparison uses the **authors' released checkpoints** for all four methods. Native retraining is a separate reproducibility question and is not required to define the locked comparator.

# Metric terminology — exact definitions

## A. Threshold-free pooled-residue metrics
All residues from all proteins are concatenated first.

- **`AP_AUPRC`**: `sklearn.average_precision_score`; this is the primary AUPRC/AP value.
- **`PR_AUC_trapezoid`**: trapezoidal area under the empirical precision-recall curve; reported separately because it is not identical to AP.
- **`AUROC`**.
- **`Brier`**.
- `n_residues`, `n_positive`, `n_negative`, `prevalence`.

## B. Pooled-residue threshold metrics
At a specified operating point, concatenate all residues before calculating one confusion matrix.

- TP, FP, TN, FN.
- accuracy.
- balanced accuracy.
- precision / PPV.
- recall / sensitivity.
- specificity.
- NPV.
- **`global_positive_F1`**: positive binding class F1 from the pooled-residue confusion matrix. This is what earlier notes called “micro/global F1”.
- **`global_MCC`**: MCC from the pooled-residue confusion matrix.
- F2.
- Jaccard.
- predicted-positive rate.

For completeness only:

- `negative_class_F1`.
- `class_macro_F1` = mean F1 of the negative and positive classes.
- `class_weighted_F1`.
- `class_micro_F1`. In binary single-label classification this essentially tracks accuracy and **is not** the binding-paper F1.

## C. Per-protein mean metrics
For every protein separately, calculate its residue confusion matrix and positive-class metrics. Include a protein iff

\[
TP + FP + FN > 0.
\]

Therefore a false-alarm-only protein is included, while a completely TN-only protein is excluded.

The notebook reports mean per-protein:

- precision.
- recall.
- specificity.
- **F1**.
- **MCC**.
- accuracy.

and reports:

- legacy 95% t-CI half-width.
- standard 95% t-CI half-width.
- protein-bootstrap 95% CI for F1 and MCC.
- number of included proteins.
- false-alarm-only proteins.
- TN-only excluded proteins.

These **per-protein mean F1/MCC** are the relevant quantities when checking the published bindEmbed21DL/bindNode24 tables.

## D. Operating points

1. **`fixed_0.5`**: common primary threshold for a simple apples-to-apples table.
2. **`OOF_F1`**: threshold selected from native development OOF predictions to maximize pooled-residue positive F1.
3. **`OOF_MCC`**: a *separate* native development OOF threshold selected to maximize MCC.
4. Test-oracle thresholds are diagnostic only and are never used for claims.

In [ ]:
# ============================================================
# 0. COLAB + DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, shutil, subprocess, pickle, hashlib, time
import numpy as np
import pandas as pd

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')
WORK = BASE/'baseline_reproduction_master_v6'
REPOS = WORK/'repos'
ENVS = WORK/'envs'
CACHE = WORK/'cache'
PRED = WORK/'predictions'
RESULTS = WORK/'results'
MANIFEST = WORK/'manifest'
CANON = WORK/'canonical_inputs'

for p in [WORK,REPOS,ENVS,CACHE,PRED,RESULTS,MANIFEST,CANON]:
    p.mkdir(parents=True,exist_ok=True)

TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

print('BASE   =',BASE)
print('WORK   =',WORK)
print('RESULTS=',RESULTS)

Mounted at /content/drive
BASE   = /content/drive/MyDrive/Protein-binding-Phi
WORK   = /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6
RESULTS= /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/results


In [ ]:
# ============================================================
# 1. MASTER SWITCHES — v6 ONE-CLICK DEFAULTS
# ============================================================
RUN_BIND21 = True
RUN_BINDNODE24 = True

# Reconstruct development OOF predictions for F1/MCC operating-point sensitivity.
RUN_OOF_THRESHOLDS = True

# Both metal-specific transfer branches are ON by default in v6.
RUN_ESMBIND_TRANSFER = True
RUN_METALBIND_TRANSFER = True

# Native MetalBind Zenodo reproduction is optional and not needed for the common
# locked-checkpoint Test300/Test46 comparison.
RUN_METALBIND_NATIVE = False
RUN_METALBIND_NATIVE_REFERENCE_CHECK = False

REBUILD_PROTT5 = False
REBUILD_ESMBIND_ESM2 = False
REBUILD_ESMBIND_ESMIF = False
REBUILD_METALBIND_ESMC = False
REBUILD_METALBIND_SURFACE = False

FIXED_THRESHOLD = 0.50
BOOTSTRAP_REPS = 2000
RANDOM_SEED = 42

# Published-scale reproduction guard for bindEmbed21DL / bindNode24.
REPRO_TOLERANCE_ABS = 0.04

# MetalBind is a GPU-only branch in the authors' executable model because their
# forward path explicitly invokes torch.cuda synchronization.
REQUIRE_GPU_FOR_METALBIND = True

# Fair locked-model protocol

| Method | Frozen model source | TestSet300 | TestSetNew46 | Common target(s) |
|---|---|---:|---:|---|
| **bindEmbed21DL** | authors' released 5-CNN ensemble from DevSet1014 development | ✅ | ✅ | metal, nucleic, small, any |
| **bindNode24** | authors' released 5-GNN ensemble from DevSet1010 development | ✅ | ✅ | metal, nucleic, small, any |
| **ESMBind** | authors' released 5-model, 7-ion ensemble | ✅ all proteins | ✅ all proteins | **metal only** |
| **MetalBind** | authors' released 13 ion-specific checkpoints | ✅ all proteins | ✅ all proteins | **metal only** |

For ESMBind/MetalBind, the external datasets are **not filtered to metal-positive proteins**. A protein with no metal-binding residues stays in the set and contributes true negatives / possible false positives.

For the two ion-specific methods, the prespecified pooled continuous score is:

\[
p_{metal}(r)=\max_j p_j(r).
\]

The common primary threshold-dependent table uses `0.5`; AUPRC/AUROC use the raw continuous pooled score.

# 2. Pin public repositories

Pinned revisions prevent future repository changes from silently altering the result.

- bindEmbed21/bindPredict: `c9b12e7369641a9cbc3afbef85ed2e55ea433aff`
- bindNode24: `86b85818dc5ec6567ca425a0c98dc4930ea62394`
- ESMBind: `5d81fb5b3965f973bcbbdde443132b31bef07348`
- MetalBind: `6d2d13c4566477a1b79ac8e22ac1aaa6840c25a0`

In [ ]:
# ============================================================
# 2A. CLONE + PIN
# ============================================================
REPO_SPECS={
    'bind21':('https://github.com/Rostlab/bindPredict.git','c9b12e7369641a9cbc3afbef85ed2e55ea433aff'),
    'bindnode24':('https://github.com/erckert/bindNode24.git','86b85818dc5ec6567ca425a0c98dc4930ea62394'),
    'esmbind':('https://github.com/Structurebiology-BNL/ESMBind.git','5d81fb5b3965f973bcbbdde443132b31bef07348'),
    'metalbind':('https://github.com/zpliulab/MetalBind.git','6d2d13c4566477a1b79ac8e22ac1aaa6840c25a0'),
}

def sh(cmd,cwd=None,check=True,capture=False):
    print('+',cmd)
    return subprocess.run(cmd,shell=True,cwd=cwd,check=check,text=True,
                          stdout=subprocess.PIPE if capture else None,
                          stderr=subprocess.STDOUT if capture else None)

for name,(url,sha) in REPO_SPECS.items():
    dst=REPOS/name
    if not dst.exists(): sh(f'git clone {url} "{dst}"')
    sh('git fetch --all --tags',cwd=dst)
    sh(f'git checkout --detach {sha}',cwd=dst)
    got=subprocess.check_output('git rev-parse HEAD',cwd=dst,shell=True,text=True).strip()
    assert got==sha,(name,got,sha)
    print(name,'=>',got)

(MANIFEST/'repo_pins.json').write_text(json.dumps(
    {k:{'url':v[0],'sha':v[1]} for k,v in REPO_SPECS.items()},indent=2
))

+ git fetch --all --tags
+ git checkout --detach c9b12e7369641a9cbc3afbef85ed2e55ea433aff
bind21 => c9b12e7369641a9cbc3afbef85ed2e55ea433aff
+ git fetch --all --tags
+ git checkout --detach 86b85818dc5ec6567ca425a0c98dc4930ea62394
bindnode24 => 86b85818dc5ec6567ca425a0c98dc4930ea62394
+ git fetch --all --tags
+ git checkout --detach 5d81fb5b3965f973bcbbdde443132b31bef07348
esmbind => 5d81fb5b3965f973bcbbdde443132b31bef07348
+ git fetch --all --tags
+ git checkout --detach 6d2d13c4566477a1b79ac8e22ac1aaa6840c25a0
metalbind => 6d2d13c4566477a1b79ac8e22ac1aaa6840c25a0


537

# 3. Environment isolation

The four repositories use different PyTorch / PyG generations. They should not share one mutable environment.

This notebook uses `micromamba run -n ...` for the baseline repositories while the ordinary Colab kernel is used for the common evaluator and embedding generation.

In [ ]:
# ============================================================
# 3A. MICROMAMBA HELPERS — LOCAL /content ENVIRONMENTS
# ============================================================
# IMPORTANT:
# Do NOT create conda/micromamba environments inside Google Drive.
# Colab's Drive mount is FUSE-backed and executable entry points can fail
# with shell exit status 126 ("cannot execute / permission denied").
#
# Results, caches, predictions remain on Drive.
# Only executable environments are ephemeral and local under /content.

MAMBA = Path('/usr/local/bin/micromamba')
if not MAMBA.exists():
    sh(
        'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest '
        '| tar -xvj -C /tmp bin/micromamba'
    )
    sh('cp /tmp/bin/micromamba /usr/local/bin/micromamba')
    sh('chmod +x /usr/local/bin/micromamba')

LOCAL_MAMBA_ROOT = Path('/content/micromamba_binding_baselines')
LOCAL_MAMBA_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['MAMBA_ROOT_PREFIX'] = str(LOCAL_MAMBA_ROOT)

print('Micromamba executable :', MAMBA)
print('Micromamba env root   :', LOCAL_MAMBA_ROOT)
print('Persistent results    :', RESULTS)


def env_path(name):
    return LOCAL_MAMBA_ROOT/'envs'/name


def env_healthy(name):
    """
    Check that an existing local environment can actually execute Python.
    A directory merely existing is not enough.
    """
    p = env_path(name)
    if not p.exists():
        return False
    r = subprocess.run(
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
        f'micromamba run -n {name} python -c "import sys; print(sys.executable)"',
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if r.returncode != 0:
        print(f'BROKEN ENV {name}: return code {r.returncode}')
        print(r.stderr[-2000:])
        return False
    return True


def create_env(name, python='3.10'):
    if env_healthy(name):
        print('REUSE LOCAL ENV', name)
        return

    # Remove a partially-created/broken local environment before recreating.
    p = env_path(name)
    if p.exists():
        print('REMOVE BROKEN LOCAL ENV', name)
        sh(f'rm -rf "{p}"')

    sh(
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
        f'micromamba create -y -n {name} -c conda-forge '
        f'python={python} pip'
    )

    if not env_healthy(name):
        raise RuntimeError(f'Environment {name} was created but cannot execute Python.')


def mex(name, cmd, cwd=None, check=True, capture=False):
    """
    Execute a command inside a LOCAL micromamba environment.
    Use `python -m pip` rather than the pip entry-point whenever possible.
    """
    prefix = (
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
        f'micromamba run -n {name}'
    )
    return sh(
        f'{prefix} {cmd}',
        cwd=cwd,
        check=check,
        capture=capture
    )


+ curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /tmp bin/micromamba
+ cp /tmp/bin/micromamba /usr/local/bin/micromamba
+ chmod +x /usr/local/bin/micromamba
Micromamba executable : /usr/local/bin/micromamba
Micromamba env root   : /content/micromamba_binding_baselines
Persistent results    : /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/results


## Colab note: old Drive-hosted environments are intentionally ignored

Earlier notebook revisions used:

`/content/drive/MyDrive/Protein-binding-Phi/.../envs/mamba_root`

for micromamba environments. That location is **not used anymore**.

The old directory may remain on Drive, but v7 never executes anything from it.
All environments are recreated under:

`/content/micromamba_binding_baselines`

This fixes the common Colab exit-status **126** failure caused by attempting to
execute environment entry points from the Google Drive/FUSE mount.

After a Colab runtime restart, local environments disappear and will be
recreated automatically. Model outputs and expensive persistent feature caches
remain on Google Drive.


# 4. Canonical project datasets: Dev, TestSet300, TestSetNew46

The **final common evaluator always uses the project CSV labels**. This guarantees that the imported baseline probabilities are scored on the exact same labels as GEAR.

The notebook also checks sequence and label lengths before any metric is calculated.

In [ ]:
# ============================================================
# 4A. LOAD / AUDIT PROJECT CSVs
# ============================================================
LABELS=['any_ligand_binding_sites','metal_binding','small_binding','nuclear_binding']

def parse_list(v):
    if isinstance(v,str): return list(ast.literal_eval(v))
    return list(v)

def load_frame(path):
    df=pd.read_csv(path)
    df['prot_id']=df['prot_id'].astype(str)
    for c in LABELS:
        if c not in df.columns: raise KeyError(f'{path}: missing {c}')
        df[c]=df[c].apply(parse_list)
    return df

train_df=load_frame(TRAIN_CSV)
test300_df=load_frame(TEST300_CSV)
test46_df=load_frame(TEST46_CSV)

DATASETS={
    'TestSet300':test300_df,
    'TestSetNew46':test46_df,
}

rows=[]
for name,df in [('DevSet1010_project',train_df),*DATASETS.items()]:
    assert df.prot_id.is_unique,name
    for label in LABELS:
        for r in df.itertuples():
            assert len(r.sequence)==len(getattr(r,label)),(name,label,r.prot_id)
        nres=int(sum(len(x) for x in df[label])); pos=int(sum(sum(x) for x in df[label]))
        rows.append([name,label,len(df),nres,pos,nres-pos,pos/max(nres,1)])

audit_df=pd.DataFrame(rows,columns=['dataset','label','proteins','residues','positives','negatives','prevalence'])
display(audit_df)
audit_df.to_csv(RESULTS/'dataset_audit.csv',index=False)

,dataset,label,proteins,residues,positives,negatives,prevalence
0,DevSet1010_project,any_ligand_binding_sites,1014,170686,13999,156687,0.082016
1,DevSet1010_project,metal_binding,1014,170686,2374,168312,0.013909
2,DevSet1010_project,small_binding,1014,170686,9281,161405,0.054375
3,DevSet1010_project,nuclear_binding,1014,170686,2689,167997,0.015754
4,TestSet300,any_ligand_binding_sites,300,62689,5869,56820,0.093621
5,TestSet300,metal_binding,300,62689,881,61808,0.014054
6,TestSet300,small_binding,300,62689,3906,58783,0.062308
7,TestSet300,nuclear_binding,300,62689,1470,61219,0.023449
8,TestSetNew46,any_ligand_binding_sites,46,6227,575,5652,0.092340
9,TestSetNew46,metal_binding,46,6227,77,6150,0.012366


In [ ]:
# ============================================================
# 4B. CANONICAL FASTA / ID FILES FOR BOTH EXTERNAL SETS
# ============================================================
def write_fasta(df,path,style='plain'):
    with open(path,'w') as f:
        for r in df.itertuples():
            if style=='plain': header=r.prot_id
            elif style=='esmbind': header=f'sp|{r.prot_id}|{r.prot_id}'
            else: raise ValueError(style)
            f.write(f'>{header}\n{r.sequence}\n')

def write_ids(df,path):
    with open(path,'w') as f:
        for pid in df.prot_id: f.write(str(pid)+'\n')

for name,df in DATASETS.items():
    write_fasta(df,CANON/f'{name}.fasta','plain')
    write_fasta(df,CANON/f'{name}_esmbind.fasta','esmbind')
    write_ids(df,CANON/f'{name}_ids.txt')

print(sorted(p.name for p in CANON.iterdir()))

['TestSet300.fasta', 'TestSet300_esmbind.fasta', 'TestSet300_ids.txt', 'TestSetNew46.fasta', 'TestSetNew46_esmbind.fasta', 'TestSetNew46_ids.txt']


## Author-sequence provenance audit

Before scoring a public checkpoint, the notebook checks whether the project external sequences match sequences shipped by the corresponding author repository whenever those sequences are available. A same-ID sequence mismatch is treated as a hard error for a claimed native reproduction.

## Notebook-kernel dependency guard

The isolated baseline environments (`bind21`, `bindnode24`, `esmbind`, `metalbind`)
are separate from the **Colab notebook kernel**.

Some orchestration/audit cells run in the notebook kernel itself and therefore need
their own lightweight dependencies. In particular, `Bio.SeqIO` comes from **Biopython**.

This guard installs Biopython into the active notebook kernel only when it is missing.
It does not modify any of the isolated author-model environments.


In [ ]:
# ============================================================
# 4B.1 NOTEBOOK-KERNEL DEPENDENCY GUARD
# ============================================================
import importlib.util
import subprocess
import sys

def ensure_kernel_package(import_name, pip_spec):
    if importlib.util.find_spec(import_name) is not None:
        print(f'Kernel dependency OK: {import_name}')
        return

    print(f'Kernel dependency missing: {import_name}')
    print(f'Installing into active notebook kernel: {pip_spec}')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pip_spec],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            f'Failed to install {pip_spec} into notebook kernel '
            f'(return code {r.returncode})'
        )

    # Import-system caches may not know about a package installed moments ago.
    importlib.invalidate_caches()
    if importlib.util.find_spec(import_name) is None:
        raise RuntimeError(
            f'{pip_spec} installed without an obvious pip error, '
            f'but import {import_name} is still unavailable in this kernel.'
        )

    print(f'Installed kernel dependency: {pip_spec}')

ensure_kernel_package('Bio', 'biopython>=1.83,<2')

# Hard read-back test used immediately by the next provenance-audit cell.
from Bio import SeqIO
import Bio
print('Biopython:', Bio.__version__)
print('Bio.SeqIO import: OK')


Kernel dependency missing: Bio
Installing into active notebook kernel: biopython>=1.83,<2
Installed kernel dependency: biopython>=1.83,<2
Biopython: 1.88
Bio.SeqIO import: OK


In [ ]:
# ============================================================
# 4C. AUTHOR SEQUENCE PROVENANCE AUDIT
# ============================================================
from Bio import SeqIO

def collect_author_fastas(root):
    seqs={}
    for fp in Path(root).rglob('*'):
        if fp.suffix.lower() not in ['.fa','.faa','.fasta','.fas']: continue
        try:
            for rec in SeqIO.parse(str(fp),'fasta'):
                seqs.setdefault(str(rec.id),[]).append((str(fp),str(rec.seq)))
        except Exception:
            pass
    return seqs

def audit_author_sequences(df,author_root,method,dataset,require_all=False):
    idx=collect_author_fastas(author_root)
    rows=[]; mismatches=[]; missing=[]
    for r in df.itertuples():
        candidates=idx.get(str(r.prot_id),[])
        if not candidates:
            missing.append(str(r.prot_id)); rows.append([method,dataset,r.prot_id,'missing',False,'']); continue
        equal=[(fp,seq) for fp,seq in candidates if seq==r.sequence]
        ok=bool(equal)
        rows.append([method,dataset,r.prot_id,'found',ok,equal[0][0] if equal else candidates[0][0]])
        if not ok: mismatches.append(str(r.prot_id))
    if mismatches:
        raise RuntimeError(f'{method}/{dataset}: same-ID author sequence mismatch for {mismatches[:10]}')
    if require_all and missing:
        raise RuntimeError(f'{method}/{dataset}: author sequence unavailable for {missing[:10]}')
    return pd.DataFrame(rows,columns=['method','dataset','prot_id','author_status','sequence_equal','author_file'])

# bindEmbed21 ships both benchmark lineages; require complete matching where discoverable by exact ID.
# bindNode24 officially ships TestSet300 but not TestSetNew46.
seq_audits=[]
if (REPOS/'bind21').exists():
    seq_audits.append(audit_author_sequences(test300_df,REPOS/'bind21','bindEmbed21DL','TestSet300',False))
    seq_audits.append(audit_author_sequences(test46_df,REPOS/'bind21','bindEmbed21DL','TestSetNew46',False))
if (REPOS/'bindnode24').exists():
    seq_audits.append(audit_author_sequences(test300_df,REPOS/'bindnode24','bindNode24','TestSet300',False))
sequence_audit=pd.concat(seq_audits,ignore_index=True) if seq_audits else pd.DataFrame()
display(sequence_audit.groupby(['method','dataset','author_status','sequence_equal'],dropna=False).size().reset_index(name='n'))
sequence_audit.to_csv(RESULTS/'author_sequence_provenance_audit.csv',index=False)

,method,dataset,author_status,sequence_equal,n
0,bindEmbed21DL,TestSet300,found,True,300
1,bindEmbed21DL,TestSetNew46,found,True,46
2,bindNode24,TestSet300,found,True,300


# 5. Full common evaluator

## Threshold-free metrics

- `AP` = `average_precision_score`; this is the primary AUPRC-style metric used by GEAR.
- `PR_AUC_trapezoid` = trapezoidal integration of the precision-recall curve, saved separately because it is **not numerically identical** to AP.
- `AUROC`
- `Brier`

## Global residue-level threshold metrics

These are the **micro/global** metrics:

- TP, FP, TN, FN
- accuracy
- balanced accuracy
- precision / PPV
- recall / sensitivity
- specificity
- NPV
- F1
- F2
- Jaccard
- MCC
- predicted-positive rate

## Per-protein mean (`macro_positive`) metrics

A protein contributes if `(TP+FP+FN)>0`. The notebook saves:

- per-protein precision, recall, specificity, F1, MCC, accuracy;
- mean across included proteins;
- historical t-based CI;
- standard t-based CI;
- percentile bootstrap CI for F1/MCC;
- number of included proteins;
- number of false-alarm-only proteins;
- number of completely TN-only proteins excluded.

**These per-protein means are the quantities closest to the published bindEmbed21/bindNode24 F1/MCC tables.**

In [ ]:
# ============================================================
# 5A. FULL METRIC IMPLEMENTATION
# ============================================================
!pip -q install -U scipy scikit-learn biopython h5py tqdm

from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    f1_score, fbeta_score, matthews_corrcoef, precision_score, recall_score,
    accuracy_score, balanced_accuracy_score, jaccard_score, brier_score_loss,
    auc, roc_curve
)
from scipy.stats import t as student_t

EPS=1e-15

def confusion(y,pred):
    y=np.asarray(y,np.int8); pred=np.asarray(pred,np.int8)
    tp=int(np.sum((y==1)&(pred==1))); fp=int(np.sum((y==0)&(pred==1)))
    tn=int(np.sum((y==0)&(pred==0))); fn=int(np.sum((y==1)&(pred==0)))
    return tp,fp,tn,fn

def safe_div(a,b): return float(a/b) if b else 0.0

def ci_legacy(x):
    x=np.asarray(x,float)
    return np.nan if len(x)<2 else float(np.std(x,ddof=0)/np.sqrt(len(x))*student_t.ppf(.975,len(x)))

def ci_standard(x):
    x=np.asarray(x,float)
    return np.nan if len(x)<2 else float(np.std(x,ddof=1)/np.sqrt(len(x))*student_t.ppf(.975,len(x)-1))

def bootstrap_mean_ci(x,reps=2000,seed=42):
    x=np.asarray(x,float)
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(seed)
    vals=np.empty(reps,float)
    for i in range(reps): vals[i]=np.mean(rng.choice(x,size=len(x),replace=True))
    return tuple(np.percentile(vals,[2.5,97.5]).astype(float))

def threshold_free_metrics(y,p):
    y=np.asarray(y,np.int8); p=np.asarray(p,float)
    pr,rc,_=precision_recall_curve(y,p)
    # recall is returned descending; reverse before trapezoidal integration.
    pr_auc=float(auc(rc[::-1],pr[::-1]))
    return {
        'n_residues':int(len(y)),
        'n_positive':int(y.sum()),
        'n_negative':int(len(y)-y.sum()),
        'prevalence':float(y.mean()),
        'AP_AUPRC':float(average_precision_score(y,p)),
        'PR_AUC_trapezoid':pr_auc,
        'AUROC':float(roc_auc_score(y,p)),
        'Brier':float(brier_score_loss(y,p)),
    }

def global_threshold_metrics(y,p,t):
    y=np.asarray(y,np.int8); p=np.asarray(p,float); pred=(p>=t).astype(np.int8)
    tp,fp,tn,fn=confusion(y,pred)
    spec=safe_div(tn,tn+fp); npv=safe_div(tn,tn+fn)
    return {
        'threshold':float(t),'TP':tp,'FP':fp,'TN':tn,'FN':fn,
        'global_accuracy':float(accuracy_score(y,pred)),
        'global_balanced_accuracy':float(balanced_accuracy_score(y,pred)),
        'global_precision_PPV':float(precision_score(y,pred,zero_division=0)),
        'global_recall_sensitivity':float(recall_score(y,pred,zero_division=0)),
        'global_specificity':spec,'global_NPV':npv,
        'global_positive_F1':float(f1_score(y,pred,average='binary',pos_label=1,zero_division=0)),
        'global_F1':float(f1_score(y,pred,average='binary',pos_label=1,zero_division=0)),  # legacy alias
        'negative_class_F1':float(f1_score(y,pred,average='binary',pos_label=0,zero_division=0)),
        'class_macro_F1':float(f1_score(y,pred,average='macro',zero_division=0)),
        'class_weighted_F1':float(f1_score(y,pred,average='weighted',zero_division=0)),
        'class_micro_F1':float(f1_score(y,pred,average='micro',zero_division=0)),
        'global_F2':float(fbeta_score(y,pred,beta=2,zero_division=0)),
        'global_Jaccard':float(jaccard_score(y,pred,zero_division=0)),
        'global_MCC':float(matthews_corrcoef(y,pred)) if len(np.unique(pred))>1 else 0.0,
        'predicted_positive_rate':float(pred.mean()),
    }

def per_protein_table(df,probs,label,t):
    rows=[]
    for r in df.itertuples():
        pid=str(r.prot_id); y=np.asarray(getattr(r,label),np.int8); p=np.asarray(probs[pid],float)
        assert len(y)==len(p),(pid,len(y),len(p))
        pred=(p>=t).astype(np.int8); tp,fp,tn,fn=confusion(y,pred)
        include=(tp+fp+fn)>0
        false_alarm_only=((tp+fn)==0 and fp>0)
        precision=safe_div(tp,tp+fp); recall=safe_div(tp,tp+fn); specificity=safe_div(tn,tn+fp)
        f1=safe_div(2*tp,2*tp+fp+fn)
        den=(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
        mcc=float((tp*tn-fp*fn)/np.sqrt(den)) if den>0 else 0.0
        accuracy=safe_div(tp+tn,tp+fp+tn+fn)
        rows.append([pid,include,false_alarm_only,tp,fp,tn,fn,precision,recall,specificity,f1,mcc,accuracy])
    return pd.DataFrame(rows,columns=[
        'prot_id','included_macro_positive','false_alarm_only','TP','FP','TN','FN',
        'precision','recall','specificity','F1','MCC','accuracy'
    ])

def per_protein_summary(pt,reps=2000,seed=42):
    inc=pt[pt.included_macro_positive].copy()
    out={
        'protein_macro_positive_n':int(len(inc)),
        'protein_TN_only_excluded_n':int((~pt.included_macro_positive).sum()),
        'protein_false_alarm_only_n':int(pt.false_alarm_only.sum()),
    }
    for metric in ['precision','recall','specificity','F1','MCC','accuracy']:
        x=inc[metric].to_numpy(float)
        out[f'protein_mean_{metric}']=float(np.mean(x)) if len(x) else np.nan
        out[f'protein_{metric}_CI95_legacy_halfwidth']=ci_legacy(x)
        out[f'protein_{metric}_CI95_standard_halfwidth']=ci_standard(x)
        if metric in ['F1','MCC']:
            lo,hi=bootstrap_mean_ci(x,reps,seed)
            out[f'protein_{metric}_bootstrap95_low']=lo
            out[f'protein_{metric}_bootstrap95_high']=hi
    return out

def evaluate_probability_dict(df,probs,label,method,dataset,operating_point,t,comparison_tier):
    ids=[str(x) for x in df.prot_id]
    missing=[pid for pid in ids if pid not in probs]
    assert not missing,f'{method}/{dataset}/{label}: missing {missing[:10]}'
    y=np.concatenate([np.asarray(v,np.int8) for v in df[label]])
    p=np.concatenate([np.asarray(probs[str(pid)],float) for pid in df.prot_id])
    tf=threshold_free_metrics(y,p)
    gt=global_threshold_metrics(y,p,t)
    pt=per_protein_table(df,probs,label,t)
    ps=per_protein_summary(pt,BOOTSTRAP_REPS,RANDOM_SEED)
    out={'method':method,'dataset':dataset,'label':label,'comparison_tier':comparison_tier,
         'operating_point':operating_point,**tf,**gt,**ps}
    return out,pt

def save_probs(probs,path):
    rows=[]
    for pid,p in probs.items():
        for i,v in enumerate(np.asarray(p,float)): rows.append((pid,i,float(v)))
    pd.DataFrame(rows,columns=['prot_id','residue_index','probability']).to_csv(path,index=False,compression='gzip')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 594.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.5 MB/s eta 0:00:00


# 6. OOF threshold selection functions

Two separate thresholds are learned from **development OOF predictions only**:

- `OOF-F1 threshold`: exact threshold maximizing global F1;
- `OOF-MCC threshold`: exact threshold maximizing global MCC.

This avoids the earlier confusion where MCC was evaluated at an F1-optimized threshold.

In [ ]:
# ============================================================
# 6A. EXACT OOF THRESHOLD FUNCTIONS
# ============================================================
def choose_f1_threshold(y,p):
    y=np.asarray(y,np.int8); p=np.asarray(p,float)
    pr,rc,th=precision_recall_curve(y,p)
    if len(th)==0: return .5
    f=2*pr[:-1]*rc[:-1]/(pr[:-1]+rc[:-1]+1e-15)
    return float(th[int(np.nanargmax(f))])

def choose_mcc_threshold(y,p):
    y=np.asarray(y,np.int8); p=np.asarray(p,float)
    fpr,tpr,th=roc_curve(y,p,drop_intermediate=False)
    P=float(y.sum()); N=float(len(y)-y.sum())
    tp=tpr*P; fp=fpr*N; fn=P-tp; tn=N-fp
    den=(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
    mcc=np.zeros_like(th,dtype=float)
    ok=den>0
    mcc[ok]=(tp[ok]*tn[ok]-fp[ok]*fn[ok])/np.sqrt(den[ok])
    idx=int(np.nanargmax(mcc))
    t=float(th[idx])
    if not np.isfinite(t): t=float(np.nextafter(np.max(p),-np.inf))
    return t

def thresholds_from_oof(y_by_label,p_by_label):
    rows=[]; out={}
    for label in LABELS:
        y=np.asarray(y_by_label[label],np.int8); p=np.asarray(p_by_label[label],float)
        tf1=choose_f1_threshold(y,p); tmcc=choose_mcc_threshold(y,p)
        out[label]={'OOF_F1':tf1,'OOF_MCC':tmcc}
        mf1=global_threshold_metrics(y,p,tf1); mmcc=global_threshold_metrics(y,p,tmcc)
        rows.append([label,tf1,mf1['global_F1'],mf1['global_MCC'],tmcc,mmcc['global_F1'],mmcc['global_MCC']])
    return out,pd.DataFrame(rows,columns=[
        'label','OOF_F1_threshold','OOF_F1_at_threshold','OOF_MCC_at_F1_threshold',
        'OOF_MCC_threshold','OOF_F1_at_MCC_threshold','OOF_MCC_at_threshold'
    ])

# 7. Canonical ProtT5 embeddings for Dev + 300 + 46

Both bindEmbed21DL and bindNode24 use 1024-dimensional per-residue ProtT5 embeddings.

The notebook generates **full `L × 1024` HDF5 embeddings** and asserts every sequence length. No output probabilities are rounded before the common evaluator.

In [ ]:
# ============================================================
# 7A. PROTT5 GENERATOR
# ============================================================
!pip -q install -U "transformers<5" sentencepiece h5py

import h5py, torch
from transformers import T5EncoderModel,T5Tokenizer
from tqdm.auto import tqdm

PROTT5_MODEL='Rostlab/prot_t5_xl_half_uniref50-enc'

def generate_prott5_h5(df,path,rebuild=False):
    path=Path(path)
    if path.exists() and not rebuild:
        with h5py.File(path,'r') as h5:
            good=(len(h5.keys())==len(df) and all(str(r.prot_id) in h5 and h5[str(r.prot_id)].shape==(len(r.sequence),1024) for r in df.itertuples()))
        if good:
            print('REUSE',path); return path
        print('Existing H5 failed audit; rebuilding',path)
        path.unlink()

    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok=T5Tokenizer.from_pretrained(PROTT5_MODEL,do_lower_case=False)
    model=T5EncoderModel.from_pretrained(
        PROTT5_MODEL,
        torch_dtype=torch.float16 if device.type=='cuda' else torch.float32
    ).to(device).eval()

    with h5py.File(path,'w') as h5:
        for r in tqdm(df.itertuples(),total=len(df)):
            seq=''.join('X' if aa in 'UZOB' else aa for aa in str(r.sequence))
            enc=tok(' '.join(seq),return_tensors='pt',add_special_tokens=True)
            enc={k:v.to(device) for k,v in enc.items()}
            with torch.no_grad(): z=model(**enc).last_hidden_state[0][:len(seq)].float().cpu().numpy()
            assert z.shape==(len(seq),1024),(r.prot_id,z.shape,len(seq))
            h5.create_dataset(str(r.prot_id),data=z.astype(np.float32),compression='gzip')

    del model
    if device.type=='cuda': torch.cuda.empty_cache()
    return path

PROTT5={}
for name,df in DATASETS.items():
    PROTT5[name]=generate_prott5_h5(df,CACHE/f'{name}_ProtT5.h5',REBUILD_PROTT5)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
REUSE /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/TestSet300_ProtT5.h5
REUSE /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/TestSetNew46_ProtT5.h5


# 8. bindEmbed21DL: released-checkpoint reproduction on 300 + 46

### Native training/test protocol

- Train/development: DevSet1014 with five predefined protein folds.
- Five CNN models are produced; each fold is validation once and the remaining four folds are training.
- External inference averages the five model probabilities residue-wise.
- Native/published external sets: TestSet300 and TestSetNew46.
- Output channels: metal, nucleic, small. `any` is the union/maximum of those three outputs.

### Why released checkpoints are primary here

The objective of the main table is to reproduce the released author model as faithfully as possible. Therefore external numbers use the public pretrained checkpoints. A full from-scratch retrain is a separate reproducibility sensitivity analysis and is not required to recover the author-model external probabilities.

### Full-precision note for bindEmbed21DL

The released `ProteinResults` class rounds each checkpoint prediction and the final 5-model average to **3 decimals**. That is adequate for the original threshold-0.5 tables but unnecessarily quantizes AUPRC/AUROC.

This notebook monkey-patches **only the probability storage/averaging methods** so the CNN forward outputs are accumulated without 3-decimal rounding. The trained checkpoints, architecture, input embeddings, and 5-model ensemble remain unchanged. The published-scale guard is still used to verify that the resulting fixed-threshold per-protein F1/MCC remains consistent with the original paper.

## bindEmbed21 compatibility note

`bindEmbed21DL` is a 2021-era PyTorch implementation. Its released checkpoints are
kept on PyTorch 1.13.1 in this reproduction environment. The scientific Python
stack is therefore pinned to **NumPy 1.26.4** rather than the current NumPy 2.x
ABI.

The inference wrapper also captures the complete subprocess output. If anything
fails, the notebook now prints the actual inner Python traceback instead of only
showing a generic `CalledProcessError`.


In [ ]:
# ============================================================
# 8A. bindEmbed21 ENV — LEGACY-COMPATIBLE PINNED STACK
# ============================================================
B21 = REPOS/'bind21'

if RUN_BIND21 or RUN_OOF_THRESHOLDS:
    create_env('bind21', '3.10')

    # torch 1.13.x predates NumPy 2.x.  Using the newest NumPy can make
    # Tensor <-> NumPy conversion fail at inference time with
    # "Numpy is not available" / ABI errors.  Pin a coherent legacy stack.
    bind21_specs = [
        'numpy==1.26.4',
        'pandas==2.0.3',
        'scipy==1.11.4',
        'scikit-learn==1.3.2',
        'h5py==3.10.0',
        'torch==1.13.1',
    ]

    r = mex(
        'bind21',
        'python -m pip install --upgrade --force-reinstall ' +
        ' '.join([f'"{x}"' for x in bind21_specs]),
        check=False,
        capture=True,
    )
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(
            f'bind21 dependency installation failed (return code {r.returncode}). '
            'See the pip output immediately above.'
        )

    # Hard environment audit with exact versions and Tensor<->NumPy bridge.
    check = mex(
        'bind21',
        "python - <<'PY'\n"
        "import torch,numpy,pandas,scipy,sklearn,h5py\n"
        "print('torch',torch.__version__)\n"
        "print('numpy',numpy.__version__)\n"
        "print('pandas',pandas.__version__)\n"
        "print('scipy',scipy.__version__)\n"
        "print('sklearn',sklearn.__version__)\n"
        "print('h5py',h5py.__version__)\n"
        "print('cuda',torch.cuda.is_available())\n"
        "x=torch.tensor([1.,2.])\n"
        "a=x.numpy()\n"
        "y=torch.from_numpy(a)\n"
        "print('torch<->numpy bridge OK',a.tolist(),y.tolist())\n"
        "PY",
        capture=True
    )
    print(check.stdout)


+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba create -y -n bind21 -c conda-forge python=3.10 pip
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n bind21 python -m pip install --upgrade --force-reinstall "numpy==1.26.4" "pandas==2.0.3" "scipy==1.11.4" "scikit-learn==1.3.2" "h5py==3.10.0" "torch==1.13.1"
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 227.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 118.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 140.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 152.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 97.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.5/887.5 MB 32.3 MB/s  0:00:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 58.8 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 134.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# 8B. bindEmbed21 EXTERNAL ADAPTER FOR BOTH SETS
# ============================================================
def write_bind21_external_adapter(dataset,df,h5):
    outdir=PRED/'bindEmbed21DL'/dataset
    outdir.mkdir(parents=True,exist_ok=True)
    script=B21/f'run_{dataset}_fullprecision.py'

    script.write_text(f"""import sys
from pathlib import Path
import numpy as np
import h5py
import torch

sys.path.insert(0,r"{B21}")

from config import FileSetter
from bindEmbed21DL import BindEmbed21DL
from data_preparation import ProteinResults

H5 = r"{h5}"
IDS = r"{CANON/f"{dataset}_ids.txt"}"
FASTA = r"{CANON/f"{dataset}.fasta"}"
REPO = Path(r"{B21}")
OUT = Path(r"{outdir}")

print("=== bindEmbed21 adapter preflight ===")
print("python", sys.version)
print("torch", torch.__version__)
print("numpy", np.__version__)
print("cuda", torch.cuda.is_available())
print("H5", H5)

ids=[x.strip() for x in open(IDS) if x.strip()]
print("n ids", len(ids))

# Strict embedding audit inside the SAME environment that will run inference.
with h5py.File(H5,'r') as fh:
    missing=[pid for pid in ids if pid not in fh]
    if missing:
        raise RuntimeError(f"Missing {{len(missing)}} embeddings; first={{missing[:10]}}")
    for pid in ids:
        shape=fh[pid].shape
        if len(shape)!=2 or shape[1]!=1024:
            raise RuntimeError(f"{{pid}}: bad ProtT5 shape {{shape}}, expected (L,1024)")
print("embedding key/dimension audit PASS")

# IMPORTANT:
# Released ProteinResults rounds probabilities to 3 decimals internally.
# For threshold-free common metrics we remove ONLY that rounding.
def _as_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def _set_full(self,predictions):
    arr=_as_numpy(predictions)
    self.predictions=np.asarray(arr.T,dtype=np.float64)

def _add_full(self,predictions):
    arr=_as_numpy(predictions)
    self.predictions=np.add(self.predictions,np.asarray(arr,dtype=np.float64))

def _norm_full(self,norm_factor):
    self.predictions=self.predictions/float(norm_factor)

ProteinResults.set_predictions=_set_full
ProteinResults.add_predictions=_add_full
ProteinResults.normalize_predictions=_norm_full

FileSetter.embeddings_input=staticmethod(lambda:H5)

# Check that all five released checkpoints exist before loading any model.
for k in range(1,6):
    ckpt=REPO/"trained_models"/f"checkpoint{{k}}.pt"
    if not ckpt.exists():
        raise FileNotFoundError(ckpt)
print("5 released checkpoints present")

proteins=BindEmbed21DL.prediction_pipeline(
    model_prefix=str(REPO/"trained_models"/"checkpoint"),
    cutoff=.5,
    result_folder=None,
    ids=ids,
    fasta_file=FASTA,
    ri=False
)

OUT.mkdir(parents=True,exist_ok=True)
for pid,obj in proteins.items():
    arr=np.asarray(obj.predictions,dtype=np.float64).squeeze()
    if arr.ndim!=2 or arr.shape[1]!=3:
        raise RuntimeError(f"{{pid}}: unexpected prediction shape {{arr.shape}}")
    if not np.all(np.isfinite(arr)):
        raise RuntimeError(f"{{pid}}: non-finite probabilities")
    np.save(OUT/f"{{pid}}.npy",arr.astype(np.float32))

if len(proteins)!=len(ids):
    raise RuntimeError(f"Predicted {{len(proteins)}}/{{len(ids)}} proteins")

print("saved",len(proteins),"proteins to",OUT)
print("=== bindEmbed21 adapter PASS ===")
""")
    return script,outdir

B21_OUT={}
if RUN_BIND21:
    for dataset,df in DATASETS.items():
        script,outdir=write_bind21_external_adapter(dataset,df,PROTT5[dataset])

        # Capture subprocess output so a future failure exposes the REAL
        # inner traceback instead of only CalledProcessError.
        r=mex(
            'bind21',
            f'python "{script.name}"',
            cwd=B21,
            check=False,
            capture=True
        )
        print(r.stdout)

        if r.returncode != 0:
            raise RuntimeError(
                f'bindEmbed21 inference failed on {dataset} '
                f'(return code {r.returncode}). '
                'The complete inner traceback is printed immediately above.'
            )

        B21_OUT[dataset]=outdir


+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n bind21 python "run_TestSet300_fullprecision.py"
=== bindEmbed21 adapter preflight ===
python 3.10.21 | packaged by conda-forge | (main, Aug 21 2026, 22:42:31) [GCC 14.4.0]
torch 1.13.1+cu117
numpy 1.26.4
cuda True
H5 /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/TestSet300_ProtT5.h5
n ids 300
embedding key/dimension audit PASS
5 released checkpoints present
Prepare data
Load model
Calculate predictions
Number of input features: 1024
Load model
Calculate predictions
Number of input features: 1024
Load model
Calculate predictions
Number of input features: 1024
Load model
Calculate predictions
Number of input features: 1024
Load model
Calculate predictions
Number of input features: 1024
saved 300 proteins to /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/predictions/bindEmbed21DL/TestSet300
=== bindEmbed21 adapter PASS ===

+ MAMBA_ROOT_PREFIX="/cont

In [ ]:
# ============================================================
# 8C. NORMALIZE bindEmbed21 OUTPUTS
# ============================================================
def load_three_class_npy(df,folder):
    metal={}; nuc={}; small={}
    for r in df.itertuples():
        arr=np.load(Path(folder)/f'{r.prot_id}.npy')
        assert arr.shape==(len(r.sequence),3),(r.prot_id,arr.shape,len(r.sequence))
        metal[r.prot_id]=arr[:,0]; nuc[r.prot_id]=arr[:,1]; small[r.prot_id]=arr[:,2]
    anyp={pid:np.maximum.reduce([metal[pid],nuc[pid],small[pid]]) for pid in metal}
    return {'metal_binding':metal,'nuclear_binding':nuc,'small_binding':small,'any_ligand_binding_sites':anyp}

bind21_probs={}
if RUN_BIND21:
    for dataset,df in DATASETS.items():
        bind21_probs[dataset]=load_three_class_npy(df,B21_OUT[dataset])
        for label,d in bind21_probs[dataset].items():
            save_probs(d,PRED/f'bindEmbed21DL__{dataset}__{label}.csv.gz')

# 9. bindEmbed21 OOF thresholds (optional but recommended)

The released checkpoints map to the five original DevSet1014 folds. OOF reconstruction uses **checkpoint k only on split k**, never the 5-model ensemble on its own training data.

This section produces task-specific frozen `OOF-F1` and `OOF-MCC` thresholds that can then be applied unchanged to both external sets.

In [ ]:
# ============================================================
# 9A. BUILD bindEmbed21 PUBLIC DEV DATAFRAME
# ============================================================
from Bio import SeqIO

def read_fasta_dict(path): return {rec.id:str(rec.seq) for rec in SeqIO.parse(str(path),'fasta')}
def read_binding_file(path):
    d={}
    with open(path) as f:
        for line in f:
            s=line.strip().split()
            if len(s)>1: d[s[0]]=[int(x) for x in s[1].split(',') if x!='']
    return d

def build_public_bind21_dev():
    seqs=read_fasta_dict(B21/'data/development_set/all.fasta')
    ids=[]; fold_map={}
    for k in range(1,6):
        z=[x.strip() for x in open(B21/f'data/development_set/ids_split{k}.txt') if x.strip()]
        ids+=z
        for x in z: fold_map[x]=k
    bind={lab:read_binding_file(B21/f'data/development_set/binding_residues_2.5_{short}.txt') for lab,short in [
        ('metal_binding','metal'),('nuclear_binding','nuclear'),('small_binding','small')]}
    rows=[]
    for pid in ids:
        seq=seqs[pid]; rec={'prot_id':pid,'sequence':seq,'fold':fold_map[pid]}
        for lab in ['metal_binding','nuclear_binding','small_binding']:
            y=np.zeros(len(seq),dtype=np.int8)
            # Public residue positions are 1-based.
            for pos in bind[lab].get(pid,[]):
                if 1<=pos<=len(seq): y[pos-1]=1
            rec[lab]=y.tolist()
        rec['any_ligand_binding_sites']=np.maximum.reduce([
            rec['metal_binding'],rec['nuclear_binding'],rec['small_binding']]).astype(int).tolist()
        rows.append(rec)
    return pd.DataFrame(rows)

bind21_dev_df=build_public_bind21_dev() if RUN_OOF_THRESHOLDS else None
if bind21_dev_df is not None:
    display(bind21_dev_df.groupby('fold').size())

,0
fold,
1,203
2,203
3,203
4,203
5,202


In [ ]:
# ============================================================
# 9B. PROTT5 FOR bindEmbed21 PUBLIC DEV + OOF ADAPTER
# ============================================================
if RUN_OOF_THRESHOLDS:
    B21_DEV_H5=generate_prott5_h5(bind21_dev_df,CACHE/'bindEmbed21_DevSet1014_ProtT5.h5',REBUILD_PROTT5)
    oofdir=PRED/'bindEmbed21DL'/'DevOOF'; oofdir.mkdir(parents=True,exist_ok=True)
    script=B21/'run_dev_oof_fullprecision.py'
    script.write_text(f"""import sys\nfrom pathlib import Path\nimport numpy as np\nsys.path.insert(0,r"{B21}")\nfrom config import FileSetter,FileManager\nfrom data_preparation import ProteinInformation,ProteinResults\nfrom ml_predictor import MLPredictor\n# Disable the author code's 3-decimal probability rounding for full-precision common metrics.\ndef _set_full(self,predictions): self.predictions=np.asarray(predictions).T.astype(np.float64,copy=False)\nProteinResults.set_predictions=_set_full\nFileSetter.embeddings_input=staticmethod(lambda:r"{B21_DEV_H5}")\nemb=FileManager.read_embeddings(r"{B21_DEV_H5}")\nseqs=FileManager.read_fasta(r"{B21/'data/development_set/all.fasta'}")\nout=Path(r"{oofdir}");out.mkdir(parents=True,exist_ok=True)\nfor k in range(1,6):\n ids=FileManager.read_ids(r"{B21}/data/development_set/ids_split"+str(k)+".txt")\n model=FileManager.load_classifier_torch(r"{B21}/trained_models/checkpoint"+str(k)+".pt")\n # labels are not needed for inference; get_data_predictions creates compatible ProteinInformation objects.\n sequences,max_length,labels=ProteinInformation.get_data_predictions(ids,r"{B21/'data/development_set/all.fasta'}")\n pred=MLPredictor(model).predict_per_protein(ids,sequences,emb,labels,max_length)\n for pid,obj in pred.items():\n  arr=np.asarray(obj.predictions).squeeze()\n  assert arr.ndim==2 and arr.shape[1]==3,(pid,arr.shape)\n  np.save(out/f"{{pid}}.npy",arr.astype(np.float32))\nprint("done")\n""")
    mex('bind21',f'python "{script.name}"',cwd=B21)

REUSE /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/bindEmbed21_DevSet1014_ProtT5.h5
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n bind21 python "run_dev_oof_fullprecision.py"


In [ ]:
# ============================================================
# 9C. SELECT bindEmbed21 OOF THRESHOLDS
# ============================================================
bind21_thresholds=None
if RUN_OOF_THRESHOLDS:
    b21_oof_probs=load_three_class_npy(bind21_dev_df,PRED/'bindEmbed21DL'/'DevOOF')
    y_by={lab:np.concatenate([np.asarray(x,np.int8) for x in bind21_dev_df[lab]]) for lab in LABELS}
    p_by={lab:np.concatenate([np.asarray(b21_oof_probs[lab][pid],float) for pid in bind21_dev_df.prot_id]) for lab in LABELS}
    bind21_thresholds,b21_thr_df=thresholds_from_oof(y_by,p_by)
    display(b21_thr_df)
    b21_thr_df.to_csv(RESULTS/'bindEmbed21DL_OOF_thresholds.csv',index=False)

,label,OOF_F1_threshold,OOF_F1_at_threshold,OOF_MCC_at_F1_threshold,OOF_MCC_threshold,OOF_F1_at_MCC_threshold,OOF_MCC_at_threshold
0,any_ligand_binding_sites,0.548763,0.452147,0.400344,0.563823,0.452010,0.400491
1,metal_binding,0.789580,0.455627,0.456704,0.820055,0.453024,0.458759
2,small_binding,0.491117,0.394309,0.359113,0.435245,0.392177,0.361083
3,nuclear_binding,0.730035,0.477543,0.469293,0.703487,0.477241,0.469542


# 10. bindNode24: TestSet300 direct + TestSetNew46 new external application

### Native training/test protocol

- The original DevSet1014 lineage is reduced to DevSet1010 because four Dev proteins have sequence/structure length mismatches.
- Five predefined protein folds are used to train five graph models.
- Released inference averages the five model probabilities.
- Output channels: metal, nucleic, small; `any` = union/maximum.
- **Published/native final external evaluation is TestSet300.**

### TestSetNew46 nuance

The bindNode24 paper/repository does not establish TestSetNew46 as a native published benchmark for bindNode24. In this notebook we apply the frozen released bindNode24 ensemble to TestSetNew46 using project ProtT5 embeddings and project structures. This row must therefore be labelled **new external application**, not published reproduction.

In [ ]:
# ============================================================
# 10A. bindNode24 ENV — AUTHOR-COMPATIBLE EXACT STACK
# ============================================================
BN24 = REPOS/'bindnode24'

# First time after the old failed environment, leave this True.
# Once this cell succeeds, you can change it to False for later reruns.
RESET_BINDNODE24_ENV = True

if RUN_BINDNODE24 or RUN_OOF_THRESHOLDS:
    if RESET_BINDNODE24_ENV:
        p = env_path('bindnode24')
        if p.exists():
            print('Removing previous bindNode24 env:', p)
            sh(f'rm -rf "{p}"')

    create_env('bindnode24', '3.10')

    def _bn24_install(cmd, label):
        r = mex('bindnode24', cmd, check=False, capture=True)
        print(r.stdout)
        if r.returncode != 0:
            raise RuntimeError(
                f'bindNode24 {label} failed with return code {r.returncode}. '
                'Full installer output is printed immediately above.'
            )

    # Core author-compatible scientific stack.
    _bn24_install(
        'python -m pip install --upgrade '
        '"numpy==1.26.4" '
        '"pandas==2.2.0" '
        '"scipy==1.12.0" '
        '"scikit-learn==1.4.0" '
        '"h5py==3.10.0" '
        '"biopython==1.83" '
        '"fastapy==1.0.3" '
        '"colorama==0.4.6" '
        '"pathvalidate==3.2.0"',
        'core dependency installation'
    )

    # Authors used torch 2.2.0 + CUDA 11.8.
    _bn24_install(
        'python -m pip install --upgrade '
        '--index-url https://download.pytorch.org/whl/cu118 '
        '"torch==2.2.0" "torchvision==0.17.0" "torchaudio==2.2.0"',
        'PyTorch CUDA 11.8 installation'
    )

    # PyG compiled wheels must match torch 2.2.0 + cu118.
    _bn24_install(
        'python -m pip install --upgrade '
        '-f https://data.pyg.org/whl/torch-2.2.0+cu118.html '
        '"torch-scatter==2.1.2" '
        '"torch-sparse==0.6.18" '
        '"torch-cluster==1.6.3" '
        '"torch-spline-conv==1.2.2"',
        'PyG compiled extension installation'
    )

    _bn24_install(
        'python -m pip install --upgrade "torch-geometric==2.5.0"',
        'torch-geometric installation'
    )

    # Hard ABI/CUDA/model-layer smoke test.
    r = mex(
        'bindnode24',
        """python -c "
import torch, numpy, pandas, scipy, sklearn, h5py
import torch_geometric
import torch_scatter, torch_sparse, torch_cluster, torch_spline_conv
from torch_geometric.nn import SAGEConv
print('torch=', torch.__version__)
print('torch CUDA build=', torch.version.cuda)
print('cuda available=', torch.cuda.is_available())
print('torch_geometric=', torch_geometric.__version__)
print('numpy=', numpy.__version__)
device='cuda:0' if torch.cuda.is_available() else 'cpu'
x=torch.randn(4,8,device=device)
edge_index=torch.tensor([[0,1,2],[1,2,3]],dtype=torch.long,device=device)
m=SAGEConv(8,4).to(device)
y=m(x,edge_index)
print('SAGEConv smoke test=', tuple(y.shape), y.device)
" """,
        check=False,
        capture=True
    )
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(
            'bindNode24 environment smoke test failed. '
            'See complete output immediately above.'
        )

    print('bindNode24 environment: PASS')


+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba create -y -n bindnode24 -c conda-forge python=3.10 pip
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n bindnode24 python -m pip install --upgrade "numpy==1.26.4" "pandas==2.2.0" "scipy==1.12.0" "scikit-learn==1.4.0" "h5py==3.10.0" "biopython==1.83" "fastapy==1.0.3" "colorama==0.4.6" "pathvalidate==3.2.0"
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached h5py-3.10.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.5 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2

In [ ]:
# ============================================================
# 10B. FIND PROJECT STRUCTURES FOR TestSetNew46
# ============================================================
PDB_CANDIDATES=[BASE/'pdb/testset300',BASE/'pdb/testset46',BASE/'esmFold_pdb_files',BASE/'pdb/all']

def find_project_pdb(pid):
    for root in PDB_CANDIDATES:
        if not root.exists(): continue
        p=root/f'{pid}.pdb'
        if p.exists(): return p
        hit=list(root.rglob(f'{pid}.pdb'))
        if hit: return hit[0]
    return None

PROJECT_PDB={name:{pid:find_project_pdb(pid) for pid in df.prot_id} for name,df in DATASETS.items()}
for name,d in PROJECT_PDB.items():
    missing=[k for k,v in d.items() if v is None]
    print(name,'project PDB found',len(d)-len(missing),'/',len(d),'missing',missing[:5])

TestSet300 project PDB found 300 / 300 missing []
TestSetNew46 project PDB found 46 / 46 missing []


In [ ]:
# ============================================================
# 10C. PATCH OUTPUT PRECISION ONLY
# ============================================================
if RUN_BINDNODE24 or RUN_OOF_THRESHOLDS:
    writer=BN24/'src/output/outputFileWriting.py'
    txt=writer.read_text()
    if ':.3f' in txt:
        writer.write_text(txt.replace(':.3f',':.9f'))
        print('Patched text serialization 3 -> 9 decimals only.')

In [ ]:
# ============================================================
# 10D. GENERIC bindNode24 CONFIG + FULL-PRECISION ADAPTER
# ============================================================

def bindnode_config(
    result_dir,
    embeddings,
    id_list,
    fasta,
    pdb_dir,
    weight_dir,
    dssp_dir,
    cache_dir,
):
    """
    Prediction-only config matching the released bindNode24 architecture.

    use_cache=False is deliberate for external inference:
    it prevents stale distance matrices generated from a different structure
    directory from being silently reused.
    """
    return f"""[DEFAULT]
mode = predict

[FILE_PATHS]
result_dir = {result_dir}
embeddings = {embeddings}
id_list = {id_list}
cv_split_id_files_folder = {BN24/'dataset/cv_splits/predicted_structures'}
3d_structure_dir = {pdb_dir}
precomputed_DSSP_feature_dir = {dssp_dir}
fasta_file_path = {fasta}
cache_folder = {cache_dir}

[LABELS]
metal_label_file = {BN24/'dataset/binding_residues_2.5_metal.txt'}
small_label_file = {BN24/'dataset/binding_residues_2.5_small.txt'}
nuclear_label_file = {BN24/'dataset/binding_residues_2.5_nuclear.txt'}

[UTILITY]
do_logging = yes
use_cache = False

[MODEL]
weight_folder = {weight_dir}
model_type = SAGEConv
cutoff = 0.5
in_channels = 1024
nr_additional_features = 20
use_dssp = False
out_channels = 3

[DATASET]
cutoff_structure = 17

[OUTPUT]
write_ri = False

[MODEL_PARAMETERS]
features = 128
dropout = 0.7
epochs = 100
early_stopping = True
activation = F.leaky_relu
batchsize = 406
dropout_fcn = 0.8

[LOSS_FUNCTION_PARAMETERS]
weights = [8.9, 7.7, 4.4]

[OPTIMIZER_PARAMETERS]
learning_rate = 0.01
betas = [0.9, 0.999]
epsilon = 1e-8
weight_decay = 0.0
"""


def write_bindnode24_fullprecision_adapter(dataset, cfg, expected_ids, expected_lengths):
    """
    Run released bindNode24 models but bypass the authors' text writer,
    which rounds probabilities to 3 decimals.

    Model forward pass, graph construction, five checkpoints, and ensemble
    averaging are unchanged. Only serialization is replaced by .npy float32.
    """
    outdir = PRED/'bindNode24'/dataset/'fullprecision_npy'
    outdir.mkdir(parents=True, exist_ok=True)

    expected_json = WORK/f'bindNode24_{dataset}_expected.json'
    import json
    expected_json.write_text(json.dumps({
        'ids': [str(x) for x in expected_ids],
        'lengths': {str(k): int(v) for k,v in expected_lengths.items()}
    }))

    script = BN24/f'run_{dataset}_fullprecision.py'
    script.write_text(f"""import sys, json
from pathlib import Path
import numpy as np

REPO = Path(r"{BN24}")
sys.path.insert(0, str(REPO/'src'))

from setup.configProcessor import (
    validate_config, use_cache, get_model_parameter_dict, get_weight_dir
)
from misc.Cache import setup_chache
from machine_learning.predict import run_prediction
from data_processing.post_processing import get_averaged_predictions

OUT = Path(r"{outdir}")
EXPECTED = json.loads(Path(r"{expected_json}").read_text())

print("=== bindNode24 full-precision adapter preflight ===")
validate_config()

weights = sorted(Path(get_weight_dir()).glob('*.state_dict'))
print("released weight files:", [p.name for p in weights])
if len(weights) != 5:
    raise RuntimeError(f"Expected 5 bindNode24 state_dict files, found {{len(weights)}}")

if use_cache():
    setup_chache()

params = get_model_parameter_dict(only_first_value=True)
print("model parameters:", params)

predictions = run_prediction(params)
avg = get_averaged_predictions(predictions)

expected_ids = EXPECTED['ids']
lengths = EXPECTED['lengths']

missing = sorted(set(expected_ids) - set(avg))
extra = sorted(set(avg) - set(expected_ids))
if missing:
    raise RuntimeError(f"Missing predictions for {{len(missing)}} proteins: {{missing[:10]}}")
if extra:
    raise RuntimeError(f"Unexpected proteins in prediction dict: {{extra[:10]}}")

OUT.mkdir(parents=True, exist_ok=True)
for pid in expected_ids:
    arr = np.asarray(avg[pid], dtype=np.float64)
    L = int(lengths[pid])

    if arr.shape != (L,3):
        raise RuntimeError(
            f"{{pid}}: prediction shape {{arr.shape}}, expected {{(L,3)}}"
        )
    if not np.all(np.isfinite(arr)):
        raise RuntimeError(f"{{pid}}: non-finite probabilities")

    np.save(OUT/f"{{pid}}.npy", arr.astype(np.float32))

print("saved", len(expected_ids), "proteins to", OUT)
print("=== bindNode24 full-precision adapter PASS ===")
""")

    return script, outdir


def load_bindnode24_npy(df, folder):
    folder = Path(folder)
    metal, nuc, small = {}, {}, {}

    for r in df.itertuples():
        pid = str(r.prot_id)
        fp = folder/f'{pid}.npy'
        if not fp.exists():
            raise FileNotFoundError(fp)

        arr = np.load(fp)
        if arr.shape != (len(r.sequence),3):
            raise RuntimeError(
                f'{pid}: prediction shape {arr.shape}; '
                f'expected {(len(r.sequence),3)}'
            )
        if not np.all(np.isfinite(arr)):
            raise RuntimeError(f'{pid}: non-finite bindNode24 probability')

        metal[pid] = arr[:,0].astype(np.float64)
        nuc[pid]   = arr[:,1].astype(np.float64)
        small[pid] = arr[:,2].astype(np.float64)

    anyp = {
        pid: np.maximum.reduce([metal[pid],nuc[pid],small[pid]])
        for pid in metal
    }

    return {
        'metal_binding': metal,
        'nuclear_binding': nuc,
        'small_binding': small,
        'any_ligand_binding_sites': anyp,
    }


def run_or_recover_bindnode24(
    dataset,
    df,
    fasta,
    id_list,
    pdb_dir,
    dssp_dir,
):
    """
    Disk-state-safe bindNode24 inference.

    If all expected .npy predictions already exist with the correct shapes,
    inference is skipped. This makes Colab disconnects/restarts recoverable.
    """
    outdir = PRED/'bindNode24'/dataset/'fullprecision_npy'
    outdir.mkdir(parents=True, exist_ok=True)

    expected = {str(r.prot_id): len(r.sequence) for r in df.itertuples()}

    good = True
    for pid,L in expected.items():
        fp = outdir/f'{pid}.npy'
        if not fp.exists():
            good = False
            break
        try:
            a = np.load(fp, mmap_mode='r')
            if a.shape != (L,3):
                good = False
                break
        except Exception:
            good = False
            break

    if good and len(list(outdir.glob('*.npy'))) >= len(expected):
        print(f'{dataset}: all {len(expected)} bindNode24 predictions already exist -> SKIP inference')
        return outdir

    result = PRED/'bindNode24'/dataset
    result.mkdir(parents=True, exist_ok=True)

    cfg = WORK/f'bindNode24_{dataset}.ini'
    cfg.write_text(bindnode_config(
        result_dir=result,
        embeddings=PROTT5[dataset],
        id_list=id_list,
        fasta=fasta,
        pdb_dir=pdb_dir,
        weight_dir=BN24/'trained_model_dicts/bindNode24',
        dssp_dir=dssp_dir,
        cache_dir=CACHE/f'bindNode24_{dataset}_fresh',
    ))

    script, outdir = write_bindnode24_fullprecision_adapter(
        dataset=dataset,
        cfg=cfg,
        expected_ids=list(expected.keys()),
        expected_lengths=expected,
    )

    # `config.py` consumes -f from sys.argv when imported.
    r = mex(
        'bindnode24',
        f'python "{script.name}" -f "{cfg}"',
        cwd=BN24,
        check=False,
        capture=True,
    )

    print(r.stdout)
    print('return code:', r.returncode)

    if r.returncode != 0:
        raise RuntimeError(
            f'bindNode24 inference failed on {dataset}. '
            'The complete INNER traceback is printed immediately above.'
        )

    # Strict post-run completeness audit.
    missing=[]
    bad=[]
    for pid,L in expected.items():
        fp=outdir/f'{pid}.npy'
        if not fp.exists():
            missing.append(pid)
            continue
        a=np.load(fp,mmap_mode='r')
        if a.shape != (L,3):
            bad.append((pid,a.shape,L))

    if missing or bad:
        raise RuntimeError(
            f'{dataset}: incomplete bindNode24 output. '
            f'missing={missing[:10]}, bad={bad[:10]}'
        )

    print(f'{dataset}: bindNode24 output audit PASS ({len(expected)} proteins)')
    return outdir


In [ ]:
# ============================================================
# 10E. RUN / RECOVER bindNode24 ON TestSet300
#      (AUTHOR STRUCTURES, FULL-PRECISION PROBABILITIES)
# ============================================================
bind24_probs = {}

if RUN_BINDNODE24:
    out300 = run_or_recover_bindnode24(
        dataset='TestSet300',
        df=test300_df,
        fasta=BN24/'dataset/all.fasta',
        id_list=BN24/'dataset/TestSet300_ids.txt',
        pdb_dir=BN24/'dataset/AlphaFold_predictions',
        dssp_dir=BN24/'dataset/DSSP_features',
    )

    bind24_probs['TestSet300'] = load_bindnode24_npy(
        test300_df,
        out300
    )

    for label,d in bind24_probs['TestSet300'].items():
        save_probs(
            d,
            PRED/f'bindNode24__TestSet300__{label}.csv.gz'
        )

    print('bindNode24 TestSet300: COMPLETE')


TestSet300: all 300 bindNode24 predictions already exist -> SKIP inference
bindNode24 TestSet300: COMPLETE


In [ ]:
# ============================================================
# 10F. PREPARE TestSetNew46 STRUCTURES + DUMMY DSSP
#      THEN RUN / RECOVER bindNode24
# ============================================================
from copy import deepcopy
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.SeqUtils import seq1

AA3_BN24 = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
}

BN24_46_PDB = CACHE/'bindNode24_TestSetNew46_normalized_pdb'
BN24_46_DSSP = CACHE/'bindNode24_TestSetNew46_dummy_dssp'
BN24_46_PDB.mkdir(parents=True,exist_ok=True)
BN24_46_DSSP.mkdir(parents=True,exist_ok=True)


def _bn24_standard_residues(chain):
    return [
        r for r in chain
        if r.id[0]==' ' and r.resname.strip().upper() in AA3_BN24
    ]


def _bn24_seq(residues):
    return ''.join(seq1(r.resname.strip().upper()) for r in residues)


def _normalize_bn24_external_pdb(src, sequence, dst):
    parser=PDBParser(QUIET=True)
    structure=parser.get_structure('src',str(src))
    model=next(structure.get_models())

    matches=[]
    for chain in model:
        residues=_bn24_standard_residues(chain)
        if _bn24_seq(residues)==sequence:
            matches.append((chain,residues))

    if not matches:
        raise RuntimeError(
            f'{src}: no chain exactly matches canonical sequence length {len(sequence)}'
        )

    # deterministic: prefer source chain A
    matches=sorted(matches,key=lambda x:(0 if str(x[0].id)=='A' else 1,str(x[0].id)))
    chain,residues=matches[0]

    out_s=Structure('normalized')
    out_m=Model(0)
    out_c=Chain('A')
    out_s.add(out_m)
    out_m.add(out_c)

    for i,res in enumerate(residues,start=1):
        rr=deepcopy(res)
        rr.detach_parent()
        rr.id=(' ',i,' ')
        out_c.add(rr)

    io=PDBIO()
    io.set_structure(out_s)
    io.save(str(dst))

    # read-back exact sequence audit
    chk=PDBParser(QUIET=True).get_structure('chk',str(dst))
    cm=next(chk.get_models())
    seq2=_bn24_seq(_bn24_standard_residues(cm['A']))
    if seq2 != sequence:
        raise RuntimeError(f'{dst}: normalized read-back sequence mismatch')


def _write_dummy_dssp_csv(pid, L, dst):
    """
    bindNode24's dataset loader requests DSSP arrays even when use_dssp=False.
    For Test46 no author DSSP files exist, so make shape-correct dummy files.
    The released SAGEConv checkpoint ignores these channels because
    use_dssp=False; they do not enter the forward computation.
    """
    rows=[]
    for i in range(1,L+1):
        rows.append([
            i,       # absolute position
            'X',     # unused residue column
            'C',     # secondary-structure symbol
            0.0,     # solvent accessibility
            0.0,     # phi
            0.0,     # psi
            0,0.0,   # NH_O_1 index/energy
            0,0.0,   # O_NH_1 index/energy
            0,0.0,   # NH_O_2 index/energy
            0,0.0,   # O_NH_2 index/energy
        ])
    pd.DataFrame(rows).to_csv(dst,sep=';',header=False,index=False)


if RUN_BINDNODE24:
    # Build exact canonical single-chain structures.
    for r in test46_df.itertuples():
        pid=str(r.prot_id)

        src = PROJECT_PDB.get('TestSetNew46',{}).get(pid)
        if src is None or not Path(src).exists():
            raise FileNotFoundError(
                f'TestSetNew46/{pid}: project PDB not found'
            )

        dst=BN24_46_PDB/f'{pid}.pdb'
        _normalize_bn24_external_pdb(src,r.sequence,dst)

        dssp=BN24_46_DSSP/f'{pid}.csv'
        _write_dummy_dssp_csv(pid,len(r.sequence),dssp)

    assert len(list(BN24_46_PDB.glob('*.pdb'))) >= len(test46_df)
    assert len(list(BN24_46_DSSP.glob('*.csv'))) >= len(test46_df)

    out46 = run_or_recover_bindnode24(
        dataset='TestSetNew46',
        df=test46_df,
        fasta=CANON/'TestSetNew46.fasta',
        id_list=CANON/'TestSetNew46_ids.txt',
        pdb_dir=BN24_46_PDB,
        dssp_dir=BN24_46_DSSP,
    )

    bind24_probs['TestSetNew46'] = load_bindnode24_npy(
        test46_df,
        out46
    )

    for label,d in bind24_probs['TestSetNew46'].items():
        save_probs(
            d,
            PRED/f'bindNode24__TestSetNew46__{label}.csv.gz'
        )

    print('bindNode24 TestSetNew46: COMPLETE')


TestSetNew46: all 46 bindNode24 predictions already exist -> SKIP inference
bindNode24 TestSetNew46: COMPLETE


# 11. bindNode24 OOF thresholds

The public training code saves `trained_model_i.state_dict` after holding out CV split `i`. Therefore OOF reconstruction can be done without retraining:

1. expose only `trained_model_i` in a temporary weight folder;
2. predict only `ids_split(i+1)`;
3. repeat over 5 folds;
4. concatenate the held-out predictions;
5. choose F1- and MCC-optimal thresholds from these OOF scores.

In [ ]:
d# ============================================================
# 11A. PUBLIC DevSet1010 DATAFRAME FOR bindNode24
# ============================================================
def build_bn24_dev_df():
    seqs=read_fasta_dict(BN24/'dataset/all.fasta')
    folds={};ids=[]
    for k in range(1,6):
        z=[x.strip() for x in open(BN24/f'dataset/cv_splits/predicted_structures/ids_split{k}.txt') if x.strip()]
        ids+=z
        for x in z: folds[x]=k
    bind={lab:read_binding_file(BN24/f'dataset/binding_residues_2.5_{short}.txt') for lab,short in [
        ('metal_binding','metal'),('nuclear_binding','nuclear'),('small_binding','small')]}
    rows=[]
    for pid in ids:
        seq=seqs[pid]; rec={'prot_id':pid,'sequence':seq,'fold':folds[pid]}
        for lab in ['metal_binding','nuclear_binding','small_binding']:
            y=np.zeros(len(seq),dtype=np.int8)
            for pos in bind[lab].get(pid,[]):
                if 1<=pos<=len(seq): y[pos-1]=1
            rec[lab]=y.tolist()
        rec['any_ligand_binding_sites']=np.maximum.reduce([rec['metal_binding'],rec['nuclear_binding'],rec['small_binding']]).astype(int).tolist()
        rows.append(rec)
    return pd.DataFrame(rows)

bn24_dev_df=build_bn24_dev_df() if RUN_OOF_THRESHOLDS else None
if bn24_dev_df is not None:
    display(bn24_dev_df.groupby('fold').size())

,0
fold,
1,202
2,203
3,203
4,202
5,200


In [ ]:
# ============================================================
# 11B. Dev ProtT5 + SINGLE-FOLD bindNode24 OOF RUNS
#      LOCAL WEIGHT COPY — NO GOOGLE-DRIVE SYMLINKS
# ============================================================

import shutil
import json

bind24_thresholds = None

if RUN_OOF_THRESHOLDS:

    # --------------------------------------------------------
    # 1. DevSet1010 ProtT5
    # --------------------------------------------------------
    BN24_DEV_H5 = generate_prott5_h5(
        bn24_dev_df,
        CACHE / 'bindNode24_DevSet1010_ProtT5.h5',
        REBUILD_PROTT5
    )

    oofroot = PRED / 'bindNode24' / 'DevOOF_fullprecision'
    oofroot.mkdir(parents=True, exist_ok=True)

    # IMPORTANT:
    # do NOT put executable/symlink logic on Google Drive
    local_weight_root = Path('/content/bindnode24_oof_single_weights')
    local_weight_root.mkdir(parents=True, exist_ok=True)

    local_cache_root = Path('/content/bindnode24_oof_cache')
    local_cache_root.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Helper: one held-out fold
    # --------------------------------------------------------
    def run_bn24_oof_fold(k):

        fold_df = (
            bn24_dev_df[
                bn24_dev_df['fold'] == k
            ]
            .copy()
            .reset_index(drop=True)
        )

        expected = {
            str(r.prot_id): len(r.sequence)
            for r in fold_df.itertuples()
        }

        fold_out = oofroot / f'fold{k}'
        npy_out = fold_out / 'fullprecision_npy'
        npy_out.mkdir(parents=True, exist_ok=True)

        # ----------------------------------------------------
        # Recover existing complete predictions
        # ----------------------------------------------------
        complete = True

        for pid, L in expected.items():

            fp = npy_out / f'{pid}.npy'

            if not fp.exists():
                complete = False
                break

            try:
                a = np.load(
                    fp,
                    mmap_mode='r'
                )

                if a.shape != (L, 3):
                    complete = False
                    break

                if not np.all(
                    np.isfinite(a)
                ):
                    complete = False
                    break

            except Exception:
                complete = False
                break

        if complete:

            print(
                f'Fold {k}: '
                f'{len(expected)} valid OOF predictions '
                f'already exist -> SKIP inference'
            )

            return npy_out

        # ----------------------------------------------------
        # 2. LOCAL single-checkpoint folder
        # ----------------------------------------------------
        wdir = (
            local_weight_root /
            f'fold{k}'
        )

        if wdir.exists():
            shutil.rmtree(wdir)

        wdir.mkdir(
            parents=True,
            exist_ok=True
        )

        src = (
            BN24 /
            'trained_model_dicts' /
            'bindNode24' /
            f'trained_model_{k-1}.state_dict'
        )

        if not src.exists():

            raise FileNotFoundError(
                f'Missing released checkpoint: {src}'
            )

        dst = (
            wdir /
            src.name
        )

        # COPY, not symlink
        shutil.copy2(
            src,
            dst
        )

        copied = list(
            wdir.glob(
                '*.state_dict'
            )
        )

        if len(copied) != 1:

            raise RuntimeError(
                f'Fold {k}: expected exactly '
                f'1 checkpoint; found '
                f'{len(copied)}'
            )

        if (
            copied[0].stat().st_size
            !=
            src.stat().st_size
        ):

            raise RuntimeError(
                f'Fold {k}: copied '
                f'checkpoint size mismatch'
            )

        print(
            f'Fold {k}: local checkpoint =',
            copied[0]
        )

        # ----------------------------------------------------
        # 3. Config
        # ----------------------------------------------------
        result = (
            fold_out /
            'native_result'
        )

        result.mkdir(
            parents=True,
            exist_ok=True
        )

        cfg = (
            WORK /
            f'bindNode24_DevOOF_fold{k}.ini'
        )

        cfg.write_text(
            bindnode_config(
                result_dir=result,

                embeddings=
                    BN24_DEV_H5,

                id_list=(
                    BN24 /
                    'dataset' /
                    'cv_splits' /
                    'predicted_structures' /
                    f'ids_split{k}.txt'
                ),

                fasta=
                    BN24 /
                    'dataset' /
                    'all.fasta',

                pdb_dir=
                    BN24 /
                    'dataset' /
                    'AlphaFold_predictions',

                weight_dir=
                    wdir,

                dssp_dir=
                    BN24 /
                    'dataset' /
                    'DSSP_features',

                cache_dir=
                    local_cache_root /
                    f'fold{k}'
            )
        )

        # ----------------------------------------------------
        # 4. Expected proteins
        # ----------------------------------------------------
        expected_json = (
            WORK /
            f'bindNode24_DevOOF_fold{k}_expected.json'
        )

        expected_json.write_text(
            json.dumps({
                'ids':
                    list(
                        expected.keys()
                    ),

                'lengths':
                    expected
            })
        )

        # ----------------------------------------------------
        # 5. Full-precision prediction adapter
        # ----------------------------------------------------
        script = (
            BN24 /
            f'run_DevOOF_fold{k}_fullprecision.py'
        )

        script.write_text(
f"""import sys
import json

from pathlib import Path

import numpy as np

REPO = Path(r"{BN24}")

sys.path.insert(
    0,
    str(REPO / 'src')
)

from setup.configProcessor import (
    validate_config,
    use_cache,
    get_model_parameter_dict,
    get_weight_dir
)

from misc.Cache import (
    setup_chache
)

from machine_learning.predict import (
    run_prediction
)

from data_processing.post_processing import (
    get_averaged_predictions
)


OUT = Path(r"{npy_out}")

EXPECTED = json.loads(
    Path(
        r"{expected_json}"
    ).read_text()
)


print(
    "=== bindNode24 OOF "
    "fold {k} preflight ==="
)

validate_config()

weights = sorted(
    Path(
        get_weight_dir()
    ).glob(
        '*.state_dict'
    )
)

print(
    "weight files:",
    [p.name for p in weights]
)

if len(weights) != 1:

    raise RuntimeError(
        f"OOF fold {k}: "
        f"expected exactly "
        f"1 checkpoint, "
        f"found {{len(weights)}}"
    )


if use_cache():
    setup_chache()


params = (
    get_model_parameter_dict(
        only_first_value=True
    )
)

print(
    "model parameters:",
    params
)


predictions = run_prediction(
    params
)

avg = get_averaged_predictions(
    predictions
)


expected_ids = (
    EXPECTED['ids']
)

lengths = (
    EXPECTED['lengths']
)


missing = sorted(
    set(expected_ids)
    -
    set(avg)
)

extra = sorted(
    set(avg)
    -
    set(expected_ids)
)


if missing:

    raise RuntimeError(
        f"OOF fold {k}: "
        f"missing predictions "
        f"for {{len(missing)}} "
        f"proteins: "
        f"{{missing[:10]}}"
    )


if extra:

    raise RuntimeError(
        f"OOF fold {k}: "
        f"unexpected proteins: "
        f"{{extra[:10]}}"
    )


OUT.mkdir(
    parents=True,
    exist_ok=True
)


for pid in expected_ids:

    arr = np.asarray(
        avg[pid],
        dtype=np.float64
    )

    L = int(
        lengths[pid]
    )

    if arr.shape != (
        L,
        3
    ):

        raise RuntimeError(
            f"{{pid}}: "
            f"prediction shape "
            f"{{arr.shape}}, "
            f"expected "
            f"{{(L,3)}}"
        )

    if not np.all(
        np.isfinite(arr)
    ):

        raise RuntimeError(
            f"{{pid}}: "
            f"non-finite "
            f"OOF probabilities"
        )

    np.save(
        OUT /
        f"{{pid}}.npy",

        arr.astype(
            np.float32
        )
    )


print(
    "saved",
    len(expected_ids),
    "OOF proteins"
)

print(
    "=== bindNode24 OOF "
    "fold {k} PASS ==="
)
"""
        )

        # ----------------------------------------------------
        # 6. Run
        # ----------------------------------------------------
        r = mex(
            'bindnode24',

            f'python '
            f'"{script.name}" '
            f'-f "{cfg}"',

            cwd=BN24,

            check=False,
            capture=True
        )

        print(
            r.stdout
        )

        print(
            f'fold {k} return code:',
            r.returncode
        )

        if r.returncode != 0:

            raise RuntimeError(
                f'bindNode24 OOF fold '
                f'{k} failed. '
                f'The complete inner '
                f'traceback is printed '
                f'above.'
            )

        # ----------------------------------------------------
        # 7. Post-run audit
        # ----------------------------------------------------
        missing = []
        bad = []

        for pid, L in (
            expected.items()
        ):

            fp = (
                npy_out /
                f'{pid}.npy'
            )

            if not fp.exists():
                missing.append(pid)
                continue

            a = np.load(
                fp,
                mmap_mode='r'
            )

            if a.shape != (
                L,
                3
            ):

                bad.append(
                    (
                        pid,
                        a.shape,
                        L
                    )
                )

        if missing or bad:

            raise RuntimeError(
                f'Fold {k}: '
                f'incomplete OOF output; '
                f'missing={missing[:10]}, '
                f'bad={bad[:10]}'
            )

        print(
            f'Fold {k}: '
            f'OOF output audit PASS '
            f'({len(expected)} proteins)'
        )

        return npy_out


    # --------------------------------------------------------
    # 8. Run all 5 folds
    # --------------------------------------------------------
    BN24_OOF_DIRS = {}

    for k in range(1, 6):

        print(
            '\n' +
            '=' * 72
        )

        print(
            f'BINDNODE24 OOF '
            f'FOLD {k}/5'
        )

        print(
            '=' * 72
        )

        BN24_OOF_DIRS[k] = (
            run_bn24_oof_fold(k)
        )


    # --------------------------------------------------------
    # 9. Reassemble all Dev predictions
    # --------------------------------------------------------
    metal = {}
    nuc = {}
    small = {}

    for r in (
        bn24_dev_df.itertuples()
    ):

        pid = str(
            r.prot_id
        )

        fold = int(
            r.fold
        )

        fp = (
            BN24_OOF_DIRS[fold] /
            f'{pid}.npy'
        )

        if not fp.exists():

            raise FileNotFoundError(
                f'Missing OOF '
                f'probability for '
                f'{pid}: {fp}'
            )

        arr = np.load(
            fp
        )

        if arr.shape != (
            len(r.sequence),
            3
        ):

            raise RuntimeError(
                f'{pid}: OOF shape '
                f'{arr.shape}; '
                f'expected '
                f'{(len(r.sequence),3)}'
            )

        metal[pid] = (
            arr[:,0]
            .astype(
                np.float64
            )
        )

        nuc[pid] = (
            arr[:,1]
            .astype(
                np.float64
            )
        )

        small[pid] = (
            arr[:,2]
            .astype(
                np.float64
            )
        )


    if len(metal) != (
        len(bn24_dev_df)
    ):

        raise RuntimeError(
            f'Expected OOF '
            f'predictions for '
            f'{len(bn24_dev_df)} '
            f'Dev proteins; '
            f'got {len(metal)}'
        )


    anyp = {
        pid:
        np.maximum.reduce([
            metal[pid],
            nuc[pid],
            small[pid]
        ])
        for pid in metal
    }


    oofprobs = {

        'metal_binding':
            metal,

        'nuclear_binding':
            nuc,

        'small_binding':
            small,

        'any_ligand_binding_sites':
            anyp
    }


    # --------------------------------------------------------
    # 10. Concatenate residues
    # --------------------------------------------------------
    y_by = {

        lab:
        np.concatenate([
            np.asarray(
                x,
                dtype=np.int8
            )
            for x
            in bn24_dev_df[lab]
        ])

        for lab in LABELS
    }


    p_by = {

        lab:
        np.concatenate([
            np.asarray(
                oofprobs[
                    lab
                ][
                    str(pid)
                ],
                dtype=np.float64
            )

            for pid
            in bn24_dev_df.prot_id
        ])

        for lab in LABELS
    }


    for lab in LABELS:

        if (
            len(y_by[lab])
            !=
            len(p_by[lab])
        ):

            raise RuntimeError(
                f'{lab}: '
                f'OOF label/probability '
                f'length mismatch '
                f'{len(y_by[lab])} '
                f'!= '
                f'{len(p_by[lab])}'
            )


    # --------------------------------------------------------
    # 11. Threshold selection from Dev OOF only
    # --------------------------------------------------------
    bind24_thresholds, bn24_thr_df = (
        thresholds_from_oof(
            y_by,
            p_by
        )
    )

    display(
        bn24_thr_df
    )

    bn24_thr_df.to_csv(
        RESULTS /
        'bindNode24_OOF_thresholds.csv',

        index=False
    )

    print(
        '\nbindNode24 OOF '
        'THRESHOLD RECONSTRUCTION: '
        'COMPLETE'
    )

REUSE /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/bindNode24_DevSet1010_ProtT5.h5

BINDNODE24 OOF FOLD 1/5
Fold 1: local checkpoint = /content/bindnode24_oof_single_weights/fold1/trained_model_0.state_dict
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n bindnode24 python "run_DevOOF_fold1_fullprecision.py" -f "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/bindNode24_DevOOF_fold1.ini"
Checking for alternative file path to .ini file
Used config filepath is: /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/bindNode24_DevOOF_fold1.ini
=== bindNode24 OOF fold 1 preflight ===
Config file is valid
weight files: ['trained_model_0.state_dict']
model parameters: {'cutoff': 0.5, 'in_channels': 1024, 'out_channels': 3, 'features': 128, 'dropout': 0.7, 'epochs': 100, 'early_stopping': True, 'weights': [8.9, 7.7, 4.4], 'cutoff_structure': 17, 'activation': 'F.leaky_relu', 'batchs

,label,OOF_F1_threshold,OOF_F1_at_threshold,OOF_MCC_at_F1_threshold,OOF_MCC_threshold,OOF_F1_at_MCC_threshold,OOF_MCC_at_threshold
0,any_ligand_binding_sites,0.497651,0.475025,0.425661,0.528093,0.474938,0.425816
1,metal_binding,0.725768,0.476631,0.482650,0.731446,0.475846,0.482718
2,small_binding,0.492627,0.424740,0.391016,0.475196,0.423814,0.391194
3,nuclear_binding,0.642428,0.513232,0.506523,0.580189,0.510658,0.508089



bindNode24 OOF THRESHOLD RECONSTRUCTION: COMPLETE


# 12. Full direct-baseline evaluation on BOTH external sets

For each method/task/dataset, the notebook creates rows for:

- `fixed_0.5`
- `OOF_F1` if reconstructed
- `OOF_MCC` if reconstructed

Threshold-free AP/AUPRC and AUROC are identical across those operating-point rows; threshold-dependent metrics differ.

In [ ]:
# ============================================================
# 12A. SCORE bindEmbed21DL + bindNode24
# ============================================================
all_rows=[]; per_protein_frames=[]

def add_method(method,all_probs,thresholds,tier_by_dataset):
    for dataset,df in DATASETS.items():
        if dataset not in all_probs: continue
        for label in LABELS:
            ops=[('fixed_0.5',.5)]
            if thresholds is not None:
                ops += [('OOF_F1',thresholds[label]['OOF_F1']),('OOF_MCC',thresholds[label]['OOF_MCC'])]
            for opname,t in ops:
                out,pt=evaluate_probability_dict(df,all_probs[dataset][label],label,method,dataset,opname,t,tier_by_dataset[dataset])
                all_rows.append(out)
                pt.insert(0,'operating_point',opname);pt.insert(0,'label',label);pt.insert(0,'dataset',dataset);pt.insert(0,'method',method)
                per_protein_frames.append(pt)

if RUN_BIND21:
    add_method('bindEmbed21DL',bind21_probs,bind21_thresholds,
               {'TestSet300':'A_direct_native','TestSetNew46':'A_direct_native'})
if RUN_BINDNODE24:
    add_method('bindNode24',bind24_probs,bind24_thresholds,
               {'TestSet300':'A_direct_native','TestSetNew46':'C_external_application_project_structures'})

full_direct=pd.DataFrame(all_rows)
full_direct.to_csv(RESULTS/'FULL_DIRECT_BASELINE_METRICS_300_46.csv',index=False)
if per_protein_frames:
    pd.concat(per_protein_frames,ignore_index=True).to_csv(RESULTS/'FULL_DIRECT_PER_PROTEIN_300_46.csv',index=False)

display(full_direct.sort_values(['dataset','label','method','operating_point']))

,method,dataset,label,comparison_tier,operating_point,n_residues,n_positive,n_negative,prevalence,AP_AUPRC,...,protein_F1_bootstrap95_low,protein_F1_bootstrap95_high,protein_mean_MCC,protein_MCC_CI95_legacy_halfwidth,protein_MCC_CI95_standard_halfwidth,protein_MCC_bootstrap95_low,protein_MCC_bootstrap95_high,protein_mean_accuracy,protein_accuracy_CI95_legacy_halfwidth,protein_accuracy_CI95_standard_halfwidth
1,bindEmbed21DL,TestSet300,any_ligand_binding_sites,A_direct_native,OOF_F1,62689,5869,56820,0.093621,0.452821,...,0.403834,0.452767,0.403454,0.024842,0.024884,0.378691,0.427939,0.896226,0.008152,0.008166
2,bindEmbed21DL,TestSet300,any_ligand_binding_sites,A_direct_native,OOF_MCC,62689,5869,56820,0.093621,0.452821,...,0.400260,0.448608,0.400831,0.024671,0.024712,0.375915,0.425022,0.897469,0.008145,0.008159
0,bindEmbed21DL,TestSet300,any_ligand_binding_sites,A_direct_native,fixed_0.5,62689,5869,56820,0.093621,0.452821,...,0.410093,0.458410,0.405981,0.024496,0.024537,0.381751,0.430766,0.890687,0.008068,0.008082
25,bindNode24,TestSet300,any_ligand_binding_sites,A_direct_native,OOF_F1,62689,5869,56820,0.093621,0.472592,...,0.415628,0.466693,0.421100,0.025671,0.025714,0.395816,0.447088,0.896233,0.009055,0.009070
26,bindNode24,TestSet300,any_ligand_binding_sites,A_direct_native,OOF_MCC,62689,5869,56820,0.093621,0.472592,...,0.409990,0.462454,0.418329,0.026172,0.026216,0.392641,0.444354,0.900349,0.009143,0.009159
24,bindNode24,TestSet300,any_ligand_binding_sites,A_direct_native,fixed_0.5,62689,5869,56820,0.093621,0.472592,...,0.415211,0.466964,0.420644,0.025845,0.025889,0.396089,0.446310,0.896407,0.009072,0.009087
4,bindEmbed21DL,TestSet300,metal_binding,A_direct_native,OOF_F1,62689,881,61808,0.014054,0.307221,...,0.228022,0.331552,0.290163,0.052278,0.052425,0.235917,0.342428,0.976892,0.005435,0.005450
5,bindEmbed21DL,TestSet300,metal_binding,A_direct_native,OOF_MCC,62689,881,61808,0.014054,0.307221,...,0.233484,0.337355,0.296949,0.054031,0.054196,0.243644,0.351234,0.975433,0.005858,0.005876
3,bindEmbed21DL,TestSet300,metal_binding,A_direct_native,fixed_0.5,62689,881,61808,0.014054,0.307221,...,0.180285,0.262779,0.222255,0.040934,0.041018,0.183181,0.267393,0.974864,0.004084,0.004092
28,bindNode24,TestSet300,metal_binding,A_direct_native,OOF_F1,62689,881,61808,0.014054,0.321109,...,0.246819,0.355256,0.309715,0.055656,0.055824,0.257312,0.366550,0.976408,0.005698,0.005716


# 13. Reproduction guards against published-scale values

A run that silently drifts away from the authors' published per-protein F1/MCC should **not** be accepted just because the code completed.

The guard checks fixed-0.5 per-protein mean values against known published-scale values.

- bindEmbed21DL: TestSet300 and TestSetNew46
- bindNode24: TestSet300 only

bindNode24 TestSetNew46 is a new application using project structures, so it has no native-public reproduction target here.

In [ ]:
# ============================================================
# 13A. PUBLISHED-SCALE REPRODUCTION CHECK
# ============================================================
published=pd.DataFrame([
    # dataset,label,method,F1,MCC
    ['TestSet300','any_ligand_binding_sites','bindEmbed21DL',.43,.41],
    ['TestSet300','metal_binding','bindEmbed21DL',.22,.22],
    ['TestSet300','nuclear_binding','bindEmbed21DL',.24,.22],
    ['TestSet300','small_binding','bindEmbed21DL',.33,.31],
    ['TestSetNew46','any_ligand_binding_sites','bindEmbed21DL',.37,np.nan],
    ['TestSetNew46','metal_binding','bindEmbed21DL',.26,np.nan],
    ['TestSetNew46','nuclear_binding','bindEmbed21DL',.19,np.nan],
    ['TestSetNew46','small_binding','bindEmbed21DL',.29,np.nan],
    ['TestSet300','any_ligand_binding_sites','bindNode24',.44,.42],
    ['TestSet300','metal_binding','bindNode24',.26,.26],
    ['TestSet300','nuclear_binding','bindNode24',.26,.25],
    ['TestSet300','small_binding','bindNode24',.35,.34],
],columns=['dataset','label','method','published_protein_F1','published_protein_MCC'])

fixed=full_direct[full_direct.operating_point.eq('fixed_0.5')].copy()
chk=fixed.merge(published,on=['dataset','label','method'],how='inner')
chk['delta_F1']=chk['protein_mean_F1']-chk['published_protein_F1']
chk['delta_MCC']=chk['protein_mean_MCC']-chk['published_protein_MCC']
chk['F1_within_tolerance']=chk.delta_F1.abs()<=REPRO_TOLERANCE_ABS
chk['MCC_within_tolerance']=chk.apply(lambda r: True if pd.isna(r.published_protein_MCC) else abs(r.delta_MCC)<=REPRO_TOLERANCE_ABS,axis=1)
display(chk[['dataset','label','method','protein_mean_F1','published_protein_F1','delta_F1','F1_within_tolerance',
             'protein_mean_MCC','published_protein_MCC','delta_MCC','MCC_within_tolerance']])
chk.to_csv(RESULTS/'REPRODUCTION_GUARD.csv',index=False)

bad=chk[~(chk.F1_within_tolerance & chk.MCC_within_tolerance)]
if len(bad):
    raise RuntimeError(
        'Reproduction guard failed. DO NOT REPORT these numbers yet. Inspect ProtT5 generation, sequence mapping, '
        'checkpoint mapping, and label convention.\n'+bad[['dataset','label','method','delta_F1','delta_MCC']].to_string(index=False)
    )
print('Published-scale reproduction guard PASSED.')

,dataset,label,method,protein_mean_F1,published_protein_F1,delta_F1,F1_within_tolerance,protein_mean_MCC,published_protein_MCC,delta_MCC,MCC_within_tolerance
0,TestSet300,any_ligand_binding_sites,bindEmbed21DL,0.433813,0.43,0.003813,True,0.405981,0.41,-0.004019,True
1,TestSet300,metal_binding,bindEmbed21DL,0.219101,0.22,-0.000899,True,0.222255,0.22,0.002255,True
2,TestSet300,small_binding,bindEmbed21DL,0.331199,0.33,0.001199,True,0.307604,0.31,-0.002396,True
3,TestSet300,nuclear_binding,bindEmbed21DL,0.238822,0.24,-0.001178,True,0.224839,0.22,0.004839,True
4,TestSetNew46,any_ligand_binding_sites,bindEmbed21DL,0.371654,0.37,0.001654,True,0.348555,NaN,NaN,True
5,TestSetNew46,metal_binding,bindEmbed21DL,0.261565,0.26,0.001565,True,0.260439,NaN,NaN,True
6,TestSetNew46,small_binding,bindEmbed21DL,0.285816,0.29,-0.004184,True,0.253609,NaN,NaN,True
7,TestSetNew46,nuclear_binding,bindEmbed21DL,0.193721,0.19,0.003721,True,0.192283,NaN,NaN,True
8,TestSet300,any_ligand_binding_sites,bindNode24,0.440962,0.44,0.000962,True,0.420644,0.42,0.000644,True
9,TestSet300,metal_binding,bindNode24,0.257202,0.26,-0.002798,True,0.262066,0.26,0.002066,True


Published-scale reproduction guard PASSED.


# 14. Paper-ready compact views

The full CSV contains many columns. These views isolate the metrics most likely to appear in the manuscript.

### Main fixed-threshold table

- AP/AUPRC
- AUROC
- global residue F1@0.5
- global residue MCC@0.5
- per-protein mean F1 ± historical 95% CI
- per-protein mean MCC ± historical 95% CI

### Supplementary OOF operating-point table

Separately reports metrics at the F1-optimal and MCC-optimal OOF thresholds. This avoids mixing operating points in the main literature comparison.

**Never relabel** `global_F1/global_MCC` as macro. They are residue-global (micro/global) metrics. `protein_mean_F1/protein_mean_MCC` are per-protein means under the macro-positive inclusion rule.

In [ ]:
# ============================================================
# 14A. COMPACT TABLES
# ============================================================
main_cols=['dataset','label','method','comparison_tier','AP_AUPRC','AUROC','Brier',
           'global_positive_F1','global_MCC','global_precision_PPV','global_recall_sensitivity','global_specificity','negative_class_F1','class_macro_F1','class_weighted_F1','class_micro_F1',
           'protein_mean_F1','protein_F1_CI95_legacy_halfwidth','protein_mean_MCC','protein_MCC_CI95_legacy_halfwidth',
           'protein_macro_positive_n','protein_false_alarm_only_n','protein_TN_only_excluded_n']
main=fixed[main_cols].copy()
main.to_csv(RESULTS/'PAPER_READY_FIXED_0_5_300_46.csv',index=False)
display(main.sort_values(['dataset','label','method']))

supp=full_direct[full_direct.operating_point.ne('fixed_0.5')].copy()
supp.to_csv(RESULTS/'SUPPLEMENTARY_OOF_OPERATING_POINTS_300_46.csv',index=False)

,dataset,label,method,comparison_tier,AP_AUPRC,AUROC,Brier,global_positive_F1,global_MCC,global_precision_PPV,...,class_macro_F1,class_weighted_F1,class_micro_F1,protein_mean_F1,protein_F1_CI95_legacy_halfwidth,protein_mean_MCC,protein_MCC_CI95_legacy_halfwidth,protein_macro_positive_n,protein_false_alarm_only_n,protein_TN_only_excluded_n
0,TestSet300,any_ligand_binding_sites,bindEmbed21DL,A_direct_native,0.452821,0.844042,0.074519,0.465306,0.411348,0.474495,...,0.705616,0.900930,0.901785,0.433813,0.024303,0.405981,0.024496,300,0,0
24,TestSet300,any_ligand_binding_sites,bindNode24,A_direct_native,0.472592,0.851460,0.072007,0.480626,0.431027,0.508755,...,0.715032,0.905547,0.907847,0.440962,0.026111,0.420644,0.025845,300,0,0
3,TestSet300,metal_binding,bindEmbed21DL,A_direct_native,0.307221,0.823064,0.017540,0.360673,0.351863,0.335945,...,0.675412,0.981306,0.980603,0.219101,0.040262,0.222255,0.040934,248,124,52
27,TestSet300,metal_binding,bindNode24,A_direct_native,0.321109,0.833192,0.015705,0.402652,0.395457,0.429306,...,0.697321,0.983708,0.984192,0.257202,0.046286,0.262066,0.046866,219,95,81
9,TestSet300,nuclear_binding,bindEmbed21DL,A_direct_native,0.371532,0.921621,0.018738,0.378126,0.379045,0.501687,...,0.683101,0.973772,0.976599,0.238822,0.058816,0.224839,0.055620,90,24,210
33,TestSet300,nuclear_binding,bindNode24,A_direct_native,0.400676,0.943253,0.017891,0.383162,0.386394,0.519814,...,0.685746,0.974139,0.977093,0.261493,0.069144,0.248807,0.065380,72,6,228
6,TestSet300,small_binding,bindEmbed21DL,A_direct_native,0.394041,0.864302,0.057017,0.435777,0.396702,0.411270,...,0.697871,0.927304,0.925234,0.331199,0.031701,0.307604,0.030989,284,64,16
30,TestSet300,small_binding,bindNode24,A_direct_native,0.431709,0.869662,0.054359,0.468245,0.432216,0.458425,...,0.716043,0.932961,0.932285,0.353203,0.033338,0.337026,0.032599,273,53,27
12,TestSetNew46,any_ligand_binding_sites,bindEmbed21DL,A_direct_native,0.296174,0.817433,0.107472,0.377254,0.307553,0.313725,...,0.647853,0.868478,0.855789,0.371654,0.060660,0.348555,0.062693,46,0,0
36,TestSetNew46,any_ligand_binding_sites,bindNode24,C_external_application_project_structures,0.323975,0.825435,0.100823,0.393917,0.326488,0.337469,...,0.659164,0.875425,0.865585,0.371669,0.077100,0.350863,0.078705,46,0,0


# 15. Optional test-oracle threshold diagnostic — NOT FOR CLAIMS

This section answers a calibration question only: “what is the best possible F1/MCC threshold on this test set?”

Because it uses test labels to select a threshold, these values **must never be used as headline performance**. They are saved to a separate file whose name contains `NOT_FOR_CLAIMS`.

In [ ]:
# ============================================================
# 15A. TEST-ORACLE DIAGNOSTICS
# ============================================================
oracle_rows=[]
for method,probs,tier in [
    ('bindEmbed21DL',globals().get('bind21_probs',{}),'diagnostic'),
    ('bindNode24',globals().get('bind24_probs',{}),'diagnostic'),
]:
    for dataset,df in DATASETS.items():
        if dataset not in probs: continue
        for label in LABELS:
            y=np.concatenate([np.asarray(v,np.int8) for v in df[label]])
            p=np.concatenate([np.asarray(probs[dataset][label][pid],float) for pid in df.prot_id])
            tf1=choose_f1_threshold(y,p);tmcc=choose_mcc_threshold(y,p)
            for op,t in [('TEST_ORACLE_F1',tf1),('TEST_ORACLE_MCC',tmcc)]:
                z=global_threshold_metrics(y,p,t);z.update({'method':method,'dataset':dataset,'label':label,'operating_point':op})
                oracle_rows.append(z)
oracle=pd.DataFrame(oracle_rows)
oracle.to_csv(RESULTS/'DIAGNOSTIC_TEST_ORACLE_NOT_FOR_CLAIMS.csv',index=False)

# 16. Shared external-structure normalization for ESMBind + MetalBind

Both metal-specific methods need a protein structure, but they have different parsers. To avoid hidden chain mismatches, v6 creates one strict single-chain structure for each external protein:

1. search the project structure locations already used in this project;
2. for TestSet300, fall back to the bindNode24 released predicted structure if needed;
3. find a PDB chain whose 20-standard-amino-acid sequence exactly equals the canonical CSV sequence;
4. retain only standard amino-acid residues;
5. rename the chain to **A**;
6. renumber residues **1..L** without changing coordinates;
7. re-read the written PDB and assert exact sequence equality.

If no exact chain exists, the pipeline stops. It never aligns-and-drops residues silently.

In [ ]:
# ============================================================
# 16A. NORMALIZE ALL EXTERNAL STRUCTURES TO EXACT CANONICAL CHAIN A
# ============================================================
from copy import deepcopy
from Bio.PDB import PDBParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.SeqUtils import seq1

AA3 = {
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
}

METAL_EXTERNAL_STRUCT_ROOT = CACHE/'metal_external_normalized_structures'
METAL_EXTERNAL_STRUCT_ROOT.mkdir(parents=True,exist_ok=True)

def _standard_chain_residues(chain):
    return [r for r in chain if r.id[0]==' ' and r.resname.strip().upper() in AA3]

def _residue_seq(residues):
    return ''.join(seq1(r.resname.strip().upper()) for r in residues)

def resolve_external_structure(dataset,pid):
    p=PROJECT_PDB.get(dataset,{}).get(str(pid))
    if p is not None and Path(p).exists():
        return Path(p)
    if dataset=='TestSet300':
        q=REPOS/'bindnode24'/'dataset'/'AlphaFold_predictions'/f'{pid}.pdb'
        if q.exists():
            return q
    return None

def normalize_pdb_exact_chain_A(src,target_sequence,dst):
    """
    Select an exact-sequence chain and write a one-chain canonical PDB.
    Homomers are valid: prefer chain A, else deterministic lexical chain id.
    """
    src=Path(src); dst=Path(dst)
    parser=PDBParser(QUIET=True)
    structure=parser.get_structure('src',str(src))
    model=next(structure.get_models())

    exact=[]
    diagnostic=[]
    for chain in model:
        residues=_standard_chain_residues(chain)
        seq=_residue_seq(residues)
        diagnostic.append((str(chain.id),len(seq),seq[:20]))
        if seq==target_sequence:
            exact.append((chain,residues))

    if len(exact)==0:
        raise RuntimeError(
            f'{src}: no chain exactly matches canonical sequence length {len(target_sequence)}. '
            f'Chains={diagnostic}'
        )

    exact=sorted(exact,key=lambda x:(0 if str(x[0].id)=='A' else 1,str(x[0].id)))
    chain,residues=exact[0]
    source_chain=str(chain.id)
    n_exact_matches=len(exact)

    out_structure=Structure('normalized')
    out_model=Model(0)
    out_chain=Chain('A')
    out_structure.add(out_model)
    out_model.add(out_chain)

    for i,res in enumerate(residues,start=1):
        rr=deepcopy(res)
        rr.detach_parent()
        rr.id=(' ',i,' ')
        out_chain.add(rr)

    dst.parent.mkdir(parents=True,exist_ok=True)
    io=PDBIO()
    io.set_structure(out_structure)
    io.save(str(dst))

    chk=PDBParser(QUIET=True).get_structure('chk',str(dst))
    cm=next(chk.get_models())
    if 'A' not in cm:
        raise RuntimeError(f'{dst}: normalized chain A missing after write')
    residues2=_standard_chain_residues(cm['A'])
    seq2=_residue_seq(residues2)
    if seq2!=target_sequence:
        raise RuntimeError(f'{dst}: read-back sequence mismatch after normalization')
    if len(residues2)!=len(target_sequence):
        raise RuntimeError(f'{dst}: residue count mismatch after normalization')

    return dst,source_chain,n_exact_matches

NORMALIZED_EXTERNAL_PDB={}
structure_audit=[]

for dataset,df in DATASETS.items():
    NORMALIZED_EXTERNAL_PDB[dataset]={}
    droot=METAL_EXTERNAL_STRUCT_ROOT/dataset
    droot.mkdir(parents=True,exist_ok=True)

    for r in df.itertuples():
        pid=str(r.prot_id)
        src=resolve_external_structure(dataset,pid)
        if src is None:
            raise FileNotFoundError(f'{dataset}/{pid}: no external PDB found')

        dst=droot/f'{pid}.pdb'
        dst,source_chain,n_exact_matches=normalize_pdb_exact_chain_A(src,r.sequence,dst)

        NORMALIZED_EXTERNAL_PDB[dataset][pid]=dst
        structure_audit.append([
            dataset,pid,str(src),source_chain,n_exact_matches,
            str(dst),len(r.sequence),'PASS'
        ])

structure_audit_df=pd.DataFrame(
    structure_audit,
    columns=[
        'dataset','prot_id','source_pdb','selected_source_chain',
        'n_exact_sequence_matching_chains','normalized_pdb','length','status'
    ]
)
structure_audit_df.to_csv(
    RESULTS/'metal_specific_structure_normalization_audit.csv',
    index=False
)
display(structure_audit_df.groupby(['dataset','status']).size().reset_index(name='n'))

for dataset,df in DATASETS.items():
    assert len(NORMALIZED_EXTERNAL_PDB[dataset])==len(df)
    for r in df.itertuples():
        p=NORMALIZED_EXTERNAL_PDB[dataset][str(r.prot_id)]
        if not Path(p).exists():
            raise FileNotFoundError(p)

print('External structure normalization complete for all 300 + all 46 proteins.')


,dataset,status,n
0,TestSet300,PASS,300
1,TestSetNew46,PASS,46


External structure normalization complete for all 300 + all 46 proteins.


# 17. ESMBind — one-click 300 + 46, metal only

The public ESMBind inference artifact uses:

- ESM-2 `esm2_t33_650M_UR50D`, 1280-d per residue;
- ESM-IF1 `esm_if1_gvp4_t16_142M_UR50`, 512-d per residue;
- five released fold models;
- seven ion heads: `MG, FE, CU, CO, CA, MN, ZN`.

v6 runs the authors' inference script itself, checks all feature lengths/dimensions, preserves all seven raw ion probabilities, then forms

\[
p_{metal}=\max(p_{MG},p_{FE},p_{CU},p_{CO},p_{CA},p_{MN},p_{ZN}).
\]

Every one of the 300/46 proteins must succeed.

In [ ]:
# ============================================================
# CELL 1 — FINAL ESM-IF1 ENVIRONMENT FIX + SMOKE TEST
# ============================================================

# 1) Pin Biotite to the historical API expected by fair-esm ESM-IF1
rr = mex(
    'esmbind',
    r"""python - <<'PY'
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "numpy<2",
    "biotite==0.41.2"
])

print("Biotite 0.41.2 install PASS")
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

if rr.returncode != 0:
    raise RuntimeError(
        f"Biotite installation failed: return code {rr.returncode}"
    )


# 2) Ensure torch_scatter matches the existing PyTorch/CUDA build
rr = mex(
    'esmbind',
    r"""python - <<'PY'
import sys
import subprocess
import torch

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)

try:
    import torch_scatter
    print("Existing torch_scatter:", torch_scatter.__version__)
except Exception:
    cuda = torch.version.cuda

    if cuda is None:
        tag = "cpu"
    elif cuda.startswith("12.4"):
        tag = "cu124"
    elif cuda.startswith("12.1"):
        tag = "cu121"
    elif cuda.startswith("11.8"):
        tag = "cu118"
    else:
        raise RuntimeError(
            f"Unsupported CUDA for automatic torch_scatter wheel: {cuda}"
        )

    wheel_url = (
        f"https://data.pyg.org/whl/"
        f"torch-2.4.0+{tag}.html"
    )

    print("Installing torch_scatter from:", wheel_url)

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch-scatter==2.1.2",
        "-f",
        wheel_url
    ])

    import torch_scatter
    print("Installed torch_scatter:", torch_scatter.__version__)
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

if rr.returncode != 0:
    raise RuntimeError(
        f"torch_scatter setup failed: return code {rr.returncode}"
    )


# 3) Final dependency + model-load smoke test
rr = mex(
    'esmbind',
    r"""python - <<'PY'
import torch
import torch_scatter
import torch_geometric
import biotite

from biotite.structure import filter_backbone

import esm
import esm.inverse_folding

print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("torch_scatter:", torch_scatter.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("biotite:", biotite.__version__)
print("filter_backbone import: PASS")

assert torch.cuda.is_available(), "CUDA unavailable"

model, alphabet = (
    esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
)

model = model.eval().cuda()

print("ESM-IF1 MODEL LOAD PASS")
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

if rr.returncode != 0:
    raise RuntimeError(
        f"ESM-IF1 smoke test failed: return code {rr.returncode}"
    )

print("\nFINAL ESM-IF ENVIRONMENT: PASS")

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python - <<'PY'
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "numpy<2",
    "biotite==0.41.2"
])

print("Biotite 0.41.2 install PASS")
PY
Biotite 0.41.2 install PASS

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python - <<'PY'
import sys
import subprocess
import torch

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)

try:
    import torch_scatter
    print("Existing torch_scatter:", torch_scatter.__version__)
except Exception:
    cuda = torch.version.cuda

    if cuda is None:
        tag = "cpu"
    elif cuda.startswith("12.4"):
        tag = "cu124"
    elif cuda.startswith("12.1"):
        tag = "cu121"
    elif cuda.startswith("11.8"):
        tag = "cu118"
    else:
        raise RuntimeError(
            f"Unsupported CUDA 

In [ ]:
# ============================================================
# 17A. ESMBind ISOLATED ENVIRONMENT
# ============================================================
ESMB=REPOS/'esmbind'

if RUN_ESMBIND_TRANSFER:
    create_env('esmbind','3.10')
    mex('esmbind',"python -m pip install -q --upgrade 'pip<25' setuptools wheel")
    mex('esmbind',
        "python -m pip install -q 'numpy<2' scipy scikit-learn biopython tqdm "
        "ml-collections 'torchmetrics<1.5' 'torch==2.4.1' 'fair-esm==2.0.0'"
    )
    mex('esmbind',
        "python -c \"import torch,esm,torchmetrics,ml_collections; "
        "print('torch',torch.__version__,'cuda',torch.cuda.is_available()); "
        "assert torch.cuda.is_available(), 'ESMBind feature extraction is configured for GPU'\""
    )

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba create -y -n esmbind -c conda-forge python=3.10 pip
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python -m pip install -q --upgrade 'pip<25' setuptools wheel
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python -m pip install -q 'numpy<2' scipy scikit-learn biopython tqdm ml-collections 'torchmetrics<1.5' 'torch==2.4.1' 'fair-esm==2.0.0'
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python -c "import torch,esm,torchmetrics,ml_collections; print('torch',torch.__version__,'cuda',torch.cuda.is_available()); assert torch.cuda.is_available(), 'ESMBind feature extraction is configured for GPU'"


In [ ]:
# ============================================================
# CELL 2 / 17B
# ESMBind FEATURE + OFFICIAL INFERENCE HELPER — FINAL
# ============================================================

def _write_esmbind_fasta(df, path):
    """
    Full FASTA for ESM-IF and final ESMBind inference.
    Keep ALL protein IDs, including exact duplicate sequences.
    """
    with open(path, 'w') as f:
        for r in df.itertuples():
            f.write(
                f'>sp|{r.prot_id}|{r.prot_id}\n'
                f'{r.sequence}\n'
            )


def _valid_npy(path, shape):
    path = Path(path)

    if not path.exists():
        return False

    try:
        a = np.load(path, mmap_mode='r')

        return (
            a.shape == shape
            and np.all(np.isfinite(a))
        )

    except Exception:
        return False


def _prepare_unique_esm2_fasta(
    dataset,
    df,
    inp,
    rebuild=False
):
    """
    Official ESMBind ESM2 extractor rejects duplicate sequences.

    ESM2 is sequence-only, therefore exact duplicate amino-acid
    sequences legitimately share the same ESM2 representation.

    We:
      1. retain ALL benchmark proteins;
      2. generate ESM2 once per unique sequence;
      3. broadcast the embedding back to every original protein ID.
    """

    esm_dir = inp / 'esm'
    esm_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    seq_groups = {}

    for r in df.itertuples():
        seq_groups.setdefault(
            str(r.sequence),
            []
        ).append(
            str(r.prot_id)
        )

    duplicate_groups = {
        seq: ids
        for seq, ids
        in seq_groups.items()
        if len(ids) > 1
    }

    print(
        f'{dataset}: '
        f'proteins={len(df)}, '
        f'unique_sequences={len(seq_groups)}, '
        f'duplicate_groups={len(duplicate_groups)}'
    )

    reps_to_generate = []

    for seq, ids in seq_groups.items():

        expected = (
            len(seq),
            1280
        )

        source = None

        if not rebuild:

            for pid in ids:

                fp = (
                    esm_dir
                    /
                    f'{pid}.npy'
                )

                if _valid_npy(
                    fp,
                    expected
                ):
                    source = fp
                    break

        if source is not None:

            arr = np.load(
                source
            ).astype(
                np.float32,
                copy=False
            )

            for pid in ids:

                dst = (
                    esm_dir
                    /
                    f'{pid}.npy'
                )

                if not _valid_npy(
                    dst,
                    expected
                ):
                    np.save(
                        dst,
                        arr
                    )

        else:

            reps_to_generate.append(
                (
                    ids[0],
                    seq,
                    ids
                )
            )

    unique_fasta = (
        inp
        /
        f'{dataset}_unique_for_esm2.fasta'
    )

    with open(
        unique_fasta,
        'w'
    ) as f:

        for (
            rep_pid,
            seq,
            _
        ) in reps_to_generate:

            f.write(
                f'>sp|{rep_pid}|{rep_pid}\n'
                f'{seq}\n'
            )

    return (
        unique_fasta,
        reps_to_generate
    )


def _broadcast_generated_esm2(
    dataset,
    inp,
    reps_to_generate
):

    esm_dir = (
        inp
        /
        'esm'
    )

    for (
        rep_pid,
        seq,
        ids
    ) in reps_to_generate:

        src = (
            esm_dir
            /
            f'{rep_pid}.npy'
        )

        expected = (
            len(seq),
            1280
        )

        if not _valid_npy(
            src,
            expected
        ):
            raise RuntimeError(
                f'{dataset}/{rep_pid}: '
                f'representative ESM2 missing/invalid '
                f'after official extraction; '
                f'expected {expected}'
            )

        arr = np.load(
            src
        ).astype(
            np.float32,
            copy=False
        )

        for pid in ids:

            if pid == rep_pid:
                continue

            dst = (
                esm_dir
                /
                f'{pid}.npy'
            )

            np.save(
                dst,
                arr
            )


def _audit_esmbind_features(
    dataset,
    df,
    inp
):

    bad = []

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        e1 = (
            inp
            /
            'esm'
            /
            f'{pid}.npy'
        )

        e2 = (
            inp
            /
            'esm_if'
            /
            f'{pid}.npy'
        )

        if (
            not e1.exists()
            or
            not e2.exists()
        ):
            bad.append(
                (
                    pid,
                    'missing feature file'
                )
            )

            continue

        a = np.load(
            e1,
            mmap_mode='r'
        )

        b = np.load(
            e2,
            mmap_mode='r'
        )

        if a.shape != (
            len(r.sequence),
            1280
        ):
            bad.append(
                (
                    pid,
                    f'ESM2 {a.shape}'
                )
            )

        elif not np.all(
            np.isfinite(a)
        ):
            bad.append(
                (
                    pid,
                    'ESM2 non-finite'
                )
            )

        if b.shape != (
            len(r.sequence),
            512
        ):
            bad.append(
                (
                    pid,
                    f'ESMIF {b.shape}'
                )
            )

        elif not np.all(
            np.isfinite(b)
        ):
            bad.append(
                (
                    pid,
                    'ESMIF non-finite'
                )
            )

    if bad:
        raise RuntimeError(
            f'{dataset}: '
            f'ESMBind feature audit failed: '
            f'{bad[:10]}'
        )


def run_esmbind_transfer(
    dataset,
    df
):

    print(
        '\n'
        +
        '=' * 72
    )

    print(
        'ESMBind:',
        dataset
    )

    print(
        '=' * 72
    )

    inp = (
        CACHE
        /
        f'esmbind_{dataset}'
    )

    for sub in [
        'esm',
        'esm_if',
        'pdb'
    ]:
        (
            inp
            /
            sub
        ).mkdir(
            parents=True,
            exist_ok=True
        )

    # --------------------------------------------------------
    # FULL FASTA
    # --------------------------------------------------------

    fasta = (
        inp
        /
        f'{dataset}_esmbind.fasta'
    )

    _write_esmbind_fasta(
        df,
        fasta
    )

    # --------------------------------------------------------
    # EXACT NORMALIZED STRUCTURES
    # --------------------------------------------------------

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        src = (
            NORMALIZED_EXTERNAL_PDB[
                dataset
            ][
                pid
            ]
        )

        dst = (
            inp
            /
            'pdb'
            /
            f'{pid}.pdb'
        )

        if not dst.exists():

            shutil.copy2(
                src,
                dst
            )

    # ========================================================
    # ESM2 650M
    # ========================================================

    (
        unique_fasta,
        reps_to_generate
    ) = _prepare_unique_esm2_fasta(
        dataset,
        df,
        inp,
        rebuild=REBUILD_ESMBIND_ESM2
    )

    if reps_to_generate:

        too_long = [
            (
                pid,
                len(seq)
            )
            for (
                pid,
                seq,
                _
            )
            in reps_to_generate
            if len(seq) > 1022
        ]

        if too_long:

            raise RuntimeError(
                f'{dataset}: sequence(s) >1022 aa '
                f'exceed ESM2 t33 650M limit: '
                f'{too_long[:10]}'
            )

        print(
            f'{dataset}: '
            f'generating ESM2 for '
            f'{len(reps_to_generate)} '
            f'unique missing sequence(s)'
        )

        rr = mex(
            'esmbind',
            (
                f'python '
                f'multi_modal_binding/'
                f'get_esm_embedding.py '
                f'"{unique_fasta}" '
                f'"{inp/"esm"}" '
                f'--batch_size 1 '
                f'--device cuda'
            ),
            cwd=ESMB,
            check=False,
            capture=True
        )

        if rr.stdout:
            print(
                rr.stdout
            )

        if rr.returncode != 0:

            raise RuntimeError(
                f'{dataset}: '
                f'ESM2 generation failed '
                f'with return code '
                f'{rr.returncode}. '
                f'Full subprocess log '
                f'printed above.'
            )

        _broadcast_generated_esm2(
            dataset,
            inp,
            reps_to_generate
        )

    bad_esm2 = []

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        fp = (
            inp
            /
            'esm'
            /
            f'{pid}.npy'
        )

        if not _valid_npy(
            fp,
            (
                len(r.sequence),
                1280
            )
        ):
            bad_esm2.append(
                pid
            )

    if bad_esm2:

        raise RuntimeError(
            f'{dataset}: '
            f'missing/invalid ESM2: '
            f'{bad_esm2[:10]}'
        )

    print(
        f'{dataset}: '
        f'ESM2 complete for '
        f'{len(df)}/{len(df)} proteins.'
    )

    # ========================================================
    # ESM-IF1
    # ========================================================

    failed = (
        inp
        /
        'esm_if'
        /
        'failed_ids.txt'
    )

    need_esmif = (
        REBUILD_ESMBIND_ESMIF
        or
        any(
            not _valid_npy(
                (
                    inp
                    /
                    'esm_if'
                    /
                    f'{r.prot_id}.npy'
                ),
                (
                    len(r.sequence),
                    512
                )
            )
            for r
            in df.itertuples()
        )
    )

    if need_esmif:

        if failed.exists():
            failed.unlink()

        rr = mex(
            'esmbind',
            (
                f'python '
                f'multi_modal_binding/'
                f'get_esm_if_embedding.py '
                f'"{fasta}" '
                f'"{inp/"esm_if"}" '
                f'"{inp/"pdb"}" '
                f'--chain_id A'
            ),
            cwd=ESMB,
            check=False,
            capture=True
        )

        if rr.stdout:
            print(
                rr.stdout
            )

        if rr.returncode != 0:

            raise RuntimeError(
                f'{dataset}: '
                f'ESM-IF process failed '
                f'with return code '
                f'{rr.returncode}. '
                f'Full subprocess log '
                f'printed above.'
            )

    if failed.exists():

        bad = [
            x.strip()
            for x
            in failed.read_text().splitlines()
            if x.strip()
        ]

        if bad:

            raise RuntimeError(
                f'{dataset}: '
                f'ESM-IF failed proteins '
                f'cannot be dropped: '
                f'{bad[:10]}'
            )

    _audit_esmbind_features(
        dataset,
        df,
        inp
    )

    print(
        f'{dataset}: '
        f'ESM2 + ESM-IF feature audit PASS.'
    )

    # ========================================================
    # OFFICIAL 5-MODEL ESMBind ENSEMBLE
    # ========================================================

    conf = {
        'general': {
            'seed': 1,
            'gpu_id': 0,
            'debug': False
        },
        'inference': {
            'ensemble_path': str(
                ESMB
                /
                'multi_modal_binding'
                /
                'model'
                /
                'trained_weights'
            ),
            'fasta_path': str(
                fasta
            ),
            'precomputed_feature': str(
                inp
            ),
        }
    }

    confp = (
        WORK
        /
        f'esmbind_{dataset}.json'
    )

    confp.write_text(
        json.dumps(
            conf,
            indent=2
        )
    )

    started = time.time()

    rr = mex(
        'esmbind',
        (
            f'python '
            f'multi_modal_binding/'
            f'inference.py '
            f'--config "{confp}"'
        ),
        cwd=ESMB,
        check=False,
        capture=True
    )

    if rr.stdout:
        print(
            rr.stdout
        )

    if rr.returncode != 0:

        raise RuntimeError(
            f'{dataset}: '
            f'official ESMBind inference '
            f'failed with return code '
            f'{rr.returncode}. '
            f'Full subprocess log printed above.'
        )

    cand = [
        p
        for p
        in ESMB.rglob(
            'predictions.pkl'
        )
        if p.stat().st_mtime
        >=
        started - 2
    ]

    cand = sorted(
        cand,
        key=lambda p:
        p.stat().st_mtime,
        reverse=True
    )

    if not cand:

        raise RuntimeError(
            f'{dataset}: '
            f'no fresh predictions.pkl produced'
        )

    out_copy = (
        PRED
        /
        'ESMBind'
        /
        dataset
    )

    out_copy.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        cand[0],
        out_copy
        /
        'predictions.pkl'
    )

    with open(
        cand[0],
        'rb'
    ) as f:
        z = pickle.load(
            f
        )

    ions = [
        'MG',
        'FE',
        'CU',
        'CO',
        'CA',
        'MN',
        'ZN'
    ]

    for key in (
        ions
        +
        [
            'threshold_f1',
            'threshold_mcc'
        ]
    ):

        if key not in z:

            raise KeyError(
                f'{dataset}: '
                f'ESMBind output missing '
                f'key {key}'
            )

    ion_probs = {
        ion: {}
        for ion
        in ions
    }

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        for ion in ions:

            a = np.asarray(
                z[ion][pid],
                dtype=float
            )

            if a.shape != (
                len(r.sequence),
            ):

                raise RuntimeError(
                    f'{dataset}/{pid}/{ion}: '
                    f'prediction shape '
                    f'{a.shape}'
                )

            if (
                not np.all(
                    np.isfinite(a)
                )
                or
                np.min(a) < 0
                or
                np.max(a) > 1
            ):

                raise RuntimeError(
                    f'{dataset}/{pid}/{ion}: '
                    f'invalid probabilities'
                )

            ion_probs[
                ion
            ][
                pid
            ] = a

    # Save raw ion probabilities
    for ion, d in ion_probs.items():

        save_probs(
            d,
            out_copy
            /
            f'{ion}.csv.gz'
        )

    # ========================================================
    # PRE-SPECIFIED POOLED METAL SCORE
    # ========================================================

    pooled = {}

    for r in df.itertuples():

        pid = str(
            r.prot_id
        )

        pooled[
            pid
        ] = np.maximum.reduce(
            [
                ion_probs[
                    ion
                ][
                    pid
                ]
                for ion
                in ions
            ]
        )

    thresholds = {
        'threshold_f1': {
            k: float(v)
            for k, v
            in z[
                'threshold_f1'
            ].items()
        },
        'threshold_mcc': {
            k: float(v)
            for k, v
            in z[
                'threshold_mcc'
            ].items()
        },
    }

    (
        out_copy
        /
        'author_native_ion_thresholds.json'
    ).write_text(
        json.dumps(
            thresholds,
            indent=2
        )
    )

    print(
        f'{dataset}: '
        f'official ESMBind inference PASS.'
    )

    return (
        pooled,
        ion_probs,
        thresholds
    )

print("17B helper definitions loaded.")

17B helper definitions loaded.


In [ ]:
# ============================================================
# CELL 3 / 17C
# RUN + FULL-SCORE ESMBind ON ALL 300 + ALL 46
# ============================================================

esmb_rows = []

esmbind_pooled_probs = {}

esmbind_ion_probs = {}

esmbind_author_thresholds = {}


if RUN_ESMBIND_TRANSFER:

    for dataset, df in DATASETS.items():

        print(
            '\nSTART ESMBind external evaluation:',
            dataset
        )

        (
            pooled,
            ion_probs,
            thresholds
        ) = run_esmbind_transfer(
            dataset,
            df
        )

        esmbind_pooled_probs[
            dataset
        ] = pooled

        esmbind_ion_probs[
            dataset
        ] = ion_probs

        esmbind_author_thresholds[
            dataset
        ] = thresholds


        # ----------------------------------------------------
        # Save pooled raw probabilities
        # ----------------------------------------------------

        save_probs(
            pooled,
            PRED
            /
            (
                f'ESMBind_7ion_max_TRANSFER'
                f'__{dataset}'
                f'__metal_binding.csv.gz'
            )
        )


        # ----------------------------------------------------
        # COMMON EVALUATOR — fixed threshold 0.5
        # ----------------------------------------------------

        out, pt = evaluate_probability_dict(
            df,
            pooled,
            'metal_binding',
            'ESMBind_7ion_max_TRANSFER',
            dataset,
            'fixed_0.5',
            0.5,
            'C_cross_benchmark_transfer'
        )

        esmb_rows.append(
            out
        )


        # ----------------------------------------------------
        # Per-protein results
        # ----------------------------------------------------

        pt.to_csv(
            RESULTS
            /
            f'ESMBind_{dataset}_per_protein.csv',
            index=False
        )


        print(
            'FINISHED ESMBind external evaluation:',
            dataset
        )


    # --------------------------------------------------------
    # Combined Test300 + Test46 table
    # --------------------------------------------------------

    esmb_df = pd.DataFrame(
        esmb_rows
    )

    esmb_df.to_csv(
        RESULTS
        /
        'ESMBind_TRANSFER_FULL_METRICS_300_46.csv',
        index=False
    )

    display(
        esmb_df
    )


    # --------------------------------------------------------
    # Hard completion guard
    # --------------------------------------------------------

    expected = set(
        DATASETS.keys()
    )

    done = set(
        esmb_df[
            'dataset'
        ].astype(
            str
        )
    )

    if done != expected:

        raise RuntimeError(
            f'ESMBind completion mismatch: '
            f'expected {expected}, '
            f'got {done}'
        )


    print(
        '\nESMBind Test300 + Test46 FULL-SCORE PASS.'
    )


START ESMBind external evaluation: TestSet300

ESMBind: TestSet300
TestSet300: proteins=300, unique_sequences=298, duplicate_groups=2
TestSet300: ESM2 complete for 300/300 proteins.
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python multi_modal_binding/get_esm_if_embedding.py "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/esmbind_TestSet300/TestSet300_esmbind.fasta" "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/esmbind_TestSet300/esm_if" "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/esmbind_TestSet300/pdb" --chain_id A
/content/micromamba_binding_baselines/envs/esmbind/lib/python3.10/site-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(
2026-09-09 07:46:36,589 - INFO - Successfully generated embedding for A0L5S6
2026-09-09 07:46:36,783 - INFO 

,method,dataset,label,comparison_tier,operating_point,n_residues,n_positive,n_negative,prevalence,AP_AUPRC,...,protein_F1_bootstrap95_low,protein_F1_bootstrap95_high,protein_mean_MCC,protein_MCC_CI95_legacy_halfwidth,protein_MCC_CI95_standard_halfwidth,protein_MCC_bootstrap95_low,protein_MCC_bootstrap95_high,protein_mean_accuracy,protein_accuracy_CI95_legacy_halfwidth,protein_accuracy_CI95_standard_halfwidth
0,ESMBind_7ion_max_TRANSFER,TestSet300,metal_binding,C_cross_benchmark_transfer,fixed_0.5,62689,881,61808,0.014054,0.343212,...,0.223054,0.312806,0.270833,0.048038,0.048144,0.225861,0.316958,0.977842,0.004364,0.004373
1,ESMBind_7ion_max_TRANSFER,TestSetNew46,metal_binding,C_cross_benchmark_transfer,fixed_0.5,6227,77,6150,0.012366,0.615024,...,0.175016,0.473476,0.320249,0.158163,0.161100,0.175660,0.477338,0.980975,0.009004,0.009171



ESMBind Test300 + Test46 FULL-SCORE PASS.


In [ ]:
# ============================================================
# ESM-IF1 ONE-PROTEIN FULL TRACE
# DO NOT run full 17C again yet
# ============================================================

dataset = "TestSet300"
pid = "A0L5S6"

inp = CACHE / f"esmbind_{dataset}"
pdb_path = inp / "pdb" / f"{pid}.pdb"

rr = mex(
    'esmbind',
    f"""python - <<'PY'
import torch
import esm
import esm.inverse_folding

pdb_path = r"{pdb_path}"

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

model, alphabet = esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
model = model.eval().cuda()

print("model device:", next(model.parameters()).device)

structure = esm.inverse_folding.util.load_structure(
    pdb_path,
    "A"
)

coords, seq = (
    esm.inverse_folding.util.extract_coords_from_structure(
        structure
    )
)

print("sequence length:", len(seq))
print("coords shape:", coords.shape)

# IMPORTANT:
# no try/except here -> full traceback
rep = esm.inverse_folding.util.get_encoder_output(
    model,
    alphabet,
    coords
)

print("REP:", rep.shape, rep.device)
print("ONE PROTEIN ESM-IF PASS")
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

print("return code:", rr.returncode)

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python - <<'PY'
import torch
import esm
import esm.inverse_folding

pdb_path = r"/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/cache/esmbind_TestSet300/pdb/A0L5S6.pdb"

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

model, alphabet = esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
model = model.eval().cuda()

print("model device:", next(model.parameters()).device)

structure = esm.inverse_folding.util.load_structure(
    pdb_path,
    "A"
)

coords, seq = (
    esm.inverse_folding.util.extract_coords_from_structure(
        structure
    )
)

print("sequence length:", len(seq))
print("coords shape:", coords.shape)

# IMPORTANT:
# no try/except here -> full traceback
rep = esm.inverse_folding.util.get_encoder_output(
    model,
    alphabet,
    coords
)

print("REP:", rep.shape, rep.device)
print("ONE 

In [ ]:
# ============================================================
# HOTFIX FINAL — fair-esm v2.0.0 ESM-IF1 DEVICE BUG
# ============================================================

rr = mex(
    'esmbind',
    r"""python - <<'PY'
from pathlib import Path
import esm.inverse_folding.util as util

p = Path(util.__file__)
text = p.read_text()

marker = "# ESMIF_DEVICE_PATCH_V2"

if marker not in text:
    patch_lines = [
        "",
        "",
        marker,
        "def get_encoder_output(model, alphabet, coords):",
        "    device = next(model.parameters()).device",
        "    batch_converter = CoordBatchConverter(alphabet)",
        "    batch = [(coords, None, None)]",
        "    coords, confidence, _, _, padding_mask = batch_converter(",
        "        batch, device=device",
        "    )",
        "    encoder_out = model.encoder.forward(",
        "        coords,",
        "        padding_mask,",
        "        confidence,",
        "        return_all_hiddens=False",
        "    )",
        "    return encoder_out['encoder_out'][0][1:-1, 0]",
        "",
    ]

    p.write_text(
        text + "\n".join(patch_lines)
    )

    print("Applied ESM-IF device patch to:")
    print(p)

else:
    print("ESM-IF device patch already present:")
    print(p)
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

if rr.returncode != 0:
    raise RuntimeError(
        f"ESM-IF device patch failed: return code {rr.returncode}"
    )


# ============================================================
# ONE-PROTEIN REAL SMOKE TEST
# ============================================================

dataset = "TestSet300"
pid = "A0L5S6"

inp = CACHE / f"esmbind_{dataset}"
pdb_path = inp / "pdb" / f"{pid}.pdb"

rr = mex(
    'esmbind',
    f"""python - <<'PY'
import torch
import esm
import esm.inverse_folding

pdb_path = r"{pdb_path}"

model, alphabet = (
    esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
)

model = model.eval().cuda()

print("model device:", next(model.parameters()).device)

structure = esm.inverse_folding.util.load_structure(
    pdb_path,
    "A"
)

coords, seq = (
    esm.inverse_folding.util.extract_coords_from_structure(
        structure
    )
)

print("sequence length:", len(seq))
print("coords shape:", coords.shape)

rep = esm.inverse_folding.util.get_encoder_output(
    model,
    alphabet,
    coords
)

print("representation shape:", tuple(rep.shape))
print("representation device:", rep.device)

assert tuple(rep.shape) == (len(seq), 512)

print("ONE PROTEIN ESM-IF PASS")
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

if rr.returncode != 0:
    raise RuntimeError(
        f"ESM-IF one-protein smoke test failed: return code {rr.returncode}"
    )

print("\nESM-IF DEVICE FIX: PASS")

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmbind python - <<'PY'
from pathlib import Path
import esm.inverse_folding.util as util

p = Path(util.__file__)
text = p.read_text()

marker = "# ESMIF_DEVICE_PATCH_V2"

if marker not in text:
    patch_lines = [
        "",
        "",
        marker,
        "def get_encoder_output(model, alphabet, coords):",
        "    device = next(model.parameters()).device",
        "    batch_converter = CoordBatchConverter(alphabet)",
        "    batch = [(coords, None, None)]",
        "    coords, confidence, _, _, padding_mask = batch_converter(",
        "        batch, device=device",
        "    )",
        "    encoder_out = model.encoder.forward(",
        "        coords,",
        "        padding_mask,",
        "        confidence,",
        "        return_all_hiddens=False",
        "    )",
        "    return encoder_out['encoder_out'][0][1:-1, 0]",
        "",
    ]

    p.write_text(
        te

# 18. MetalBind — one-click 300 + 46, metal only

MetalBind is the heaviest branch because the released model predicts on a molecular surface rather than directly on every sequence position.

v6 performs the following automatically:

1. create an isolated **Python 3.7** author-compatible environment;
2. install **MSMS 2.6.1**, **Reduce**, PyTorch 1.10.1/CUDA 11.3, PyG 2.0.4, PyKeOps, and the released PyMesh 0.3 wheel;
3. patch only the authors' hard-coded MSMS/Reduce binary paths;
4. generate **ESM-C 600M, 1152-d** final-layer residue representations in a separate modern environment;
5. use the authors' `MetalBindProcessor` surface code with label-dependent routines replaced by zero-label inference stubs only;
6. preserve the authors' MSMS → mesh-fix → tangent-frame → atom/pKa/ESM feature path;
7. run each of the **13 released `best.pth` checkpoints**;
8. map predicted surface-residue probabilities back to the full sequence. Non-surface residues receive probability 0 because the released site model only emits probabilities for its surface-residue set;
9. pool 13 ion probabilities by residue-wise maximum;
10. score all TestSet300/TestSetNew46 residues against `metal_binding` with the same common evaluator.

If any protein fails structure matching, ESM-C dimension checks, surface preprocessing, checkpoint loading, or full-length remapping, the branch fails loudly and produces no MetalBind comparison row.

In [ ]:
# ============================================================
# MetalBind ENV DIAGNOSTIC — CURRENT SESSION
# Do NOT rerun 18A yet
# ============================================================

tests = {
    "python": "python --version",

    "torch": r"""python - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.tensor([1.0], device="cuda")
    print("CUDA tensor test:", x, x.device)
PY""",

    "pymesh": r"""python - <<'PY'
import pymesh
print("pymesh import PASS")
print("pymesh path:", pymesh.__file__)
PY""",

    "torch_scatter": r"""python - <<'PY'
import torch_scatter
print("torch_scatter:", torch_scatter.__version__)
PY""",

    "torch_sparse": r"""python - <<'PY'
import torch_sparse
print("torch_sparse:", torch_sparse.__version__)
PY""",

    "torch_cluster": r"""python - <<'PY'
import torch_cluster
print("torch_cluster:", torch_cluster.__version__)
PY""",

    "torch_geometric": r"""python - <<'PY'
import torch_geometric
print("torch_geometric:", torch_geometric.__version__)
PY""",

    "pykeops": r"""python - <<'PY'
import pykeops
print("pykeops:", pykeops.__version__)
PY""",
}

metalbind_diag = {}

for name, cmd in tests.items():
    print("\n" + "="*70)
    print("TEST:", name)
    print("="*70)

    rr = mex(
        "metalbind",
        cmd,
        check=False,
        capture=True
    )

    print(rr.stdout)

    if getattr(rr, "stderr", None):
        print(rr.stderr)

    print("RETURN CODE:", rr.returncode)

    metalbind_diag[name] = rr.returncode


# External executables
print("\n" + "="*70)
print("EXTERNAL BINARIES")
print("="*70)

for fp in [reduce_bin, msms_bin]:
    print(
        fp,
        "exists =", fp.exists(),
        "executable =", os.access(fp, os.X_OK)
    )

print("\nSUMMARY")
print(metalbind_diag)


TEST: python
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python --version
Python 3.7.12

RETURN CODE: 0

TEST: torch
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.tensor([1.0], device="cuda")
    print("CUDA tensor test:", x, x.device)
PY
torch: 1.10.1+cu113
torch.version.cuda: 11.3
cuda available: True
GPU: NVIDIA L4
CUDA tensor test: tensor([1.], device='cuda:0') cuda:0

RETURN CODE: 0

TEST: pymesh
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python - <<'PY'
import pymesh
print("pymesh import PASS")
print("pymesh path:", pymesh.__file__)
PY
pymesh import PASS
pymesh path: /content/micromamba_binding_ba

In [ ]:
# ============================================================
# METALBIND HOTFIX — CUDA 11.3 RUNTIME FOR torch_sparse
# ============================================================

# Install CUDA 11.3 runtime libraries required by the PyG cu113 wheels.
sh(
    f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
    'micromamba install -y -n metalbind '
    '-c conda-forge "cudatoolkit=11.3"'
)

# Make the conda CUDA shared libraries visible to all future subprocesses.
metalbind_envlib = (
    LOCAL_MAMBA_ROOT
    / 'envs'
    / 'metalbind'
    / 'lib'
)

os.environ['LD_LIBRARY_PATH'] = (
    str(metalbind_envlib)
    + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)

print("MetalBind LD_LIBRARY_PATH:")
print(os.environ['LD_LIBRARY_PATH'])

# Confirm that libcusparse is really present.
cusparse_files = sorted(
    metalbind_envlib.glob('libcusparse.so*')
)

print("\ncuSPARSE files:")
for p in cusparse_files:
    print(" ", p)

if not cusparse_files:
    raise RuntimeError(
        "cudatoolkit installed but libcusparse.so* not found"
    )


# ============================================================
# FULL MetalBind GPU STACK TEST
# ============================================================

rr = mex(
    'metalbind',
    r"""python - <<'PY'
import torch
import pymesh
import torch_scatter
import torch_sparse
import torch_cluster
import torch_geometric
import pykeops

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("pymesh: PASS")
print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", torch_sparse.__version__)
print("torch_cluster:", torch_cluster.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("pykeops:", pykeops.__version__)

assert torch.cuda.is_available()

# ------------------------------------------------------------
# Real CUDA torch_sparse operation
# ------------------------------------------------------------
from torch_sparse import SparseTensor

row = torch.tensor(
    [0, 1],
    dtype=torch.long,
    device='cuda'
)

col = torch.tensor(
    [1, 0],
    dtype=torch.long,
    device='cuda'
)

value = torch.ones(
    2,
    device='cuda'
)

A = SparseTensor(
    row=row,
    col=col,
    value=value,
    sparse_sizes=(2, 2)
)

x = torch.ones(
    (2, 1),
    device='cuda'
)

y = A.matmul(x)

print("torch_sparse CUDA operation:", y.flatten().tolist())
print("torch_sparse CUDA operation: PASS")

print("\nMETALBIND GPU STACK: PASS")
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

if getattr(rr, 'stderr', None):
    print(rr.stderr)

if rr.returncode != 0:
    raise RuntimeError(
        f"MetalBind GPU-stack test failed: return code {rr.returncode}"
    )

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba install -y -n metalbind -c conda-forge "cudatoolkit=11.3"
MetalBind LD_LIBRARY_PATH:
/content/micromamba_binding_baselines/envs/metalbind/lib:/usr/local/lib/python3.13/dist-packages/cv2/../../lib64:/usr/lib64-nvidia

cuSPARSE files:
  /content/micromamba_binding_baselines/envs/metalbind/lib/libcusparse.so
  /content/micromamba_binding_baselines/envs/metalbind/lib/libcusparse.so.11
  /content/micromamba_binding_baselines/envs/metalbind/lib/libcusparse.so.11.6.0.109
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python - <<'PY'
import torch
import pymesh
import torch_scatter
import torch_sparse
import torch_cluster
import torch_geometric
import pykeops

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("pymesh: PASS")
print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", tor

In [ ]:
# ============================================================
# 18A. MetalBind AUTHOR-COMPATIBLE ENVIRONMENT
# ============================================================
METALB=REPOS/'metalbind'
METALB_IONS=['CA','CD','CO','CU','CU1','FE','FE2','K','MG','MN','NA','NI','ZN']
METALB_EXT=WORK/'metalbind_external_v6'
METALB_EXT.mkdir(parents=True,exist_ok=True)

if RUN_METALBIND_TRANSFER or RUN_METALBIND_NATIVE:
    create_env('metalbind','3.7')

    # External executables from Bioconda; exact MSMS version used by the paper.
    sh(
        f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" micromamba install -y -n metalbind '
        '-c conda-forge -c bioconda '
        'scripps-msms=2.6.1 reduce "cudatoolkit=11.3"'
    )

    mex('metalbind',"python -m pip install -q --upgrade 'pip==23.3.2' 'setuptools<69' wheel")
    mex('metalbind',
        "python -m pip install -q 'numpy==1.21.6' 'scipy==1.7.3' 'scikit-learn==1.0.2' "
        "'biopython==1.78' 'pandas==1.3.5' 'tqdm<5'"
    )

    # Author GPU stack.
    mex('metalbind',
        "python -m pip install -q 'torch==1.10.1+cu113' "
        "-f https://download.pytorch.org/whl/torch_stable.html"
    )
    mex('metalbind',
        "python -m pip install -q 'torch-scatter==2.0.9' 'torch-sparse==0.6.13' "
        "'torch-cluster==1.5.9' 'torch-spline-conv==1.2.1' "
        "-f https://data.pyg.org/whl/torch-1.10.1+cu113.html"
    )
    mex('metalbind',"python -m pip install -q 'torch-geometric==2.0.4' 'pykeops==2.1'")

    # Official PyMesh v0.3 prebuilt CPython-3.7 Linux wheel.
    pymesh_wheel=CACHE/'pymesh2-0.3-cp37-cp37m-linux_x86_64.whl'
    if not pymesh_wheel.exists():
        sh(
            'curl -L --fail --retry 3 '
            '"https://sourceforge.net/projects/pymesh.mirror/files/v0.3/'
            'pymesh2-0.3-cp37-cp37m-linux_x86_64.whl/download" '
            f'-o "{pymesh_wheel}"'
        )
    mex('metalbind',f'python -m pip install -q "{pymesh_wheel}"')

    envbin=LOCAL_MAMBA_ROOT/'envs'/'metalbind'/'bin'
    reduce_bin=envbin/'reduce'
    msms_bin=envbin/'msms'
    if not reduce_bin.exists() or not msms_bin.exists():
        raise RuntimeError(f'MetalBind binaries missing: reduce={reduce_bin.exists()}, msms={msms_bin.exists()}')

    # Patch only machine-specific author hard-coded binary paths.
    (METALB/'default_config/bin_path.py').write_text(
        "bin_path = {}\n"
        f"bin_path['REDUCE'] = r'{reduce_bin}'\n"
        f"bin_path['MSMS'] = r'{msms_bin}'\n"
    )

    check=mex(
        'metalbind',
        "python -c \"import torch,pymesh,torch_geometric,pykeops; "
        "print('torch',torch.__version__,'cuda',torch.cuda.is_available()); "
        "print('pyg',torch_geometric.__version__); "
        "assert torch.cuda.is_available()\"",
        capture=True
    )
    print(check.stdout)

REUSE LOCAL ENV metalbind
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba install -y -n metalbind -c conda-forge -c bioconda scripps-msms=2.6.1 reduce "cudatoolkit=11.3"
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python -m pip install -q --upgrade 'pip==23.3.2' 'setuptools<69' wheel
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python -m pip install -q 'numpy==1.21.6' 'scipy==1.7.3' 'scikit-learn==1.0.2' 'biopython==1.78' 'pandas==1.3.5' 'tqdm<5'
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python -m pip install -q 'torch==1.10.1+cu113' -f https://download.pytorch.org/whl/torch_stable.html
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python -m pip install -q 'torch-scatter==2.0.9' 'torch-sparse==0.6.13' 'torch-cluster==1.5.9' 'torch-spline-conv==1.2.1' -f https://data.pyg.org/whl/torch-1.10.1+

In [ ]:
# ============================================================
# FINAL ESM-C 600M SMOKE TEST
# ============================================================

rr = mex(
    'esmc',
    r"""python - <<'PY'
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "biohub/ESMC-600M"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

tok = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModel.from_pretrained(
    MODEL_ID,
    dtype=torch.float16
).to("cuda").eval()

print("model:", MODEL_ID)
print("d_model:", model.config.d_model)

assert model.config.d_model == 1152

seq = "ACDEFGHIKLMNPQRSTVWY"

enc = tok(
    seq,
    return_tensors="pt",
    add_special_tokens=True,
    return_special_tokens_mask=True,
)

special = enc.pop("special_tokens_mask")[0].bool()

enc = {
    k: v.to("cuda")
    for k, v in enc.items()
}

with torch.inference_mode():
    out = model(
        **enc,
        return_dict=True
    )

h = out.last_hidden_state[0].float().cpu()

print("raw hidden:", tuple(h.shape))

h = h[~special.cpu()]

print("residue hidden:", tuple(h.shape))

assert h.shape == (len(seq), 1152)
assert torch.isfinite(h).all()

print("ESM-C 600M / 1152-d SMOKE TEST PASS")
PY""",
    check=False,
    capture=True
)

print(rr.stdout)

if getattr(rr, "stderr", None):
    print(rr.stderr)

if rr.returncode != 0:
    raise RuntimeError(
        f"ESM-C smoke test failed: {rr.returncode}"
    )

print("\nBIOHUB ESM-C ENVIRONMENT: PASS")

+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmc python - <<'PY'
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_ID = "biohub/ESMC-600M"

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

tok = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModel.from_pretrained(
    MODEL_ID,
    dtype=torch.float16
).to("cuda").eval()

print("model:", MODEL_ID)
print("d_model:", model.config.d_model)

assert model.config.d_model == 1152

seq = "ACDEFGHIKLMNPQRSTVWY"

enc = tok(
    seq,
    return_tensors="pt",
    add_special_tokens=True,
    return_special_tokens_mask=True,
)

special = enc.pop("special_tokens_mask")[0].bool()

enc = {
    k: v.to("cuda")
    for k, v in enc.items()
}

with torch.inference_mode():
    out = model(
        **enc,
        return_dict=True
    )

h = out.last_hidden_state[0].float().cpu()

print("raw hidden:", tuple(h.shape))

h = h[~special.cpu()]

print("residue hidden:", tuple

In [ ]:
# ============================================================
# 18B. MetalBind ESM-C 600M / 1152-d FEATURES — FINAL
# ============================================================

BIOHUB_TRANSFORMERS_COMMIT = (
    "3a8956fb4d4ea16b0ec8e71deef2c2909b6a5cbf"
)

METALB_ESMC_HF_MODEL = "biohub/ESMC-600M"

METALB_ESMC_ROOT = CACHE / "metalbind_esmc600m_v6"
METALB_ESMC_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 18B-1. ESM-C ISOLATED ENVIRONMENT
# ============================================================

if RUN_METALBIND_TRANSFER:

    create_env(
        "esmc",
        "3.10"
    )

    mex(
        "esmc",
        "python -m pip install -q --upgrade "
        "'pip<25' setuptools wheel"
    )

    # Base numerical / GPU environment
    mex(
        "esmc",
        "python -m pip install -q "
        "'torch==2.4.1' "
        "'numpy<2' "
        "biopython "
        "safetensors"
    )

    # ESM-C support is in Biohub's Transformers fork.
    rr = mex(
        "esmc",
        (
            "python -m pip install -q --upgrade "
            f"'git+https://github.com/Biohub/transformers.git@"
            f"{BIOHUB_TRANSFORMERS_COMMIT}' "
            "'huggingface-hub>=0.34,<1' "
            "'tokenizers>=0.22,<=0.23' "
            "'safetensors>=0.4.3'"
        ),
        check=False,
        capture=True
    )

    if rr.stdout:
        print(rr.stdout)

    if getattr(rr, "stderr", None):
        print(rr.stderr)

    if rr.returncode != 0:
        raise RuntimeError(
            "Biohub Transformers installation failed: "
            f"{rr.returncode}"
        )


    # --------------------------------------------------------
    # Optional HuggingFace token
    # --------------------------------------------------------

    if not (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_TOKEN")
    ):
        try:
            from google.colab import userdata

            _hf = userdata.get("HF_TOKEN")

            if _hf:
                os.environ["HF_TOKEN"] = _hf
                print(
                    "HF_TOKEN loaded from Colab Secrets."
                )

        except Exception:
            pass


    # --------------------------------------------------------
    # Strict environment smoke test
    # --------------------------------------------------------

    rr = mex(
        "esmc",
        r"""python - <<'PY'
import torch
import transformers

from transformers.models.esmc.tokenization_esmc import ESMCTokenizer
from transformers.models.esmc.modeling_esmc import ESMCModel

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available()

print("ESMCTokenizer import: PASS")
print("ESMCModel import: PASS")
print("ESM-C environment: PASS")
PY""",
        check=False,
        capture=True
    )

    print(rr.stdout)

    if rr.returncode != 0:
        raise RuntimeError(
            "ESM-C environment smoke test failed: "
            f"{rr.returncode}"
        )


# ============================================================
# 18B-2. WRITE PER-DATASET ESM-C GENERATION SCRIPT
# ============================================================

def write_esmc_generation_script(
    dataset,
    df,
    outdir
):

    fasta = (
        CANON
        /
        f"{dataset}.fasta"
    )

    script = (
        WORK
        /
        f"generate_metalbind_esmc_{dataset}.py"
    )

    script.write_text(
f"""
import os
from pathlib import Path

import numpy as np
import torch

from Bio import SeqIO
from transformers import AutoTokenizer, AutoModel


MODEL_ID = {METALB_ESMC_HF_MODEL!r}

FASTA = Path(r"{fasta}")
OUT = Path(r"{outdir}")

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# GPU guard
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "MetalBind ESM-C extraction requires CUDA"
    )


# ------------------------------------------------------------
# Optional HuggingFace authentication
# ------------------------------------------------------------

token = (
    os.environ.get("HF_TOKEN")
    or
    os.environ.get("HUGGINGFACE_TOKEN")
)

kwargs = (
    {{"token": token}}
    if token
    else {{}}
)


# ------------------------------------------------------------
# Official HF-compatible Biohub ESM-C 600M
# ------------------------------------------------------------

print(
    "Loading ESM-C:",
    MODEL_ID
)

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    **kwargs
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    **kwargs
).to(
    "cuda"
).eval()


print(
    "ESM-C d_model:",
    model.config.d_model
)

if model.config.d_model != 1152:
    raise RuntimeError(
        "Unexpected ESM-C hidden dimension: "
        f"{{model.config.d_model}}"
    )


# ------------------------------------------------------------
# Generate one residue-level embedding per protein
# ------------------------------------------------------------

records = list(
    SeqIO.parse(
        str(FASTA),
        "fasta"
    )
)

print(
    "Proteins:",
    len(records)
)


for idx, rec in enumerate(
    records,
    start=1
):

    pid = str(rec.id)
    seq = str(rec.seq)

    fp = (
        OUT
        /
        f"{{pid}}.npy"
    )


    # --------------------------------------------------------
    # Cache reuse
    # --------------------------------------------------------

    if fp.exists():

        try:

            a = np.load(
                fp,
                mmap_mode="r"
            )

            if (
                a.shape
                ==
                (
                    len(seq),
                    1152
                )
                and
                np.all(
                    np.isfinite(a)
                )
            ):
                print(
                    f"[{{idx}}/{{len(records)}}] "
                    f"{{pid}} cached"
                )

                continue

        except Exception:
            pass


    # --------------------------------------------------------
    # Tokenize
    # --------------------------------------------------------

    enc = tok(
        seq,
        return_tensors="pt",
        add_special_tokens=True,
        return_special_tokens_mask=True,
        truncation=False,
    )

    special = enc.pop(
        "special_tokens_mask"
    )[0].bool()


    enc = {{
        k: v.to("cuda")
        for k, v
        in enc.items()
    }}


    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    with torch.inference_mode():

        out = model(
            **enc,
            return_dict=True
        )


    if (
        not hasattr(
            out,
            "last_hidden_state"
        )
        or
        out.last_hidden_state
        is None
    ):
        raise RuntimeError(
            f"{{pid}}: "
            "ESMCModel output missing "
            "last_hidden_state"
        )


    h = (
        out
        .last_hidden_state[0]
        .float()
        .cpu()
    )


    # --------------------------------------------------------
    # Remove CLS/EOS special tokens
    # --------------------------------------------------------

    special_cpu = (
        special
        .cpu()
    )


    if (
        special_cpu.shape[0]
        !=
        h.shape[0]
    ):
        raise RuntimeError(
            f"{{pid}}: "
            f"special-token mask length "
            f"{{special_cpu.shape[0]}} "
            f"!= hidden length "
            f"{{h.shape[0]}}"
        )


    h = h[
        ~special_cpu
    ]


    arr = (
        h.numpy()
        .astype(
            np.float32,
            copy=False
        )
    )


    # --------------------------------------------------------
    # STRICT residue alignment
    # --------------------------------------------------------

    expected = (
        len(seq),
        1152
    )


    if arr.shape != expected:
        raise RuntimeError(
            f"{{pid}}: "
            f"ESM-C shape "
            f"{{arr.shape}}, "
            f"expected "
            f"{{expected}}"
        )


    if not np.all(
        np.isfinite(arr)
    ):
        raise RuntimeError(
            f"{{pid}}: "
            "non-finite ESM-C embedding"
        )


    np.save(
        fp,
        arr
    )


    print(
        f"[{{idx}}/{{len(records)}}] "
        f"{{pid}} PASS "
        f"{{arr.shape}}"
    )


print(
    "ESM-C generation complete:",
    len(
        list(
            OUT.glob("*.npy")
        )
    )
)
"""
    )

    return script


# ============================================================
# 18B-3. GENERATE + STRICT AUDIT
# ============================================================

if RUN_METALBIND_TRANSFER:

    for dataset, df in DATASETS.items():

        print(
            "\n"
            +
            "=" * 72
        )

        print(
            "MetalBind ESM-C:",
            dataset
        )

        print(
            "=" * 72
        )


        outdir = (
            METALB_ESMC_ROOT
            /
            dataset
        )

        outdir.mkdir(
            parents=True,
            exist_ok=True
        )


        # ----------------------------------------------------
        # Check whether generation is needed
        # ----------------------------------------------------

        need = (
            REBUILD_METALBIND_ESMC
        )


        if not need:

            for r in df.itertuples():

                fp = (
                    outdir
                    /
                    f"{r.prot_id}.npy"
                )


                if not fp.exists():

                    need = True
                    break


                try:

                    a = np.load(
                        fp,
                        mmap_mode="r"
                    )

                except Exception:

                    need = True
                    break


                if (
                    a.shape
                    !=
                    (
                        len(r.sequence),
                        1152
                    )
                    or
                    not np.all(
                        np.isfinite(a)
                    )
                ):

                    need = True
                    break


        # ----------------------------------------------------
        # Generate missing/invalid embeddings
        # ----------------------------------------------------

        if need:

            script = (
                write_esmc_generation_script(
                    dataset,
                    df,
                    outdir
                )
            )


            print(
                "Generation script:",
                script
            )


            rr = mex(
                "esmc",
                f'python "{script}"',
                check=False,
                capture=True
            )


            if rr.stdout:
                print(
                    rr.stdout
                )


            if getattr(
                rr,
                "stderr",
                None
            ):
                print(
                    rr.stderr
                )


            if rr.returncode != 0:

                raise RuntimeError(
                    f"{dataset}: "
                    "ESM-C generation failed "
                    f"with return code "
                    f"{rr.returncode}. "
                    "Full subprocess log "
                    "printed above."
                )


        # ----------------------------------------------------
        # FINAL FULL-DATASET AUDIT
        # ----------------------------------------------------

        bad = []


        for r in df.itertuples():

            pid = str(
                r.prot_id
            )

            fp = (
                outdir
                /
                f"{pid}.npy"
            )


            if not fp.exists():

                bad.append(
                    (
                        pid,
                        "missing"
                    )
                )

                continue


            try:

                a = np.load(
                    fp,
                    mmap_mode="r"
                )

            except Exception as e:

                bad.append(
                    (
                        pid,
                        f"unreadable: {e}"
                    )
                )

                continue


            expected = (
                len(r.sequence),
                1152
            )


            if a.shape != expected:

                bad.append(
                    (
                        pid,
                        f"shape={a.shape}, "
                        f"expected={expected}"
                    )
                )


            elif not np.all(
                np.isfinite(a)
            ):

                bad.append(
                    (
                        pid,
                        "non-finite"
                    )
                )


        if bad:

            raise RuntimeError(
                f"{dataset}: "
                "ESM-C final audit failed: "
                f"{bad[:10]}"
            )


        print(
            f"{dataset}: "
            f"ESM-C audit PASS "
            f"{len(df)}/{len(df)} proteins."
        )


    print(
        "\n"
        "MetalBind ESM-C 600M audit "
        "PASSED for TestSet300 + TestSetNew46."
    )

REUSE LOCAL ENV esmc
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmc python -m pip install -q --upgrade 'pip<25' setuptools wheel
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmc python -m pip install -q 'torch==2.4.1' 'numpy<2' biopython safetensors
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmc python -m pip install -q --upgrade 'git+https://github.com/Biohub/transformers.git@3a8956fb4d4ea16b0ec8e71deef2c2909b6a5cbf' 'huggingface-hub>=0.34,<1' 'tokenizers>=0.22,<=0.23' 'safetensors>=0.4.3'
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n esmc python - <<'PY'
import torch
import transformers

from transformers.models.esmc.tokenization_esmc import ESMCTokenizer
from transformers.models.esmc.modeling_esmc import ESMCModel

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

assert 

In [ ]:
# ============================================================
# 18C. CONVERT ESM-C TO AUTHOR TEXT FORMAT + PREPARE MANIFESTS
# ============================================================
METALB_PREP_ROOT=METALB_EXT/'prepared'
METALB_PREP_ROOT.mkdir(parents=True,exist_ok=True)

if RUN_METALBIND_TRANSFER:
    for dataset,df in DATASETS.items():
        root=METALB_PREP_ROOT/dataset
        raw_pdb=root/'raw_pdb'
        esm_txt=root/'600M'
        raw_pdb.mkdir(parents=True,exist_ok=True)
        esm_txt.mkdir(parents=True,exist_ok=True)

        manifest_rows=[]
        for r in df.itertuples():
            pid=str(r.prot_id)
            if ':' in pid or '/' in pid:
                raise RuntimeError(f'MetalBind external adapter requires path-safe protein id, got {pid}')

            # Author extract_multimer/get_seq expects <pdb_id><chain>.pdb, hence pidA.pdb.
            pdb_dst=raw_pdb/f'{pid}A.pdb'
            shutil.copy2(NORMALIZED_EXTERNAL_PDB[dataset][pid],pdb_dst)

            txt=esm_txt/f'{pid}A.txt'
            npy=METALB_ESMC_ROOT/dataset/f'{pid}.npy'
            if REBUILD_METALBIND_ESMC or not txt.exists():
                arr=np.load(npy)
                np.savetxt(txt,arr,fmt='%.10e')

            # Round-trip author loader format check.
            aa=np.loadtxt(txt,dtype=np.float32)
            if aa.shape!=(len(r.sequence),1152):
                raise RuntimeError(f'{dataset}/{pid}: author ESM text shape {aa.shape}')

            manifest_rows.append([pid,r.sequence,str(pdb_dst),str(txt)])

        man=pd.DataFrame(manifest_rows,columns=['prot_id','sequence','pdb_path','esm_txt'])
        man.to_csv(root/'manifest.csv',index=False)
        print(dataset,'MetalBind manifest',len(man))

TestSet300 MetalBind manifest 300
TestSetNew46 MetalBind manifest 46


In [ ]:
# ============================================================
# 18D. WRITE + RUN LABEL-FREE ADAPTER AROUND AUTHORS' MetalBindProcessor
# FINAL DEBUG-SAFE VERSION
# ============================================================

METALB_PREP_SCRIPT = METALB / 'external_prepare_v6.py'

METALB_PREP_SCRIPT.write_text(r'''
import argparse
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

from MetalBindProcessor import MetalBindProcessor


# ============================================================
# Label-free inference adapters
# ============================================================

def dummy_nucleic(self):
    return []


def dummy_interface(self, anno):
    # Inference only: labels/interfaces are placeholders and
    # are not used to generate predictions.
    return np.zeros(
        self.mesh.vertices.shape[0],
        dtype=np.float32
    )


def dummy_site(self, anno):
    # Preserve exact surface-residue ordering from author's processor.
    s2i = {
        res: i
        for i, res in enumerate(self.surface_res)
    }

    point_res_id = np.zeros(
        len(self.vert_info),
        dtype=np.int64
    )

    for k, info in enumerate(self.vert_info):
        fields = info.split('_')

        if len(fields) < 2:
            raise RuntimeError(
                f'Unexpected vert_info entry: {info}'
            )

        rid = fields[0] + '_' + fields[1]

        if rid not in s2i:
            raise RuntimeError(
                f'point residue {rid} missing from surface-residue map'
            )

        point_res_id[k] = s2i[rid]

    self.site_by_biolip = set()

    return (
        np.zeros(
            len(self.surface_res),
            dtype=np.int64
        ),
        point_res_id
    )


# Patch labels only; structural preprocessing remains author's code.
MetalBindProcessor._get_nucleic_acid_atom = dummy_nucleic
MetalBindProcessor._get_interface_mental = dummy_interface
MetalBindProcessor._get_site_biolip = dummy_site


# ============================================================
# CLI
# ============================================================

ap = argparse.ArgumentParser()
ap.add_argument('--manifest', required=True)
ap.add_argument('--root', required=True)
ap.add_argument('--rebuild', action='store_true')
a = ap.parse_args()

root = Path(a.root)
manifest_path = Path(a.manifest)

if not manifest_path.exists():
    raise FileNotFoundError(
        f'Manifest not found: {manifest_path}'
    )

manifest = pd.read_csv(manifest_path)

if len(manifest) == 0:
    raise RuntimeError(
        f'Empty manifest: {manifest_path}'
    )

required_cols = {
    'prot_id',
    'sequence',
    'pdb_path',
    'esm_txt',
}

missing_cols = required_cols - set(manifest.columns)

if missing_cols:
    raise RuntimeError(
        f'Manifest missing columns: {sorted(missing_cols)}'
    )


# ============================================================
# Output directories
# ============================================================

for d in [
    'chain_pdb',
    'protonated_pdb',
    'xyzrn',
    'msms',
    'data_label',
    'ligand_dummy',
    'hmm',
    'fasta',
    'dssp',
    'save',
    'mesh',
    'mesh_fix',
    'ply',
]:
    (root / d).mkdir(
        parents=True,
        exist_ok=True
    )


raw_pdb = Path(
    manifest.iloc[0].pdb_path
).parent

esm_dir = Path(
    manifest.iloc[0].esm_txt
).parent


if not raw_pdb.exists():
    raise FileNotFoundError(
        f'raw_pdb directory not found: {raw_pdb}'
    )

if not esm_dir.exists():
    raise FileNotFoundError(
        f'ESM directory not found: {esm_dir}'
    )


opts = {
    'base_dir': str(root),
    'raw_pdb_dir': str(raw_pdb),
    'ligand_dir': str(root / 'ligand_dummy'),
    'hhm_dir': str(root / 'hmm'),
    'fasta_dir': str(root / 'fasta'),
    'dssp': str(root / 'dssp'),
    'protonated_pdb_dir': str(root / 'protonated_pdb'),
    'chain_pdb_dir': str(root / 'chain_pdb'),
    'xyzrn_dir': str(root / 'xyzrn'),
    'msms_dir': str(root / 'msms'),
    'data_label': str(root / 'data_label'),
    'save_dir': str(root / 'save'),
    'mesh_dir': str(root / 'mesh'),
    'mesh_fix_dir': str(root / 'mesh_fix'),
    'ply_dir': str(root / 'ply'),
    'esm_dir': str(esm_dir),
}


print(
    f'Preparing {len(manifest)} proteins'
)

print(
    'raw_pdb_dir =',
    opts['raw_pdb_dir']
)

print(
    'esm_dir =',
    opts['esm_dir']
)


# ============================================================
# Per-protein preprocessing
# ============================================================

prepared = 0
cached = 0


for i, row in enumerate(
    manifest.itertuples(),
    start=1
):

    pid = str(row.prot_id)
    pair = f'{pid}:A:X1'

    fp = (
        root
        /
        'data_label'
        /
        pair
    )

    mp = (
        root
        /
        'data_label'
        /
        f'{pair}.surface_map.json'
    )


    # --------------------------------------------------------
    # Cache reuse
    # --------------------------------------------------------

    if (
        fp.exists()
        and
        mp.exists()
        and
        not a.rebuild
    ):
        cached += 1

        print(
            f'[{i}/{len(manifest)}] {pid}: cached'
        )

        continue


    print(
        f'[{i}/{len(manifest)}] {pid}: START'
    )


    # --------------------------------------------------------
    # Strict input checks
    # --------------------------------------------------------

    pdb_path = Path(row.pdb_path)
    esm_path = Path(row.esm_txt)

    if not pdb_path.exists():
        raise FileNotFoundError(
            f'{pid}: PDB missing: {pdb_path}'
        )

    if not esm_path.exists():
        raise FileNotFoundError(
            f'{pid}: ESM text missing: {esm_path}'
        )


    try:

        # ----------------------------------------------------
        # AUTHORS' processor
        # ----------------------------------------------------

        proc = MetalBindProcessor(
            pair=pair,
            anno='',
            ligand_type='ZN',
            # Metal branch -> 0.5-A probe radius in author's code
            dir_opts=opts,
            RQ_window=6.0,
            # exact public prepare_one.sh setting
        )

        proc.get_data()


        # ----------------------------------------------------
        # Verify author's data object
        # ----------------------------------------------------

        if not fp.exists():
            raise RuntimeError(
                f'{pid}: proc.get_data() finished but expected '
                f'data file was not created: {fp}'
            )


        # ----------------------------------------------------
        # Surface residue -> canonical sequence position
        # ----------------------------------------------------

        mapping = []

        for s in proc.surface_res:

            fields = s.split('_')

            if len(fields) < 2:
                raise RuntimeError(
                    f'{pid}: malformed surface residue: {s}'
                )

            if fields[0] != 'A':
                raise RuntimeError(
                    f'{pid}: unexpected surface chain {fields[0]}'
                )

            resid = int(fields[1])

            if (
                resid < 1
                or
                resid > len(row.sequence)
            ):
                raise RuntimeError(
                    f'{pid}: surface residue number {resid} '
                    f'outside sequence length {len(row.sequence)}'
                )

            mapping.append(
                {
                    'surface_residue': s,
                    'canonical_index': resid - 1,
                }
            )


        if len(mapping) == 0:
            raise RuntimeError(
                f'{pid}: no surface residues produced'
            )


        # ----------------------------------------------------
        # Save exact remapping metadata
        # ----------------------------------------------------

        mp.write_text(
            json.dumps(
                {
                    'prot_id': pid,
                    'sequence_length': len(row.sequence),
                    'surface_res': list(
                        proc.surface_res
                    ),
                    'mapping': mapping,
                },
                indent=2
            )
        )


        if not mp.exists():
            raise RuntimeError(
                f'{pid}: surface map was not written'
            )


        prepared += 1

        print(
            f'[{i}/{len(manifest)}] {pid}: PASS '
            f'surface_res={len(mapping)}'
        )


    except Exception:

        print(
            '\n'
            + '=' * 80
        )

        print(
            f'FAILED PROTEIN: {pid}'
        )

        print(
            f'PAIR: {pair}'
        )

        print(
            f'PDB: {pdb_path}'
        )

        print(
            f'ESM: {esm_path}'
        )

        print(
            '=' * 80
        )

        traceback.print_exc()

        print(
            '=' * 80
            + '\n'
        )

        raise


print(
    f'PREPROCESS COMPLETE: '
    f'prepared_now={prepared}, '
    f'cached={cached}, '
    f'total={len(manifest)}'
)
''')


# ============================================================
# RUN PREPROCESSING + STRICT COMPLETENESS AUDIT
# ============================================================

if RUN_METALBIND_TRANSFER:

    for dataset, df in DATASETS.items():

        print(
            '\n'
            + '=' * 80
        )

        print(
            'MetalBind surface preprocessing:',
            dataset
        )

        print(
            '=' * 80
        )


        root = (
            METALB_PREP_ROOT
            /
            dataset
        )

        manifest_path = (
            root
            /
            'manifest.csv'
        )


        if not manifest_path.exists():

            raise FileNotFoundError(
                f'{dataset}: manifest missing: '
                f'{manifest_path}. Run 18C first.'
            )


        cmd = (
            f'python "{METALB_PREP_SCRIPT}" '
            f'--manifest "{manifest_path}" '
            f'--root "{root}"'
            +
            (
                ' --rebuild'
                if REBUILD_METALBIND_SURFACE
                else ''
            )
        )


        # IMPORTANT:
        # capture real subprocess traceback instead of hiding it
        # behind CalledProcessError.
        rr = mex(
            'metalbind',
            cmd,
            cwd=METALB,
            check=False,
            capture=True
        )


        print(
            '\n'
            + '-' * 80
        )

        print(
            f'OUTPUT: {dataset}'
        )

        print(
            '-' * 80
        )


        if rr.stdout:
            print(
                rr.stdout
            )


        print(
            f'\nRETURN CODE: {rr.returncode}'
        )


        if rr.returncode != 0:

            raise RuntimeError(
                f'{dataset}: MetalBind surface preprocessing '
                f'failed with return code {rr.returncode}. '
                f'The real subprocess traceback is printed above.'
            )


        # ====================================================
        # STRICT POST-PREPROCESSING COMPLETENESS AUDIT
        # ====================================================

        missing = []
        bad_maps = []


        for r in df.itertuples():

            pid = str(
                r.prot_id
            )

            pair = (
                f'{pid}:A:X1'
            )

            data_fp = (
                root
                /
                'data_label'
                /
                pair
            )

            map_fp = (
                root
                /
                'data_label'
                /
                f'{pair}.surface_map.json'
            )


            if not data_fp.exists():

                missing.append(
                    (
                        pid,
                        'pickle'
                    )
                )


            if not map_fp.exists():

                missing.append(
                    (
                        pid,
                        'map'
                    )
                )

                continue


            try:

                z = json.loads(
                    map_fp.read_text()
                )

            except Exception as e:

                bad_maps.append(
                    (
                        pid,
                        f'unreadable map: {e}'
                    )
                )

                continue


            if (
                int(
                    z.get(
                        'sequence_length',
                        -1
                    )
                )
                !=
                len(r.sequence)
            ):

                bad_maps.append(
                    (
                        pid,
                        'sequence_length mismatch'
                    )
                )


            mapping = z.get(
                'mapping',
                []
            )


            if len(mapping) == 0:

                bad_maps.append(
                    (
                        pid,
                        'empty mapping'
                    )
                )

                continue


            canonical_indices = [
                int(
                    x[
                        'canonical_index'
                    ]
                )
                for x
                in mapping
            ]


            if any(
                x < 0
                or
                x >= len(r.sequence)
                for x
                in canonical_indices
            ):

                bad_maps.append(
                    (
                        pid,
                        'canonical index out of range'
                    )
                )


        if missing:

            raise RuntimeError(
                f'{dataset}: MetalBind preprocessing incomplete: '
                f'{missing[:10]}'
            )


        if bad_maps:

            raise RuntimeError(
                f'{dataset}: MetalBind surface-map audit failed: '
                f'{bad_maps[:10]}'
            )


        print(
            f'{dataset}: MetalBind preprocessing PASS '
            f'{len(df)}/{len(df)} proteins.'
        )


    print(
        '\n'
        'MetalBind surface preprocessing '
        'PASSED for TestSet300 + TestSetNew46.'
    )

Replacing MSMS package with legacy Bioconda msms=2.6.1
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba remove -y -n metalbind scripps-msms || true
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba install -y -n metalbind -c bioconda -c conda-forge "msms=2.6.1"
reduce: /content/micromamba_binding_baselines/envs/metalbind/bin/reduce
msms: /content/micromamba_binding_baselines/envs/metalbind/bin/msms
reduce exists: True
msms exists: True
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind "/content/micromamba_binding_baselines/envs/metalbind/bin/msms" -h

MSMS binary output:
Usage : msms parameters 
  -probe_radius float : probe sphere radius, [1.5]
  -density float      : surface points density, [1.0]
  -hdensity float     : surface points high density, [3.0]
  -surface <tses,ases>: triangulated or Analytical SES, [tses]
  -no_area            : turns off the analytical surface area computation
  -socketName s

RuntimeError: One-protein MSMS smoke test failed. Full error is printed above.

In [ ]:
# ============================================================
# 18D-0. FINAL MSMS 2.6.1 FIX
# - use legacy Bioconda MSMS 2.6.1
# - run MSMS temporary outputs on local /content
# - strict return-code + ASCII validation
# - one-protein real smoke test
# ============================================================

# ------------------------------------------------------------
# 1. Replace current scripps-msms package with legacy MSMS 2.6.1
# ------------------------------------------------------------

print("=" * 80)
print("Replacing MSMS package with legacy Bioconda msms=2.6.1")
print("=" * 80)

sh(
    f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
    'micromamba remove -y -n metalbind scripps-msms || true'
)

sh(
    f'MAMBA_ROOT_PREFIX="{LOCAL_MAMBA_ROOT}" '
    'micromamba install -y -n metalbind '
    '-c bioconda -c conda-forge '
    '"msms=2.6.1"'
)


# ------------------------------------------------------------
# 2. Re-detect binaries and patch MetalBind machine paths
# ------------------------------------------------------------

envbin = (
    LOCAL_MAMBA_ROOT
    /
    'envs'
    /
    'metalbind'
    /
    'bin'
)

reduce_bin = (
    envbin
    /
    'reduce'
)

msms_bin = (
    envbin
    /
    'msms'
)

print("reduce:", reduce_bin)
print("msms:", msms_bin)

print(
    "reduce exists:",
    reduce_bin.exists()
)

print(
    "msms exists:",
    msms_bin.exists()
)


if not reduce_bin.exists():
    raise RuntimeError(
        f"Reduce binary missing: {reduce_bin}"
    )

if not msms_bin.exists():
    raise RuntimeError(
        f"MSMS binary missing: {msms_bin}"
    )


(
    METALB
    /
    'default_config'
    /
    'bin_path.py'
).write_text(
    "bin_path = {}\n"
    f"bin_path['REDUCE'] = r'{reduce_bin}'\n"
    f"bin_path['MSMS'] = r'{msms_bin}'\n"
)


# ------------------------------------------------------------
# 3. Check installed MSMS binary
# ------------------------------------------------------------

rr = mex(
    'metalbind',
    f'"{msms_bin}" -h',
    check=False,
    capture=True
)

print("\nMSMS binary output:")
print(rr.stdout)

print(
    "MSMS -h return code:",
    rr.returncode
)


# ============================================================
# 4. Patch apply_msms.py
#
# IMPORTANT:
# We preserve author's exact:
#   density = 3
#   hdensity = 3
#   probe radius
#   -if / -of / -af
#
# Only temporary output directory and error validation change.
# ============================================================

apply_msms_path = (
    METALB
    /
    'compute_surface'
    /
    'apply_msms.py'
)


apply_msms_path.write_text(r'''
from default_config.bin_path import bin_path
from compute_surface.read_msms import read_msms

from subprocess import Popen, PIPE

import os
import shutil
import tempfile


def computeMSMS(pair, dir_opts, probe_radius):

    # --------------------------------------------------------
    # Input XYZRN generated by author's preprocessing
    # --------------------------------------------------------

    xyzrn = os.path.join(
        dir_opts['xyzrn_dir'],
        pair + '.xyzrn'
    )

    if not os.path.exists(xyzrn):
        raise FileNotFoundError(
            f'MSMS input XYZRN missing: {xyzrn}'
        )

    if os.path.getsize(xyzrn) == 0:
        raise RuntimeError(
            f'MSMS input XYZRN is empty: {xyzrn}'
        )


    # --------------------------------------------------------
    # Run native MSMS on LOCAL scratch.
    #
    # Google Drive/FUSE is not used for native temporary
    # .vert/.face generation.
    # --------------------------------------------------------

    scratch_parent = os.environ.get(
        'METALBIND_MSMS_TMP',
        '/content'
    )

    if not os.path.isdir(scratch_parent):
        scratch_parent = '/tmp'


    tmpdir = tempfile.mkdtemp(
        prefix='metalbind_msms_',
        dir=scratch_parent
    )

    msms_file_base = os.path.join(
        tmpdir,
        'surface'
    )


    try:

        msms_bin = bin_path['MSMS']


        # ----------------------------------------------------
        # EXACT author MSMS settings
        # ----------------------------------------------------

        args = [
            msms_bin,
            '-density',
            '3',
            '-hdensity',
            '3',
            '-probe',
            str(probe_radius),
            '-if',
            xyzrn,
            '-of',
            msms_file_base,
            '-af',
            msms_file_base,
        ]


        print(
            'MSMS COMMAND:',
            ' '.join(args)
        )


        p = Popen(
            args,
            stdout=PIPE,
            stderr=PIPE
        )

        stdout, stderr = (
            p.communicate()
        )


        stdout_text = stdout.decode(
            'utf-8',
            errors='replace'
        )

        stderr_text = stderr.decode(
            'utf-8',
            errors='replace'
        )


        # ----------------------------------------------------
        # STRICT return-code check
        # ----------------------------------------------------

        if p.returncode != 0:

            raise RuntimeError(
                'MSMS failed.\n'
                f'Return code: {p.returncode}\n'
                f'STDOUT:\n{stdout_text}\n'
                f'STDERR:\n{stderr_text}'
            )


        vert_path = (
            msms_file_base
            +
            '.vert'
        )

        face_path = (
            msms_file_base
            +
            '.face'
        )


        # ----------------------------------------------------
        # STRICT output existence
        # ----------------------------------------------------

        for fp in [
            vert_path,
            face_path
        ]:

            if not os.path.exists(fp):

                raise RuntimeError(
                    'MSMS returned success but '
                    f'output is missing: {fp}\n'
                    f'STDOUT:\n{stdout_text}\n'
                    f'STDERR:\n{stderr_text}'
                )


            if os.path.getsize(fp) == 0:

                raise RuntimeError(
                    f'MSMS output is empty: {fp}'
                )


        # ----------------------------------------------------
        # MSMS .vert/.face MUST be ASCII text.
        #
        # Do not silently decode binary/corrupt files.
        # ----------------------------------------------------

        for fp in [
            vert_path,
            face_path
        ]:

            with open(
                fp,
                'rb'
            ) as f:

                raw = f.read()


            try:

                raw.decode(
                    'ascii'
                )

            except UnicodeDecodeError as e:

                head = raw[:256]

                raise RuntimeError(
                    'MSMS generated non-ASCII output.\n'
                    f'File: {fp}\n'
                    f'Size: {len(raw)} bytes\n'
                    f'First 256 bytes: {head!r}\n'
                    f'Decode error: {e}\n'
                    f'MSMS STDOUT:\n{stdout_text}\n'
                    f'MSMS STDERR:\n{stderr_text}'
                )


        # ----------------------------------------------------
        # Parse using AUTHOR'S read_msms()
        # ----------------------------------------------------

        vertices, faces, normalv, res_id = (
            read_msms(
                msms_file_base
            )
        )


        # ----------------------------------------------------
        # Basic geometry sanity checks
        # ----------------------------------------------------

        if len(vertices) == 0:

            raise RuntimeError(
                'MSMS produced zero vertices'
            )


        if len(faces) == 0:

            raise RuntimeError(
                'MSMS produced zero faces'
            )


        if len(normalv) != len(vertices):

            raise RuntimeError(
                'MSMS normal/vertex count mismatch'
            )


        if len(res_id) != len(vertices):

            raise RuntimeError(
                'MSMS residue-id/vertex count mismatch'
            )


        print(
            'MSMS PASS:',
            'vertices=',
            len(vertices),
            'faces=',
            len(faces)
        )


        return (
            vertices,
            faces,
            normalv,
            res_id
        )


    finally:

        shutil.rmtree(
            tmpdir,
            ignore_errors=True
        )
''')


print(
    "\nPatched:",
    apply_msms_path
)


# ============================================================
# 5. REAL ONE-PROTEIN MSMS SMOKE TEST
# ============================================================

dataset = 'TestSet300'
pid = 'A0L5S6'
pair = f'{pid}:A:X1'

root = (
    METALB_PREP_ROOT
    /
    dataset
)

xyzrn_fp = (
    root
    /
    'xyzrn'
    /
    f'{pair}.xyzrn'
)


print("\n" + "=" * 80)
print("ONE-PROTEIN MSMS SMOKE TEST")
print("=" * 80)

print(
    "XYZRN:",
    xyzrn_fp
)

print(
    "exists:",
    xyzrn_fp.exists()
)


if not xyzrn_fp.exists():

    raise RuntimeError(
        'The previous 18D run should have produced '
        f'this XYZRN file, but it is missing: {xyzrn_fp}'
    )


rr = mex(
    'metalbind',
    f"""python - <<'PY'
from compute_surface.apply_msms import computeMSMS

pair = {pair!r}

opts = {{
    'xyzrn_dir': r"{root/'xyzrn'}",
    'msms_dir': r"{root/'msms'}",
}}

v, f, n, rid = computeMSMS(
    pair,
    dir_opts=opts,
    probe_radius=0.5
)

print("vertices:", v.shape)
print("faces:", f.shape)
print("normals:", n.shape)
print("residue IDs:", len(rid))

assert v.ndim == 2
assert v.shape[1] == 3
assert f.ndim == 2
assert f.shape[1] == 3
assert n.shape == v.shape
assert len(rid) == len(v)

print("ONE-PROTEIN MSMS SMOKE TEST: PASS")
PY""",
    cwd=METALB,
    check=False,
    capture=True
)


print(rr.stdout)


if rr.returncode != 0:

    raise RuntimeError(
        'One-protein MSMS smoke test failed. '
        'Full error is printed above.'
    )


print(
    "\nFINAL MSMS 2.6.1 FIX: PASS"
)

Replacing MSMS package with legacy Bioconda msms=2.6.1
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba remove -y -n metalbind scripps-msms || true
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba install -y -n metalbind -c bioconda -c conda-forge "msms=2.6.1"
reduce: /content/micromamba_binding_baselines/envs/metalbind/bin/reduce
msms: /content/micromamba_binding_baselines/envs/metalbind/bin/msms
reduce exists: True
msms exists: True
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind "/content/micromamba_binding_baselines/envs/metalbind/bin/msms" -h

MSMS binary output:
Usage : msms parameters 
  -probe_radius float : probe sphere radius, [1.5]
  -density float      : surface points density, [1.0]
  -hdensity float     : surface points high density, [3.0]
  -surface <tses,ases>: triangulated or Analytical SES, [tses]
  -no_area            : turns off the analytical surface area computation
  -socketName s

RuntimeError: One-protein MSMS smoke test failed. Full error is printed above.

In [ ]:
# ============================================================
# 18D. MetalBind surface preprocessing — FINAL
# Short-path MSMS patch + label-free author processor
# ============================================================

# ============================================================
# 18D-0. PATCH AUTHORS' computeMSMS() BEFORE MetalBindProcessor RUNS
#
# Why: legacy MSMS 2.6.1 can corrupt its text output when the
#      long Google Drive/FUSE xyzrn path is passed directly.
#      We therefore copy only the XYZRN input to a very short
#      local scratch path, run MSMS entirely there, parse with
#      the AUTHORS' read_msms(), then delete the scratch files.
#
# Scientific settings are unchanged:
#   density  = 3
#   hdensity = 3
#   probe    = probe_radius supplied by MetalBindProcessor
# ============================================================

from pathlib import Path

APPLY_MSMS = METALB / 'compute_surface' / 'apply_msms.py'

if not APPLY_MSMS.exists():
    raise FileNotFoundError(
        f'MetalBind apply_msms.py not found: {APPLY_MSMS}. Run 18A first.'
    )

APPLY_MSMS.write_text(r'''
from default_config.bin_path import bin_path
from compute_surface.read_msms import read_msms

from subprocess import Popen, PIPE
from pathlib import Path

import shutil
import tempfile


def computeMSMS(pair, dir_opts, probe_radius):

    # --------------------------------------------------------
    # Author-generated XYZRN input
    # --------------------------------------------------------
    source_xyzrn = (
        Path(dir_opts['xyzrn_dir'])
        / f'{pair}.xyzrn'
    )

    if not source_xyzrn.exists():
        raise FileNotFoundError(
            f'MSMS input XYZRN missing: {source_xyzrn}'
        )

    if source_xyzrn.stat().st_size == 0:
        raise RuntimeError(
            f'MSMS input XYZRN is empty: {source_xyzrn}'
        )

    # --------------------------------------------------------
    # SHORT LOCAL PATHS ONLY for the native legacy MSMS binary
    # --------------------------------------------------------
    content_dir = Path('/content')
    scratch_parent = (
        content_dir
        if content_dir.is_dir()
        else Path(tempfile.gettempdir())
    )

    scratch = Path(
        tempfile.mkdtemp(
            prefix='mbmsms_',
            dir=str(scratch_parent),
        )
    )

    local_xyzrn = scratch / 'in.xyzrn'
    output_base = scratch / 'surface'

    try:
        shutil.copy2(
            source_xyzrn,
            local_xyzrn,
        )

        msms_bin = str(bin_path['MSMS'])

        args = [
            msms_bin,
            '-density', '3',
            '-hdensity', '3',
            '-probe', str(probe_radius),
            '-if', str(local_xyzrn),
            '-of', str(output_base),
            '-af', str(output_base),
        ]

        print(
            'MSMS COMMAND:',
            ' '.join(args),
        )

        p = Popen(
            args,
            stdout=PIPE,
            stderr=PIPE,
        )

        # IMPORTANT: this must be one assignment.
        stdout, stderr = p.communicate()

        stdout_text = stdout.decode(
            'utf-8',
            errors='replace',
        )
        stderr_text = stderr.decode(
            'utf-8',
            errors='replace',
        )

        if p.returncode != 0:
            raise RuntimeError(
                'MSMS failed.\n'
                f'Return code: {p.returncode}\n'
                f'Input: {local_xyzrn}\n'
                f'STDOUT:\n{stdout_text}\n'
                f'STDERR:\n{stderr_text}'
            )

        vert = Path(str(output_base) + '.vert')
        face = Path(str(output_base) + '.face')

        for fp in (vert, face):
            if not fp.exists():
                raise RuntimeError(
                    'MSMS returned success but output is missing: '
                    f'{fp}\n'
                    f'STDOUT:\n{stdout_text}\n'
                    f'STDERR:\n{stderr_text}'
                )
            if fp.stat().st_size == 0:
                raise RuntimeError(
                    f'MSMS output is empty: {fp}'
                )

        # Legacy MSMS .vert/.face are supposed to be ASCII text.
        for fp in (vert, face):
            raw = fp.read_bytes()
            try:
                raw.decode('ascii')
            except UnicodeDecodeError as e:
                raise RuntimeError(
                    'MSMS generated non-ASCII output despite '
                    'short-path execution.\n'
                    f'File: {fp}\n'
                    f'Size: {len(raw)} bytes\n'
                    f'First 256 bytes: {raw[:256]!r}\n'
                    f'Decode error: {e}\n'
                    f'MSMS STDOUT:\n{stdout_text}\n'
                    f'MSMS STDERR:\n{stderr_text}'
                )

        # AUTHORS' original parser.
        vertices, faces, normalv, res_id = read_msms(
            str(output_base)
        )

        # Strict geometry checks.
        if getattr(vertices, 'ndim', None) != 2 or vertices.shape[1] != 3:
            raise RuntimeError(
                f'MSMS vertices must be N x 3, got {getattr(vertices, "shape", None)}'
            )

        if getattr(faces, 'ndim', None) != 2 or faces.shape[1] != 3:
            raise RuntimeError(
                f'MSMS faces must be M x 3, got {getattr(faces, "shape", None)}'
            )

        if normalv.shape != vertices.shape:
            raise RuntimeError(
                'MSMS normal/vertex shape mismatch: '
                f'{normalv.shape} vs {vertices.shape}'
            )

        if len(res_id) != len(vertices):
            raise RuntimeError(
                'MSMS residue-ID/vertex count mismatch: '
                f'{len(res_id)} vs {len(vertices)}'
            )

        if len(vertices) == 0 or len(faces) == 0:
            raise RuntimeError(
                'MSMS produced empty geometry'
            )

        print(
            'MSMS PASS:',
            f'vertices={len(vertices)}',
            f'faces={len(faces)}',
        )

        return (
            vertices,
            faces,
            normalv,
            res_id,
        )

    finally:
        shutil.rmtree(
            scratch,
            ignore_errors=True,
        )
'''
)

print('Patched short-path MSMS:', APPLY_MSMS)


METALB_PREP_SCRIPT = METALB / 'external_prepare_v6.py'

METALB_PREP_SCRIPT.write_text(r'''
import argparse
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

from MetalBindProcessor import MetalBindProcessor


# ============================================================
# Label-free inference adapters
# ============================================================

def dummy_nucleic(self):
    return []


def dummy_interface(self, anno):
    # Inference only: labels/interfaces are placeholders and
    # are not used to generate predictions.
    return np.zeros(
        self.mesh.vertices.shape[0],
        dtype=np.float32
    )


def dummy_site(self, anno):
    # Preserve exact surface-residue ordering from author's processor.
    s2i = {
        res: i
        for i, res in enumerate(self.surface_res)
    }

    point_res_id = np.zeros(
        len(self.vert_info),
        dtype=np.int64
    )

    for k, info in enumerate(self.vert_info):
        fields = info.split('_')

        if len(fields) < 2:
            raise RuntimeError(
                f'Unexpected vert_info entry: {info}'
            )

        rid = fields[0] + '_' + fields[1]

        if rid not in s2i:
            raise RuntimeError(
                f'point residue {rid} missing from surface-residue map'
            )

        point_res_id[k] = s2i[rid]

    self.site_by_biolip = set()

    return (
        np.zeros(
            len(self.surface_res),
            dtype=np.int64
        ),
        point_res_id
    )


# Patch labels only; structural preprocessing remains author's code.
MetalBindProcessor._get_nucleic_acid_atom = dummy_nucleic
MetalBindProcessor._get_interface_mental = dummy_interface
MetalBindProcessor._get_site_biolip = dummy_site


# ============================================================
# CLI
# ============================================================

ap = argparse.ArgumentParser()
ap.add_argument('--manifest', required=True)
ap.add_argument('--root', required=True)
ap.add_argument('--rebuild', action='store_true')
a = ap.parse_args()

root = Path(a.root)
manifest_path = Path(a.manifest)

if not manifest_path.exists():
    raise FileNotFoundError(
        f'Manifest not found: {manifest_path}'
    )

manifest = pd.read_csv(manifest_path)

if len(manifest) == 0:
    raise RuntimeError(
        f'Empty manifest: {manifest_path}'
    )

required_cols = {
    'prot_id',
    'sequence',
    'pdb_path',
    'esm_txt',
}

missing_cols = required_cols - set(manifest.columns)

if missing_cols:
    raise RuntimeError(
        f'Manifest missing columns: {sorted(missing_cols)}'
    )


# ============================================================
# Output directories
# ============================================================

for d in [
    'chain_pdb',
    'protonated_pdb',
    'xyzrn',
    'msms',
    'data_label',
    'ligand_dummy',
    'hmm',
    'fasta',
    'dssp',
    'save',
    'mesh',
    'mesh_fix',
    'ply',
]:
    (root / d).mkdir(
        parents=True,
        exist_ok=True
    )


raw_pdb = Path(
    manifest.iloc[0].pdb_path
).parent

esm_dir = Path(
    manifest.iloc[0].esm_txt
).parent


if not raw_pdb.exists():
    raise FileNotFoundError(
        f'raw_pdb directory not found: {raw_pdb}'
    )

if not esm_dir.exists():
    raise FileNotFoundError(
        f'ESM directory not found: {esm_dir}'
    )


opts = {
    'base_dir': str(root),
    'raw_pdb_dir': str(raw_pdb),
    'ligand_dir': str(root / 'ligand_dummy'),
    'hhm_dir': str(root / 'hmm'),
    'fasta_dir': str(root / 'fasta'),
    'dssp': str(root / 'dssp'),
    'protonated_pdb_dir': str(root / 'protonated_pdb'),
    'chain_pdb_dir': str(root / 'chain_pdb'),
    'xyzrn_dir': str(root / 'xyzrn'),
    'msms_dir': str(root / 'msms'),
    'data_label': str(root / 'data_label'),
    'save_dir': str(root / 'save'),
    'mesh_dir': str(root / 'mesh'),
    'mesh_fix_dir': str(root / 'mesh_fix'),
    'ply_dir': str(root / 'ply'),
    'esm_dir': str(esm_dir),
}


print(
    f'Preparing {len(manifest)} proteins'
)

print(
    'raw_pdb_dir =',
    opts['raw_pdb_dir']
)

print(
    'esm_dir =',
    opts['esm_dir']
)


# ============================================================
# Per-protein preprocessing
# ============================================================

prepared = 0
cached = 0


for i, row in enumerate(
    manifest.itertuples(),
    start=1
):

    pid = str(row.prot_id)
    pair = f'{pid}:A:X1'

    fp = (
        root
        /
        'data_label'
        /
        pair
    )

    mp = (
        root
        /
        'data_label'
        /
        f'{pair}.surface_map.json'
    )


    # --------------------------------------------------------
    # Cache reuse
    # --------------------------------------------------------

    if (
        fp.exists()
        and
        mp.exists()
        and
        not a.rebuild
    ):
        cached += 1

        print(
            f'[{i}/{len(manifest)}] {pid}: cached'
        )

        continue


    print(
        f'[{i}/{len(manifest)}] {pid}: START'
    )


    # --------------------------------------------------------
    # Strict input checks
    # --------------------------------------------------------

    pdb_path = Path(row.pdb_path)
    esm_path = Path(row.esm_txt)

    if not pdb_path.exists():
        raise FileNotFoundError(
            f'{pid}: PDB missing: {pdb_path}'
        )

    if not esm_path.exists():
        raise FileNotFoundError(
            f'{pid}: ESM text missing: {esm_path}'
        )


    try:

        # ----------------------------------------------------
        # AUTHORS' processor
        # ----------------------------------------------------

        proc = MetalBindProcessor(
            pair=pair,
            anno='',
            ligand_type='ZN',
            # Metal branch -> 0.5-A probe radius in author's code
            dir_opts=opts,
            RQ_window=6.0,
            # exact public prepare_one.sh setting
        )

        proc.get_data()


        # ----------------------------------------------------
        # Verify author's data object
        # ----------------------------------------------------

        if not fp.exists():
            raise RuntimeError(
                f'{pid}: proc.get_data() finished but expected '
                f'data file was not created: {fp}'
            )


        # ----------------------------------------------------
        # Surface residue -> canonical sequence position
        # ----------------------------------------------------

        mapping = []

        for s in proc.surface_res:

            fields = s.split('_')

            if len(fields) < 2:
                raise RuntimeError(
                    f'{pid}: malformed surface residue: {s}'
                )

            if fields[0] != 'A':
                raise RuntimeError(
                    f'{pid}: unexpected surface chain {fields[0]}'
                )

            resid = int(fields[1])

            if (
                resid < 1
                or
                resid > len(row.sequence)
            ):
                raise RuntimeError(
                    f'{pid}: surface residue number {resid} '
                    f'outside sequence length {len(row.sequence)}'
                )

            mapping.append(
                {
                    'surface_residue': s,
                    'canonical_index': resid - 1,
                }
            )


        if len(mapping) == 0:
            raise RuntimeError(
                f'{pid}: no surface residues produced'
            )


        # ----------------------------------------------------
        # Save exact remapping metadata
        # ----------------------------------------------------

        mp.write_text(
            json.dumps(
                {
                    'prot_id': pid,
                    'sequence_length': len(row.sequence),
                    'surface_res': list(
                        proc.surface_res
                    ),
                    'mapping': mapping,
                },
                indent=2
            )
        )


        if not mp.exists():
            raise RuntimeError(
                f'{pid}: surface map was not written'
            )


        prepared += 1

        print(
            f'[{i}/{len(manifest)}] {pid}: PASS '
            f'surface_res={len(mapping)}'
        )


    except Exception:

        print(
            '\n'
            + '=' * 80
        )

        print(
            f'FAILED PROTEIN: {pid}'
        )

        print(
            f'PAIR: {pair}'
        )

        print(
            f'PDB: {pdb_path}'
        )

        print(
            f'ESM: {esm_path}'
        )

        print(
            '=' * 80
        )

        traceback.print_exc()

        print(
            '=' * 80
            + '\n'
        )

        raise


print(
    f'PREPROCESS COMPLETE: '
    f'prepared_now={prepared}, '
    f'cached={cached}, '
    f'total={len(manifest)}'
)
''')


# ============================================================
# RUN PREPROCESSING + STRICT COMPLETENESS AUDIT
# ============================================================

if RUN_METALBIND_TRANSFER:

    for dataset, df in DATASETS.items():

        print(
            '\n'
            + '=' * 80
        )

        print(
            'MetalBind surface preprocessing:',
            dataset
        )

        print(
            '=' * 80
        )


        root = (
            METALB_PREP_ROOT
            /
            dataset
        )

        manifest_path = (
            root
            /
            'manifest.csv'
        )


        if not manifest_path.exists():

            raise FileNotFoundError(
                f'{dataset}: manifest missing: '
                f'{manifest_path}. Run 18C first.'
            )


        cmd = (
            f'python "{METALB_PREP_SCRIPT}" '
            f'--manifest "{manifest_path}" '
            f'--root "{root}"'
            +
            (
                ' --rebuild'
                if REBUILD_METALBIND_SURFACE
                else ''
            )
        )


        # IMPORTANT:
        # capture real subprocess traceback instead of hiding it
        # behind CalledProcessError.
        rr = mex(
            'metalbind',
            cmd,
            cwd=METALB,
            check=False,
            capture=True
        )


        print(
            '\n'
            + '-' * 80
        )

        print(
            f'OUTPUT: {dataset}'
        )

        print(
            '-' * 80
        )


        if rr.stdout:
            print(
                rr.stdout
            )


        print(
            f'\nRETURN CODE: {rr.returncode}'
        )


        if rr.returncode != 0:

            raise RuntimeError(
                f'{dataset}: MetalBind surface preprocessing '
                f'failed with return code {rr.returncode}. '
                f'The real subprocess traceback is printed above.'
            )


        # ====================================================
        # STRICT POST-PREPROCESSING COMPLETENESS AUDIT
        # ====================================================

        missing = []
        bad_maps = []


        for r in df.itertuples():

            pid = str(
                r.prot_id
            )

            pair = (
                f'{pid}:A:X1'
            )

            data_fp = (
                root
                /
                'data_label'
                /
                pair
            )

            map_fp = (
                root
                /
                'data_label'
                /
                f'{pair}.surface_map.json'
            )


            if not data_fp.exists():

                missing.append(
                    (
                        pid,
                        'pickle'
                    )
                )


            if not map_fp.exists():

                missing.append(
                    (
                        pid,
                        'map'
                    )
                )

                continue


            try:

                z = json.loads(
                    map_fp.read_text()
                )

            except Exception as e:

                bad_maps.append(
                    (
                        pid,
                        f'unreadable map: {e}'
                    )
                )

                continue


            if (
                int(
                    z.get(
                        'sequence_length',
                        -1
                    )
                )
                !=
                len(r.sequence)
            ):

                bad_maps.append(
                    (
                        pid,
                        'sequence_length mismatch'
                    )
                )


            mapping = z.get(
                'mapping',
                []
            )


            if len(mapping) == 0:

                bad_maps.append(
                    (
                        pid,
                        'empty mapping'
                    )
                )

                continue


            canonical_indices = [
                int(
                    x[
                        'canonical_index'
                    ]
                )
                for x
                in mapping
            ]


            if any(
                x < 0
                or
                x >= len(r.sequence)
                for x
                in canonical_indices
            ):

                bad_maps.append(
                    (
                        pid,
                        'canonical index out of range'
                    )
                )


        if missing:

            raise RuntimeError(
                f'{dataset}: MetalBind preprocessing incomplete: '
                f'{missing[:10]}'
            )


        if bad_maps:

            raise RuntimeError(
                f'{dataset}: MetalBind surface-map audit failed: '
                f'{bad_maps[:10]}'
            )


        print(
            f'{dataset}: MetalBind preprocessing PASS '
            f'{len(df)}/{len(df)} proteins.'
        )


    print(
        '\n'
        'MetalBind surface preprocessing '
        'PASSED for TestSet300 + TestSetNew46.'
    )


Patched short-path MSMS: /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/repos/metalbind/compute_surface/apply_msms.py

MetalBind surface preprocessing: TestSet300
+ MAMBA_ROOT_PREFIX="/content/micromamba_binding_baselines" micromamba run -n metalbind python "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/repos/metalbind/external_prepare_v6.py" --manifest "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_external_v6/prepared/TestSet300/manifest.csv" --root "/content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_external_v6/prepared/TestSet300"

--------------------------------------------------------------------------------
OUTPUT: TestSet300
--------------------------------------------------------------------------------
Preparing 300 proteins
raw_pdb_dir = /content/drive/MyDrive/Protein-binding-Phi/baseline_reproduction_master_v6/metalbind_external_v6/prepared/Test

RuntimeError: TestSet300: MetalBind surface preprocessing failed with return code 134. The real subprocess traceback is printed above.

In [ ]:
# ============================================================
# 18E. RUN ALL 13 OFFICIAL MetalBind CHECKPOINTS AND REMAP FULL SEQUENCES
# ============================================================
METALB_INFER_SCRIPT=METALB/'external_infer_v6.py'
METALB_INFER_SCRIPT.write_text(r'''
import argparse, ast, json
from pathlib import Path
import _pickle as cPickle
import numpy as np
import pandas as pd
import torch

from Arguments import parser as author_parser
from models.MetalBind_model import MetalBind

ap=argparse.ArgumentParser()
ap.add_argument('--manifest',required=True)
ap.add_argument('--prepared_root',required=True)
ap.add_argument('--repo',required=True)
ap.add_argument('--output_root',required=True)
a=ap.parse_args()

if not torch.cuda.is_available():
    raise RuntimeError('MetalBind author forward path requires CUDA in this reproduction')

manifest=pd.read_csv(a.manifest)
prep=Path(a.prepared_root)
repo=Path(a.repo)
outroot=Path(a.output_root); outroot.mkdir(parents=True,exist_ok=True)

ions=['CA','CD','CO','CU','CU1','FE','FE2','K','MG','MN','NA','NI','ZN']


def tensor_data(data):
    n_points=data['xyz'].shape[0]
    xyz_type=np.asarray(data['xyz_type']).reshape(-1,1).astype(np.int64)
    chemi=torch.zeros((n_points,6),dtype=torch.float32)
    chemi.scatter_(1,torch.from_numpy(xyz_type),1.0)
    return {
        'xyz':torch.from_numpy(np.asarray(data['xyz'],dtype=np.float32)),
        'nuv':torch.from_numpy(np.asarray(data['nuv'],dtype=np.float32)).contiguous(),
        'hmm':torch.from_numpy(np.asarray(data['esm'],dtype=np.float32)),
        'geo':torch.from_numpy(np.asarray(data['pka'],dtype=np.float32)),
        'chemi':chemi,
        'y':torch.zeros(n_points,dtype=torch.long),
        'site':torch.zeros(len(data['site']),dtype=torch.long),
        'site_point':torch.from_numpy(np.asarray(data['site_point'],dtype=np.int64)),
        'xyz_batch':torch.zeros(n_points,dtype=torch.long),
    }

for ion in ions:
    ckpt=repo/'Pretrained_model'/ion
    if not (ckpt/'best.pth').exists():
        raise FileNotFoundError(f'missing released checkpoint {ckpt/"best.pth"}')

    args=author_parser.parse_args([
        '--ligand',ion,
        '--checkpoints_dir',str(ckpt),
        '--device','cuda:0',
        '--radius','6.0',
        '--n_layers','4',
        '--input_feature_type','esm','chemi','pka',
        '--batch_size','1',
        '--lr','0.001',
        '--loss_type','site',
    ])
    # dir_opts is not consumed by model construction/inference once tensors are ready.
    args.dir_opts={}
    model=MetalBind(args)
    model.load_network()
    model.eval()

    rows=[]
    with torch.no_grad():
        for row in manifest.itertuples():
            pid=str(row.prot_id); L=len(row.sequence)
            pair=f'{pid}:A:X1'
            with open(prep/'data_label'/pair,'rb') as f:
                data=cPickle.load(f)
            meta=json.loads((prep/'data_label'/f'{pair}.surface_map.json').read_text())

            td=tensor_data(data)
            model.set_input(td)
            surf=np.asarray(model.test().detach().cpu(),dtype=float).reshape(-1)
            mapping=meta['mapping']
            if len(surf) != len(mapping):
                raise RuntimeError(f'{pid}/{ion}: model emitted {len(surf)} surface residues, map has {len(mapping)}')
            if not np.all(np.isfinite(surf)) or np.min(surf)<0 or np.max(surf)>1:
                raise RuntimeError(f'{pid}/{ion}: invalid surface probabilities')

            full=np.zeros(L,dtype=np.float32)
            seen=set()
            for prob,m in zip(surf,mapping):
                idx=int(m['canonical_index'])
                if idx in seen:
                    raise RuntimeError(f'{pid}: duplicate canonical surface index {idx}')
                seen.add(idx)
                full[idx]=float(prob)

            for i,p in enumerate(full):
                rows.append((pid,i,float(p),i in seen))

    out=pd.DataFrame(rows,columns=['prot_id','residue_index','probability','is_surface_residue'])
    out.to_csv(outroot/f'{ion}.csv.gz',index=False,compression='gzip')
    print(ion,'saved',len(out),'residue rows')
    del model
    torch.cuda.empty_cache()
''')

METALBIND_EXTERNAL_ROOT=WORK/'metalbind_external_predictions_v6'
METALBIND_EXTERNAL_ROOT.mkdir(parents=True,exist_ok=True)

if RUN_METALBIND_TRANSFER:
    for dataset,df in DATASETS.items():
        root=METALB_PREP_ROOT/dataset
        outroot=METALBIND_EXTERNAL_ROOT/dataset
        outroot.mkdir(parents=True,exist_ok=True)
        mex(
            'metalbind',
            f'python "{METALB_INFER_SCRIPT}" --manifest "{root/"manifest.csv"}" '
            f'--prepared_root "{root}" --repo "{METALB}" --output_root "{outroot}"',
            cwd=METALB
        )

In [ ]:
# ============================================================
# 18F. STRICT FULL-LENGTH MetalBind INGEST + POOL + FULL METRICS
# ============================================================
def load_metalbind_ion_csv(path,df):
    z=pd.read_csv(path)
    need={'prot_id','residue_index','probability','is_surface_residue'}
    if not need.issubset(z.columns):
        raise ValueError(f'{path}: need columns {sorted(need)}')
    z['prot_id']=z['prot_id'].astype(str)
    out={}
    for r in df.itertuples():
        q=z[z.prot_id.eq(r.prot_id)].sort_values('residue_index')
        if len(q)!=len(r.sequence):
            raise RuntimeError(f'{path}: {r.prot_id}: {len(q)} rows != {len(r.sequence)} residues')
        idx=q.residue_index.to_numpy(int)
        if not np.array_equal(idx,np.arange(len(r.sequence))):
            raise RuntimeError(f'{path}: noncanonical indexing for {r.prot_id}')
        p=q.probability.to_numpy(float)
        if not np.all(np.isfinite(p)) or np.min(p)<0 or np.max(p)>1:
            raise RuntimeError(f'{path}: invalid probabilities for {r.prot_id}')
        out[r.prot_id]=p
    return out


metalb_rows=[]
metalbind_pooled_probs={}
metalbind_ion_probs={}

if RUN_METALBIND_TRANSFER:
    surface_coverage=[]
    for dataset,df in DATASETS.items():
        ion_probs={}
        for ion in METALB_IONS:
            fp=METALBIND_EXTERNAL_ROOT/dataset/f'{ion}.csv.gz'
            if not fp.exists():
                raise FileNotFoundError(f'MetalBind missing completed ion file: {fp}')
            ion_probs[ion]=load_metalbind_ion_csv(fp,df)

        pooled={}
        for r in df.itertuples():
            pooled[r.prot_id]=np.maximum.reduce([ion_probs[ion][r.prot_id] for ion in METALB_IONS])
        metalbind_pooled_probs[dataset]=pooled
        metalbind_ion_probs[dataset]=ion_probs

        save_probs(pooled,PRED/f'MetalBind_13ion_max_TRANSFER__{dataset}__metal_binding.csv.gz')
        out,pt=evaluate_probability_dict(
            df,pooled,'metal_binding','MetalBind_13ion_max_TRANSFER',dataset,
            'fixed_0.5',0.5,'C_cross_benchmark_transfer'
        )
        metalb_rows.append(out)
        pt.to_csv(RESULTS/f'MetalBind_{dataset}_per_protein.csv',index=False)

        # Surface coverage is method-internal context, not an evaluation filter.
        z=pd.read_csv(METALBIND_EXTERNAL_ROOT/dataset/f'{METALB_IONS[0]}.csv.gz')
        for pid,q in z.groupby(z.prot_id.astype(str)):
            surface_coverage.append([dataset,pid,int(q.is_surface_residue.astype(bool).sum()),len(q),float(q.is_surface_residue.astype(bool).mean())])

    metalb_df=pd.DataFrame(metalb_rows)
    metalb_df.to_csv(RESULTS/'MetalBind_TRANSFER_FULL_METRICS_300_46.csv',index=False)
    pd.DataFrame(surface_coverage,columns=['dataset','prot_id','surface_residues','total_residues','surface_fraction']).to_csv(
        RESULTS/'MetalBind_surface_coverage_300_46.csv',index=False
    )
    display(metalb_df)

## MetalBind implementation note

The external adapter changes **no learned weights and no learned forward computation**. It only removes label-dependent preprocessing requirements that are unavailable for a blind external protein:

- `_get_nucleic_acid_atom` → empty inference placeholder;
- interface labels → zeros;
- binding-site labels → zeros while preserving the exact point-to-surface-residue mapping.

These fields are needed by the authors' serialized data object / loss code but do not enter the feature encoder used to produce predictions. Geometry, ESM-C, atom type, pKa, tangent frames, site pooling, model architecture, and released checkpoints remain the authors' implementation.

# 18. Sequence-overlap audit for later transfer methods

ESMBind and MetalBind are later methods with different training sets. A TestSet300/46 transfer score is not automatically an independent benchmark.

This notebook performs exact-sequence scans over any FASTA files shipped in those repositories. For a paper, add MMseqs2 near-identity analysis before using these rows competitively.

In [ ]:
# ============================================================
# 18A. EXACT SEQUENCE OVERLAP AUDIT
# ============================================================
from Bio import SeqIO

external_seq={}
for dataset,df in DATASETS.items():
    for r in df.itertuples(): external_seq.setdefault(r.sequence,[]).append((dataset,r.prot_id))

def scan_fastas(root,method):
    rows=[]
    for fp in Path(root).rglob('*'):
        if fp.suffix.lower() not in ['.fa','.faa','.fasta','.fas']: continue
        try:
            for rec in SeqIO.parse(str(fp),'fasta'):
                seq=str(rec.seq)
                if seq in external_seq:
                    for dataset,pid in external_seq[seq]: rows.append([method,dataset,pid,str(fp),rec.id,len(seq)])
        except Exception: pass
    return rows

hits=scan_fastas(REPOS/'esmbind','ESMBind')+scan_fastas(REPOS/'metalbind','MetalBind')
overlap=pd.DataFrame(hits,columns=['method','dataset','external_id','author_file','author_record','length'])
display(overlap.head(100));overlap.to_csv(RESULTS/'exact_sequence_overlap_audit.csv',index=False)

# 19. Merge with historical KAN / GEAR outputs

After the public baselines are reproduced, merge their `PAPER_READY_FIXED_0_5_300_46.csv` with project rows.

For KAN / GEAR, use exactly the same metric names and conventions:

- `AP_AUPRC`
- `AUROC`
- `global_F1`
- `global_MCC`
- `protein_mean_F1`
- `protein_mean_MCC`

If a project prediction CSV exists, the preferred approach is to pass its probability dictionary through **this evaluator** rather than copying old summary numbers. That creates one genuinely unified table.

# 19B. Unified locked-model master table

This final table combines every successfully completed frozen-model branch.

Rows are included only when the method produced complete canonical full-length predictions for the requested external set. No failed proteins are silently omitted.

In [ ]:
# ============================================================
# 19B. MASTER LOCKED-MODEL TABLE
# ============================================================
frames=[]
if 'full_direct' in globals() and len(full_direct):
    frames.append(full_direct.copy())
if 'esmb_df' in globals() and len(esmb_df):
    frames.append(esmb_df.copy())
if 'metalb_df' in globals() and len(metalb_df):
    frames.append(metalb_df.copy())

master_locked=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
if len(master_locked):
    master_locked.to_csv(RESULTS/'MASTER_LOCKED_MODEL_FULL_METRICS_300_46.csv',index=False)
    display(master_locked.sort_values(['dataset','label','method','operating_point']))
else:
    print('No completed model branches yet.')

# Final hard completion guard

When all four default branches are enabled, v6 refuses to finish unless the fixed-0.5 master table contains:

- bindEmbed21DL: 2 datasets × 4 targets;
- bindNode24: 2 datasets × 4 targets;
- ESMBind: 2 datasets × metal;
- MetalBind: 2 datasets × metal.

That is **20 complete fixed-threshold rows**, with no missing protein or residue in the underlying probability files.

In [ ]:
# ============================================================
# PUBLIC MODEL / BENCHMARK ARTIFACT SHA256 MANIFEST
# ============================================================
import hashlib

def sha256_file(path,chunk=1024*1024):
    path=Path(path)
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

artifact_rows=[]

for name,p in [
    ('project_TestSet300',TEST300_CSV),
    ('project_TestSetNew46',TEST46_CSV),
]:
    if Path(p).exists():
        artifact_rows.append(['benchmark',name,str(p),Path(p).stat().st_size,sha256_file(p)])

for i in range(1,6):
    p=REPOS/'bind21'/'trained_models'/f'checkpoint{i}.pt'
    if p.exists():
        artifact_rows.append(['bindEmbed21DL',f'checkpoint{i}',str(p),p.stat().st_size,sha256_file(p)])

for i in range(5):
    p=REPOS/'bindnode24'/'trained_model_dicts'/'bindNode24'/f'trained_model_{i}.state_dict'
    if p.exists():
        artifact_rows.append(['bindNode24',f'trained_model_{i}',str(p),p.stat().st_size,sha256_file(p)])

for i in range(1,6):
    p=REPOS/'esmbind'/'multi_modal_binding'/'model'/'trained_weights'/f'fold_{i}.pt'
    if p.exists():
        artifact_rows.append(['ESMBind',f'fold_{i}',str(p),p.stat().st_size,sha256_file(p)])

for ion in METALB_IONS:
    p=REPOS/'metalbind'/'Pretrained_model'/ion/'best.pth'
    if p.exists():
        artifact_rows.append(['MetalBind',ion,str(p),p.stat().st_size,sha256_file(p)])

artifact_manifest=pd.DataFrame(
    artifact_rows,
    columns=['method','artifact','path','bytes','sha256']
)
artifact_manifest.to_csv(RESULTS/'PUBLIC_MODEL_ARTIFACT_SHA256_MANIFEST.csv',index=False)
display(artifact_manifest)

if RUN_BIND21:
    assert (artifact_manifest.method=='bindEmbed21DL').sum()==5
if RUN_BINDNODE24:
    assert (artifact_manifest.method=='bindNode24').sum()==5
if RUN_ESMBIND_TRANSFER:
    assert (artifact_manifest.method=='ESMBind').sum()==5
if RUN_METALBIND_TRANSFER:
    assert (artifact_manifest.method=='MetalBind').sum()==len(METALB_IONS)

print('Public checkpoint provenance audit passed.')


In [ ]:
# ============================================================
# HARD COMPLETION AUDIT
# ============================================================
expected=[]
if RUN_BIND21:
    expected += [('bindEmbed21DL',ds,lab) for ds in DATASETS for lab in LABELS]
if RUN_BINDNODE24:
    expected += [('bindNode24',ds,lab) for ds in DATASETS for lab in LABELS]
if RUN_ESMBIND_TRANSFER:
    expected += [('ESMBind_7ion_max_TRANSFER',ds,'metal_binding') for ds in DATASETS]
if RUN_METALBIND_TRANSFER:
    expected += [('MetalBind_13ion_max_TRANSFER',ds,'metal_binding') for ds in DATASETS]

fixed_master=master_locked[master_locked.operating_point.eq('fixed_0.5')].copy() if len(master_locked) else pd.DataFrame()
actual=set(zip(fixed_master.method,fixed_master.dataset,fixed_master.label)) if len(fixed_master) else set()
missing=[x for x in expected if x not in actual]
status=pd.DataFrame(expected,columns=['method','dataset','label'])
status['complete']=status.apply(lambda r:(r.method,r.dataset,r.label) in actual,axis=1)
status.to_csv(RESULTS/'FINAL_BRANCH_COMPLETION_AUDIT.csv',index=False)
display(status)

if missing:
    raise RuntimeError(
        'FINAL COMPLETION GUARD FAILED. Do not use the master comparison yet. Missing rows:\n' +
        '\n'.join(map(str,missing[:30]))
    )
print(f'FINAL COMPLETION GUARD PASSED: {len(expected)} / {len(expected)} required fixed-0.5 rows present.')

In [ ]:
# ============================================================
# 19A. FINAL REPRODUCIBILITY MANIFEST
# ============================================================
manifest={
    'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
    'repo_pins':{k:{'url':v[0],'sha':v[1]} for k,v in REPO_SPECS.items()},
    'datasets':{k:{'proteins':len(v),'csv':str(TEST300_CSV if k=='TestSet300' else TEST46_CSV)} for k,v in DATASETS.items()},
    'fixed_threshold':FIXED_THRESHOLD,
    'OOF_thresholds_enabled':RUN_OOF_THRESHOLDS,
    'AP_definition':'sklearn.average_precision_score',
    'PR_AUC_trapezoid_saved_separately':True,
    'global_metrics_definition':'all residues concatenated first',
    'per_protein_definition':'calculate per protein; include iff TP+FP+FN>0; average included proteins',
    'any_probability':'max(metal,nuclear,small)',
    'ESMBind_transfer_probability':'max(MG,FE,CU,CO,CA,MN,ZN)',
    'MetalBind_transfer_probability':'max over 13 official ion-model probabilities after author surface preprocessing and full-sequence remapping',
    'bootstrap_reps':BOOTSTRAP_REPS,
    'notes':[
        'Main paper comparison should use fixed 0.5 + per-protein means for literature consistency.',
        'OOF-F1 and OOF-MCC operating points are reported separately.',
        'Test-oracle thresholds are diagnostic only and never reportable.',
        'bindNode24 TestSetNew46 is an external application using project structures, not a public native reproduction.',
        'ESMBind/MetalBind run all external proteins and are scored only against metal_binding; no positive-only filtering.'
    ]
}
(MANIFEST/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print(MANIFEST/'run_manifest.json')

# 20. Files you should keep after the run

The most important outputs are:

- `FULL_DIRECT_BASELINE_METRICS_300_46.csv` — all direct metrics and operating points
- `FULL_DIRECT_PER_PROTEIN_300_46.csv` — protein-by-protein confusion matrices/F1/MCC
- `PAPER_READY_FIXED_0_5_300_46.csv` — compact main-table view
- `SUPPLEMENTARY_OOF_OPERATING_POINTS_300_46.csv` — frozen OOF-F1 and OOF-MCC operating points
- `REPRODUCTION_GUARD.csv` — whether published-scale performance was reproduced
- full probability CSVs under `predictions/`
- `run_manifest.json`

## Interpretation rule

Do not call a run “reproduced” merely because the script completed. Call it reproduced only when:

1. all expected proteins are present;
2. every probability vector exactly matches sequence length;
3. no protein is silently dropped;
4. fixed-0.5 per-protein F1/MCC is close to published scale;
5. the repository commit and input provenance are recorded.

# Final interpretation checklist

- bindEmbed21DL 300 and 46 = direct/native published external evaluations.
- bindNode24 300 = direct/native published evaluation.
- bindNode24 46 = new external application using project structures.
- ESMBind 300/46 = metal-only cross-benchmark transfer; evaluate only `metal_binding`.
- MetalBind 300/46 = metal-only cross-benchmark transfer; evaluate only `metal_binding`; currently guarded until full preprocessing adaptation succeeds.
- Never filter Test300/TestNew46 proteins based on whether they truly contain a metal site.
- Never compare pooled-residue F1 with per-protein mean F1 as if they were the same metric.
- `class_micro_F1` and `class_macro_F1` are provided only for completeness; the published bindEmbed/bindNode F1 is a mean of per-protein positive-class F1 values.

# FULL metric output schema

Each completed method × dataset × task × operating point can contain the following fields.

### Threshold-free
`n_residues`, `n_positive`, `n_negative`, `prevalence`, `AP_AUPRC`, `PR_AUC_trapezoid`, `AUROC`, `Brier`.

### Pooled-residue confusion/operating metrics
`threshold`, `TP`, `FP`, `TN`, `FN`, `global_accuracy`, `global_balanced_accuracy`, `global_precision_PPV`, `global_recall_sensitivity`, `global_specificity`, `global_NPV`, `global_positive_F1`, `global_MCC`, `global_F2`, `global_Jaccard`, `predicted_positive_rate`.

### Additional class-aggregation diagnostics
`negative_class_F1`, `class_macro_F1`, `class_weighted_F1`, `class_micro_F1`.

### Per-protein mean metrics
`protein_mean_precision`, `protein_mean_recall`, `protein_mean_specificity`, `protein_mean_F1`, `protein_mean_MCC`, `protein_mean_accuracy`.

### Uncertainty / coverage of protein-wise aggregation
For each protein-wise metric: legacy and standard t-CI half-widths. For F1/MCC: bootstrap 95% low/high. Also: `protein_macro_positive_n`, `protein_false_alarm_only_n`, `protein_TN_only_excluded_n`.

### Primary manuscript interpretation

- **Primary discrimination metric:** `AP_AUPRC`.
- **Contextual discrimination:** `AUROC`.
- **Common fixed operating point:** `global_positive_F1@0.5`, `global_MCC@0.5`.
- **Literature-style bindEmbed/bindNode comparison:** `protein_mean_F1@0.5`, `protein_mean_MCC@0.5` with CI.
- **Development-selected operating points:** OOF-F1 and OOF-MCC reported separately; never mix them with fixed 0.5 or with each other.

# v6 run expectations — all four comparison branches enabled

## Before pressing Run all

- Use a **GPU Colab runtime**.
- Keep the project root at `/content/drive/MyDrive/Protein-binding-Phi`, or edit `BASE` once.
- The notebook runs **all TestSet300 proteins and all TestSetNew46 proteins**.
- bindEmbed21DL / bindNode24 score metal, nucleic, small, and any-ligand.
- ESMBind / MetalBind run all proteins but score **metal only**.
- ESMBind and especially MetalBind download large model/runtime dependencies.
- MetalBind is slowest because it reproduces the authors' MSMS / Reduce / PyMesh / PyG surface pipeline.
- If ESM-C access requires authentication, put `HF_TOKEN` in **Colab Secrets**. The notebook imports it without printing it.
- A homomeric PDB with several exact-sequence chains is handled deterministically: chain A is preferred, otherwise lexical chain order.
- No branch may silently drop a protein, residue, structure, feature file, or released checkpoint. Missing material causes a hard failure.

## Full common metrics

**Threshold-free:** AP/AUPRC, trapezoidal PR-AUC, AUROC, Brier score, counts, prevalence.

**Pooled-residue threshold metrics:** TP, FP, TN, FN, accuracy, balanced accuracy, precision/PPV, recall/sensitivity, specificity, NPV, positive-class F1 (`global_positive_F1`), negative-class F1, class-macro F1, class-weighted F1, class-micro F1, F2, Jaccard, MCC, predicted-positive rate.

**Per-protein metrics:** precision, recall, specificity, positive-class F1, MCC, accuracy, macro-positive count, false-alarm-only count, TN-only excluded count, legacy and standard t-CIs, and 2000-replicate protein-bootstrap 95% CIs for F1/MCC.

## Metric naming

- `global_positive_F1`: concatenate all residues first, then calculate positive-class F1.
- `protein_mean_F1`: calculate F1 separately for each included protein, then average.
- `class_micro_F1`: class-micro F1 in a binary single-label problem; retained only as a diagnostic and **not** the binding-positive F1 used for comparison.

## Operating points

- Main comparison: fixed threshold **0.5**.
- bindEmbed21DL / bindNode24: additional native-Dev OOF F1-optimal and MCC-optimal thresholds.
- Test-oracle thresholds are diagnostic only and never valid for claims.

## Reproducibility outputs

v6 saves raw probabilities, per-protein metrics, normalized-structure audit, MetalBind surface coverage, repository commit pins, SHA256 hashes for public checkpoints, and a final branch-completion audit.

The final completion guard passes only when every requested fixed-0.5 row exists.
